# RF Predictions -> Clustering

Este notebook carrega CSVs gerados pelo notebook de Random Forest (ex: `rf_test_with_proba_10.csv` ou `rf_train_with_proba_10.csv`) e executa clustering (Kmedoids) sobre as features/probabilidades selecionadas.
- Ajuste os caminhos de entrada na célula de parâmetros.
- Saídas: `databases/UCI-THYROID-DXBIN/clusters_reports/random_forest/` (csv, images, pdf).

In [ ]:
# ================================== CONFIGURAÇÃO (3 classes) ==================================
from pathlib import Path

# caminhos de exemplo (substitua conforme necessário)
CSV_TEST_PATH  = Path('../model_reports/random_forest/basico/csv/X_test_basic_full.csv')
CSV_TRAIN_PATH = Path('../model_reports/random_forest/basico/csv/X_train_basic_full.csv')

# datasets de entrada 
X_TRAIN        = None  # Dados brutos de Treino
X_TEST         = None  # Dados brutos de Test
X_TRAIN_NORM   = None  # Dados NORMALIZADOS de Treino
X_TEST_NORM    = None  # Dados NORMALIZADOS de Test

DF_GLOBAL       = None  # Dados brutos Globais
DF_GLOBAL_NORM  = None  # Dados NORMALIZADOS Globais

# Agora com 3 classes (C0, C1, C2)
DF_GLOBAL_C0       = None  # Dados brutos Globais C0
DF_GLOBAL_C1       = None  # Dados brutos Globais C1
DF_GLOBAL_C2       = None  # Dados brutos Globais C2
DF_GLOBAL_NORM_C0  = None  # Dados NORMALIZADOS Globais C0
DF_GLOBAL_NORM_C1  = None  # Dados NORMALIZADOS Globais C1
DF_GLOBAL_NORM_C2  = None  # Dados NORMALIZADOS Globais C2

# Saídas
OUT_BASE = Path('../clusters_reports/random_forest')
OUT_CSV  = OUT_BASE / 'csv'
OUT_IMG  = OUT_BASE / 'images'
OUT_PDF  = OUT_BASE / 'pdf'

for d in [OUT_BASE, OUT_CSV, OUT_IMG, OUT_PDF]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------- Parâmetros de clustering ----------------
RANDOM_STATE = 42

# Quantidade de clusters por classe (ajuste conforme seu elbow/silhueta)
# Forneci defaults para a classe 2 para evitar KeyError.
N_CLUSTERS_ELBOW = {0: 4, 1: 3, 2: 4}
N_CLUSTERS_ELBOW_RECOMENDED = None

# Número final usado por classe
N_CLUSTERS = {0: 4, 1: 3, 2: 4} #{0: 7, 1: 5, 2: 7}

N_CLUSTERS_SILHUET = None
SILHUET = N_CLUSTERS  # use este dict no código, como já está

THRESHOLD = 0.5
DPI = 800

# ---------------- Colunas a excluir das FEATURES de clustering/PCA ----------------
# Base (sempre excluídas)
_EXCLUDE_BASE = [
    '__cls_tmp__', 'y', 'orig_index', 'diagnosis', 'id', 'ID',
    'patient_id', 'pid', 'fold', 'y_train', 'y_pred', '_target',
    'target',  # nomes comuns de alvo",
    # probabilidades binárias/legados
    'prob_0', 'prob_1', 'y_proba'
]

# Extras possíveis quando salvar multiclasse (para evitar que entrem nas features):
# - prob_2 (multiclasse)
# - y_proba_max (maior probabilidade entre as classes)
# - y_pred_argmax (classe do argmax das probabilidades)
_EXCLUDE_EXTRA = ['prob_2', 'y_proba_max', 'y_pred_argmax']

EXCLUDE_COLUMNS = _EXCLUDE_BASE + _EXCLUDE_EXTRA


In [ ]:
# Carregar CSV_TEST_PATH e CSV_TRAIN_PATH, normalizar se necessário, salvar arquivos normalizados e global no diretório configurado
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import os
from pathlib import Path

# Função para normalizar um DataFrame, exceto colunas excluídas
def normalize_df(df, exclude_columns, method):
    cols_to_normalize = [col for col in df.columns if col not in exclude_columns and pd.api.types.is_numeric_dtype(df[col])]
    if method.lower() in ['standard scale','standard','standardscale','zscore','z-score']:
        scaler = StandardScaler()
    elif method.lower() in ['minmax','min-max','minmaxscale']:
        scaler = MinMaxScaler()
    elif method.lower() in ['robust','robust scale','robustscale']:
        scaler = RobustScaler()
    else:
        raise ValueError(f"Método de normalização desconhecido: {method}")
    df_NORM = df.copy()
    if cols_to_normalize:
        df_NORM[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])
    return df_NORM

# Helper para converter Path/str
def to_str(x):
    return str(x) if not isinstance(x, str) else x

CSV_TEST_PATH  = to_str(CSV_TEST_PATH)
CSV_TRAIN_PATH = to_str(CSV_TRAIN_PATH)

# Diretório de saída para CSVs
out_csv_dir = Path(OUT_CSV) if 'OUT_CSV' in globals() else Path('./')
out_csv_dir.mkdir(parents=True, exist_ok=True)

# Carregar datasets (mantém colunas originais)
X_TEST = pd.read_csv(CSV_TEST_PATH , index_col=0)
X_TRAIN = pd.read_csv(CSV_TRAIN_PATH, index_col=0)

# Normalização condicional
if not DATASET_NORMALIZED:
    X_TEST_NORM = normalize_df(X_TEST, EXCLUDE_COLUMNS, NORMALIZED_METHOD)
    X_TRAIN_NORM = normalize_df(X_TRAIN, EXCLUDE_COLUMNS, NORMALIZED_METHOD)
else:
    X_TEST_NORM = X_TEST.copy()
    X_TRAIN_NORM = X_TRAIN.copy()

# Garantir que variáveis em DISCRETIZE_VARIABLES estejam numéricas (pelo menos normalizadas se forem numéricas)
if isinstance(DISCRETIZE_VARIABLES, dict):
    discretize_vars = list(DISCRETIZE_VARIABLES.keys())
else:
    discretize_vars = list(DISCRETIZE_VARIABLES) if DISCRETIZE_VARIABLES is not None else []

for var in discretize_vars:
    if var in X_TEST_NORM.columns and not np.issubdtype(X_TEST_NORM[var].dtype, np.number):
        # Tentativa: se categórica, aplicar codificação simples (factorize) antes de escalar
        codes, _ = pd.factorize(X_TEST_NORM[var])
        X_TEST_NORM[var] = codes.astype(float)
    if var in X_TRAIN_NORM.columns and not np.issubdtype(X_TRAIN_NORM[var].dtype, np.number):
        codes, _ = pd.factorize(X_TRAIN_NORM[var])
        X_TRAIN_NORM[var] = codes.astype(float)

# Reaplicar normalização apenas nessas variáveis discretizadas recém numéricas se dataset não era normalizado
if not DATASET_NORMALIZED and discretize_vars:
    inter_test = X_TEST_NORM[discretize_vars]
    inter_train = X_TRAIN_NORM[discretize_vars]
    # Escalar somente as discretize vars que não estão excluídas
    vars_to_scale = [v for v in discretize_vars if v not in EXCLUDE_COLUMNS and v in X_TEST_NORM.columns]
    if vars_to_scale:
        # Escolher scaler consistente com método
        if NORMALIZED_METHOD.lower() in ['standard scale','standard','standardscale','zscore','z-score']:
            scaler2 = StandardScaler()
        elif NORMALIZED_METHOD.lower() in ['minmax','min-max','minmaxscale']:
            scaler2 = MinMaxScaler()
        elif NORMALIZED_METHOD.lower() in ['robust','robust scale','robustscale']:
            scaler2 = RobustScaler()
        else:
            raise ValueError(f"Método de normalização desconhecido: {NORMALIZED_METHOD}")
        X_TRAIN_NORM[vars_to_scale] = scaler2.fit_transform(X_TRAIN_NORM[vars_to_scale])
        X_TEST_NORM[vars_to_scale] = scaler2.transform(X_TEST_NORM[vars_to_scale])

# Construir caminhos de saída
CSV_TEST_NORMALIZED = out_csv_dir / 'csv_teste_normalizado.csv'
CSV_TRAIN_NORMALIZED = out_csv_dir / 'csv_train_normalizado.csv'
CSV_GLOBAL_NORMALIZED = out_csv_dir / 'csv_global_normalizado.csv'
CSV_GLOBAL            = out_csv_dir / 'csv_global_cru.csv'

# Salvar arquivos
X_TEST_NORM.to_csv(CSV_TEST_NORMALIZED, index=False)
X_TRAIN_NORM.to_csv(CSV_TRAIN_NORMALIZED, index=False)
DF_GLOBAL_NORM = pd.concat([X_TRAIN_NORM, X_TEST_NORM], axis=0, ignore_index=True)
DF_GLOBAL_NORM.to_csv(CSV_GLOBAL_NORMALIZED, index=False)
DF_GLOBAL = pd.concat([X_TRAIN, X_TEST], axis=0, ignore_index=True)
DF_GLOBAL.to_csv(CSV_GLOBAL, index=False)

print('Arquivos normalizados salvos:')
print('  Teste Normalizado :', CSV_TEST_NORMALIZED)
print('  Treino Normalizado:', CSV_TRAIN_NORMALIZED)
print('  Global Normalizado:', CSV_GLOBAL_NORMALIZED)
print('  Global Cru:', CSV_GLOBAL)

In [ ]:
# Carregar CSV_TEST_PATH e CSV_TRAIN_PATH, normalizar se necessário, salvar arquivos normalizados e global no diretório configurado
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import os
from pathlib import Path

# ---------------- helpers ----------------
def _detect_prob_columns(df):
    """Detecta colunas de probabilidade multiclasse e auxiliares para exclusão automática."""
    cols = []
    if df is not None and isinstance(df, pd.DataFrame):
        cols += [c for c in df.columns if c.startswith('prob_')]  # prob_0, prob_1, prob_2, ...
        for extra in ('y_proba_max', 'y_pred_argmax'):
            if extra in df.columns:
                cols.append(extra)
    return sorted(list(dict.fromkeys(cols)))

def _discretize_inplace(df, discretize_spec):
    """
    Aplica mapeamentos de DISCRETIZE_VARIABLES (se houver).
    Se alguma coluna indicada não for numérica e não tiver mapeamento, realiza factorize naquela coluna.
    """
    if not isinstance(discretize_spec, dict) or df is None or not isinstance(df, pd.DataFrame):
        return df

    for col, mapping in discretize_spec.items():
        if col not in df.columns:
            continue
        if isinstance(mapping, dict) and len(mapping) > 0:
            df[col] = df[col].map(mapping)
        # Se ainda não numérico, usa factorize
        if not pd.api.types.is_numeric_dtype(df[col]):
            codes, _ = pd.factorize(df[col])
            df[col] = codes.astype(float)
    return df

def normalize_df(df, exclude_columns, method):
    """
    Normaliza apenas colunas numéricas que não estejam em exclude_columns.
    """
    cols_to_normalize = [
        col for col in df.columns
        if (col not in exclude_columns) and pd.api.types.is_numeric_dtype(df[col])
    ]
    m = method.lower()
    if m in ['standard scale','standard','standardscale','zscore','z-score']:
        scaler = StandardScaler()
    elif m in ['minmax','min-max','minmaxscale']:
        scaler = MinMaxScaler()
    elif m in ['robust','robust scale','robustscale']:
        scaler = RobustScaler()
    else:
        raise ValueError(f"Método de normalização desconhecido: {method}")

    df_NORM = df.copy()
    if cols_to_normalize:
        df_NORM[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])
    return df_NORM

def to_str(x):
    return str(x) if not isinstance(x, str) else x

# ---------------- paths ----------------
CSV_TEST_PATH  = to_str(CSV_TEST_PATH)
CSV_TRAIN_PATH = to_str(CSV_TRAIN_PATH)

# Diretório de saída para CSVs
out_csv_dir = Path(OUT_CSV) if 'OUT_CSV' in globals() else Path('./')
out_csv_dir.mkdir(parents=True, exist_ok=True)

# ---------------- load ----------------
# Carregar datasets (mantém colunas originais; index do arquivo vira índice do DF)
X_TEST  = pd.read_csv(CSV_TEST_PATH,  index_col=0)
X_TRAIN = pd.read_csv(CSV_TRAIN_PATH, index_col=0)

# ---------------- discretização (somente se especificada) ----------------
# Aplica mapeamentos de DISCRETIZE_VARIABLES antes de normalizar (mínima intervenção)
if 'DISCRETIZE_VARIABLES' in globals() and isinstance(DISCRETIZE_VARIABLES, dict) and len(DISCRETIZE_VARIABLES) > 0:
    X_TEST  = _discretize_inplace(X_TEST,  DISCRETIZE_VARIABLES)
    X_TRAIN = _discretize_inplace(X_TRAIN, DISCRETIZE_VARIABLES)

# ---------------- exclusões efetivas ----------------
# Usa EXCLUDE_COLUMNS do seu config e acrescenta automaticamente prob_*, y_proba_max, y_pred_argmax (se existirem)
_exclude_base = list(EXCLUDE_COLUMNS) if 'EXCLUDE_COLUMNS' in globals() else []
_exclude_auto = sorted(list(set(_detect_prob_columns(X_TEST) + _detect_prob_columns(X_TRAIN))))
EXCLUDE_COLUMNS_EFFECTIVE = sorted(list(dict.fromkeys(_exclude_base + _exclude_auto)))

# ---------------- normalização condicional ----------------
if not DATASET_NORMALIZED:
    X_TEST_NORM  = normalize_df(X_TEST,  EXCLUDE_COLUMNS_EFFECTIVE, NORMALIZED_METHOD)
    X_TRAIN_NORM = normalize_df(X_TRAIN, EXCLUDE_COLUMNS_EFFECTIVE, NORMALIZED_METHOD)
else:
    X_TEST_NORM  = X_TEST.copy()
    X_TRAIN_NORM = X_TRAIN.copy()

# ---------------- re-normalização de discretizadas recém-numéricas (raro, mas preservado) ----------------
if 'DISCRETIZE_VARIABLES' in globals() and isinstance(DISCRETIZE_VARIABLES, dict) and len(DISCRETIZE_VARIABLES) > 0 and (not DATASET_NORMALIZED):
    discretize_vars = list(DISCRETIZE_VARIABLES.keys())
    vars_to_scale = [v for v in discretize_vars if (v in X_TEST_NORM.columns and v not in EXCLUDE_COLUMNS_EFFECTIVE)]
    if vars_to_scale:
        m = NORMALIZED_METHOD.lower()
        if m in ['standard scale','standard','standardscale','zscore','z-score']:
            scaler2 = StandardScaler()
        elif m in ['minmax','min-max','minmaxscale']:
            scaler2 = MinMaxScaler()
        elif m in ['robust','robust scale','robustscale']:
            scaler2 = RobustScaler()
        else:
            raise ValueError(f"Método de normalização desconhecido: {NORMALIZED_METHOD}")
        X_TRAIN_NORM[vars_to_scale] = scaler2.fit_transform(X_TRAIN_NORM[vars_to_scale])
        X_TEST_NORM[vars_to_scale]  = scaler2.transform(X_TEST_NORM[vars_to_scale])

# ---------------- salvar ----------------
CSV_TEST_NORMALIZED   = out_csv_dir / 'csv_teste_normalizado.csv'
CSV_TRAIN_NORMALIZED  = out_csv_dir / 'csv_train_normalizado.csv'
CSV_GLOBAL_NORMALIZED = out_csv_dir / 'csv_global_normalizado.csv'
CSV_GLOBAL            = out_csv_dir / 'csv_global_cru.csv'

# Salva arquivos
X_TEST_NORM.to_csv(CSV_TEST_NORMALIZED, index=False)
X_TRAIN_NORM.to_csv(CSV_TRAIN_NORMALIZED, index=False)

DF_GLOBAL_NORM = pd.concat([X_TRAIN_NORM, X_TEST_NORM], axis=0, ignore_index=True)
DF_GLOBAL_NORM.to_csv(CSV_GLOBAL_NORMALIZED, index=False)

DF_GLOBAL = pd.concat([X_TRAIN, X_TEST], axis=0, ignore_index=True)
DF_GLOBAL.to_csv(CSV_GLOBAL, index=False)

print('Arquivos normalizados salvos:')
print('  Teste Normalizado :', CSV_TEST_NORMALIZED)
print('  Treino Normalizado:', CSV_TRAIN_NORMALIZED)
print('  Global Normalizado:', CSV_GLOBAL_NORMALIZED)
print('  Global Cru:', CSV_GLOBAL)
print('Colunas excluídas (efetivas) da normalização/clustering:', EXCLUDE_COLUMNS_EFFECTIVE)


In [ ]:
# ==== Análise Silhueta: clusters por Silhueta (usuário e recomendado) ====
# Suporta 2 ou 3 classes (0,1,2) dinamicamente e cria DF_GLOBAL_NORM_C* se faltarem.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn_extra.cluster import KMedoids

RESULTS_DIR = str(OUT_CSV)
IMG_DIR = str(OUT_IMG) if 'OUT_IMG' in globals() else os.path.join(RESULTS_DIR, 'images')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

# -------------------- PREP: garantir splits por classe normalizados --------------------
def _pick_label_column(df):
    """Escolhe a coluna de classe na ordem: TARGET_COLUMN -> y_pred -> y -> y_true."""
    candidates = []
    if 'TARGET_COLUMN' in globals() and isinstance(TARGET_COLUMN, str):
        candidates.append(TARGET_COLUMN)
    candidates += ['y_pred', 'y', 'y_true']
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(
        f"Não encontrei coluna-alvo em {candidates}. Colunas disponíveis: {list(df.columns)[:30]} ..."
    )

def _get_global_norm_df():
    """Obtém DF_GLOBAL_NORM; se ausente, tenta concatenar X_TRAIN_NORM + X_TEST_NORM."""
    if 'DF_GLOBAL_NORM' in globals() and isinstance(DF_GLOBAL_NORM, pd.DataFrame) and not DF_GLOBAL_NORM.empty:
        return DF_GLOBAL_NORM.copy()
    if ('X_TRAIN_NORM' in globals() and isinstance(X_TRAIN_NORM, pd.DataFrame) and not X_TRAIN_NORM.empty and
        'X_TEST_NORM'  in globals() and isinstance(X_TEST_NORM,  pd.DataFrame) and not X_TEST_NORM.empty):
        return pd.concat([X_TRAIN_NORM, X_TEST_NORM], axis=0, ignore_index=True)
    raise RuntimeError("DF_GLOBAL_NORM não encontrado e não foi possível montar a partir de X_TRAIN_NORM/X_TEST_NORM.")

# Se os DF_GLOBAL_NORM_C* não existirem, crie-os agora
need_build = False
for _c in [0, 1, 2]:
    if globals().get(f'DF_GLOBAL_NORM_C{_c}', None) is None:
        need_build = True
        break

if need_build:
    base_df_norm = _get_global_norm_df()
    label_col = _pick_label_column(base_df_norm)

    # classes presentes (apenas valores numéricos/inteiros e dentro de {0,1,2})
    present = pd.unique(base_df_norm[label_col].dropna())
    try:
        present = np.array(present, dtype=int)
    except Exception:
        raise RuntimeError(f"A coluna de classe '{label_col}' deve ser numérica (0/1/2). Valores encontrados: {present}")

    present = [c for c in sorted(present) if c in (0, 1, 2)]
    if len(present) == 0:
        raise RuntimeError(f"Nenhuma classe 0/1/2 encontrada na coluna '{label_col}'.")

    # cria os DFs globais normalizados por classe
    for _c in present:
        globals()[f'DF_GLOBAL_NORM_C{_c}'] = base_df_norm[base_df_norm[label_col] == _c].reset_index(drop=True)

    # se alguma classe ausente, deixe como DataFrame vazio (para evitar NameError)
    for _c in [0, 1, 2]:
        if globals().get(f'DF_GLOBAL_NORM_C{_c}', None) is None:
            globals()[f'DF_GLOBAL_NORM_C{_c}'] = pd.DataFrame()

# -------------------- Funções auxiliares --------------------
def run_kmedoids_exemplars(X, k, metric="euclidean", random_state=0):
    if len(X) < 3 or k is None or k < 2 or k > (len(X) - 1):
        return np.array([]), np.array([]), np.nan, None
    model = KMedoids(n_clusters=int(k), metric=metric, random_state=random_state)
    labels = model.fit_predict(X)
    meds = model.medoid_indices_
    cost = getattr(model, "inertia_", np.nan)
    return meds, labels, cost, model

def _build_X_np(df_c):
    if df_c is None or not isinstance(df_c, pd.DataFrame) or df_c.empty:
        return np.empty((0, 0)), []
    feats = [col for col in df_c.columns
             if col not in EXCLUDE_COLUMNS
             and col != TARGET_COLUMN
             and pd.api.types.is_numeric_dtype(df_c[col])]
    X_np = df_c[feats].to_numpy() if len(feats) > 0 else np.empty((len(df_c), 0))
    return X_np, feats

# Quais classes temos efetivamente?
classes = [c for c in [0, 1, 2] if (f'DF_GLOBAL_NORM_C{c}' in globals())]
if len(classes) == 0:
    raise RuntimeError('Nenhum DF_GLOBAL_NORM_C{c} encontrado. Verifique a etapa anterior de preparação por classe.')

X_np_by_class = {}
features_by_class = {}
for c in classes:
    dfc = globals()[f'DF_GLOBAL_NORM_C{c}']
    X_np_by_class[c], features_by_class[c] = _build_X_np(dfc)

# -------------------- Curvas de silhueta por classe --------------------
def compute_silhouette_df(X, k_min=2, k_max=15, metric="euclidean", random_state=0):
    n = len(X)
    if n < 3:
        return pd.DataFrame(columns=["k", "silhouette"])  # insuficiente
    k_min_eff = max(2, int(k_min))
    k_max_eff = min(int(k_max), n - 1)
    if k_min_eff > k_max_eff:
        return pd.DataFrame(columns=["k", "silhouette"])  # range vazio
    rows = []
    for k in range(k_min_eff, k_max_eff + 1):
        meds, labels, _, _ = run_kmedoids_exemplars(X, k, metric=metric, random_state=random_state)
        if labels.size == 0:
            continue
        try:
            sil = float(silhouette_score(X, labels, metric=metric))
        except Exception:
            sil = np.nan
        rows.append({"k": int(k), "silhouette": sil})
    return pd.DataFrame(rows)

K_MIN = 2
K_MAX = 10

df_sil_curve = {}  # classe -> DataFrame
for c in classes:
    Xc = X_np_by_class[c]
    df_sil_curve[c] = compute_silhouette_df(Xc, k_min=K_MIN, k_max=K_MAX, metric="euclidean", random_state=RANDOM_STATE) if Xc.size else pd.DataFrame(columns=["k","silhouette"])
    if not df_sil_curve[c].empty:
        df_sil_curve[c].to_csv(os.path.join(RESULTS_DIR, f'silhouette_curve_class_{c}.csv'), index=False)

# -------------------- k recomendado (argmax silhueta) --------------------
def pick_best_k(df_curve: pd.DataFrame):
    if df_curve is None or df_curve.empty:
        return None
    df2 = df_curve.dropna(subset=['silhouette'])
    if df2.empty:
        return None
    row = df2.sort_values(['silhouette','k'], ascending=[False, True]).iloc[0]
    return int(row['k'])

k_best_by_class = {c: pick_best_k(df_sil_curve[c]) for c in classes}

# -------------------- k usuário (N_CLUSTERS) --------------------
if 'N_CLUSTERS' not in globals() or not isinstance(N_CLUSTERS, dict):
    raise RuntimeError('N_CLUSTERS (k do usuário) não definido como dict.')

k_user_by_class = {c: int(N_CLUSTERS.get(c, k_best_by_class.get(c, 2))) for c in classes}

def _sanitize_k(X_np, k_desired):
    n = len(X_np)
    if n < 3 or k_desired is None:
        return None
    return max(2, min(int(k_desired), n - 1))

for c in classes:
    Xc = X_np_by_class[c]
    k_user_by_class[c] = _sanitize_k(Xc, k_user_by_class[c]) if Xc.size else None
    kb = k_best_by_class.get(c, None)
    k_best_by_class[c] = _sanitize_k(Xc, kb) if kb is not None else None

N_CLUSTERS_SILHUET = {c: k_best_by_class[c] for c in classes}

# -------------------- Clustering (user/best) --------------------
labels_user = {}
labels_best = {}
sample_sil_user = {}
sample_sil_best = {}

for c in classes:
    Xc = X_np_by_class[c]
    # user
    if k_user_by_class[c] is not None:
        _, labels_user[c], _, _ = run_kmedoids_exemplars(Xc, k_user_by_class[c], metric='euclidean', random_state=RANDOM_STATE)
        try:
            sample_sil_user[c] = silhouette_samples(Xc, labels_user[c], metric='euclidean')
        except Exception:
            sample_sil_user[c] = None
    else:
        labels_user[c] = np.array([])
        sample_sil_user[c] = None
    # best
    if k_best_by_class[c] is not None:
        _, labels_best[c], _, _ = run_kmedoids_exemplars(Xc, k_best_by_class[c], metric='euclidean', random_state=RANDOM_STATE)
        try:
            sample_sil_best[c] = silhouette_samples(Xc, labels_best[c], metric='euclidean')
        except Exception:
            sample_sil_best[c] = None
    else:
        labels_best[c] = np.array([])
        sample_sil_best[c] = None

# -------------------- Plot helpers --------------------
def plot_silhouette_curve(ax, df_curve, titulo, k_line_used=None, k_line_best=None):
    if df_curve is None or df_curve.empty:
        ax.text(0.5, 0.5, 'Sem dados suficientes', ha='center', va='center')
        ax.set_axis_off()
        return
    ax.plot(df_curve['k'], df_curve['silhouette'], marker='o', label='Silhueta')
    ax.set_title(titulo)
    ax.set_xlabel('k')
    ax.set_ylabel('Score silhueta')
    ax.grid(alpha=0.3)
    if k_line_best is not None:
        ax.axvline(k_line_best, linestyle=':', color='green', alpha=0.8, label=f'k best={k_line_best}')
    if k_line_used is not None:
        ax.axvline(k_line_used, linestyle='--', color='red',   alpha=0.8, label=f'k usado={k_line_used}')
    ax.set_xticks(df_curve['k'].tolist())
    ax.legend(fontsize=8)

# -------------------- Figura recomendado (best) — grade n_classes x 2 --------------------
fig_rows = len(classes)
fig_best, axes_best = plt.subplots(fig_rows, 2, figsize=(12, 4.0 * fig_rows), constrained_layout=True)
if fig_rows == 1:
    axes_best = np.array([axes_best])  # garante 2D

for i, c in enumerate(classes):
    # hist clusters
    ax_hist = axes_best[i, 0]
    lbls = labels_best[c]
    kb = k_best_by_class[c]
    if lbls.size and kb is not None:
        ax_hist.hist(lbls, bins=np.arange(kb + 1) - 0.5, rwidth=0.8)
        ax_hist.set_title(f'Classe {c} (k_best={kb}) - Distribuição')
    else:
        ax_hist.text(0.5, 0.5, 'Sem clusters', ha='center')
    # curva silhueta
    ax_curve = axes_best[i, 1]
    plot_silhouette_curve(ax_curve, df_sil_curve[c], f'Classe {c} — silhueta × k', k_line_used=k_user_by_class[c], k_line_best=kb)

fig_best.savefig(os.path.join(IMG_DIR, 'silhouette_best_grid.png'), dpi=DPI, bbox_inches='tight', pad_inches=0)
plt.show()

# -------------------- Figura usuário — grade n_classes x 2 --------------------
fig_user, axes_user = plt.subplots(fig_rows, 2, figsize=(12, 4.0 * fig_rows), constrained_layout=True)
if fig_rows == 1:
    axes_user = np.array([axes_user])  # garante 2D

for i, c in enumerate(classes):
    # hist clusters
    ax_hist = axes_user[i, 0]
    lbls = labels_user[c]
    ku = k_user_by_class[c]
    if lbls.size and ku is not None:
        ax_hist.hist(lbls, bins=np.arange(ku + 1) - 0.5, rwidth=0.8)
        ax_hist.set_title(f'Classe {c} (k_user={ku}) - Distribuição')
    else:
        ax_hist.text(0.5, 0.5, 'Sem clusters', ha='center')
    # curva silhueta
    ax_curve = axes_user[i, 1]
    plot_silhouette_curve(ax_curve, df_sil_curve[c], f'Classe {c} — silhueta × k (ref)', k_line_used=ku, k_line_best=k_best_by_class[c])

fig_user.savefig(os.path.join(IMG_DIR, 'silhouette_user_grid.png'), dpi=DPI, bbox_inches='tight', pad_inches=0)
plt.show()

# -------------------- Resumos --------------------
def _cluster_counts(labels, k):
    labels = np.asarray(labels).ravel()
    if labels.size == 0 or k is None:
        return []
    return [int(x) for x in np.bincount(labels.astype(int), minlength=int(k))[:int(k)]]

def _sil_at_k(df_curve, k):
    if df_curve is None or df_curve.empty or k is None:
        return np.nan
    row = df_curve[df_curve['k'] == int(k)]
    if row.empty:
        return np.nan
    return float(row.iloc[0]['silhouette'])

def _resumo_classe_silhouette(class_id, X_np, labels_used, k_used, k_best, df_curve):
    try:
        sil_used = float(silhouette_score(X_np, labels_used, metric='euclidean')) if (labels_used is not None and np.size(labels_used) > 0) else np.nan
    except Exception:
        sil_used = np.nan
    sil_best_curve = _sil_at_k(df_curve, k_best)
    tam_used = _cluster_counts(labels_used, k_used)
    n = len(X_np)
    if (k_best is not None) and (k_best != k_used) and (n > int(k_best)) and (int(k_best) >= 2):
        try:
            _, labels_best_aux, _, _ = run_kmedoids_exemplars(X_np, int(k_best), metric='euclidean', random_state=RANDOM_STATE)
            tam_best = _cluster_counts(labels_best_aux, k_best)
        except Exception:
            tam_best = []
    else:
        tam_best = tam_used if (k_best == k_used) else []
    return {
        'classe': int(class_id),
        'k_usado_diagrama': int(k_used) if k_used is not None else None,
        'silhueta_usada': sil_used,
        'k_sugerido_curva': int(k_best) if k_best is not None else None,
        'silhueta_sugerida': sil_best_curve,
        'tam_clusters_usado': tam_used,
        'tam_clusters_k_sugerido': tam_best,
    }

res_l_best, res_l_user = [], []
for c in classes:
    Xc = X_np_by_class[c]
    # recomendado
    res_l_best.append(_resumo_classe_silhouette(c, Xc, labels_best[c], k_best_by_class[c], k_best_by_class[c], df_sil_curve[c]))
    # usuário
    res_l_user.append(_resumo_classe_silhouette(c, Xc, labels_user[c], k_user_by_class[c], k_best_by_class[c], df_sil_curve[c]))

df_resumo_silhueta_best = pd.DataFrame(res_l_best)
df_resumo_silhueta_user = pd.DataFrame(res_l_user)

if not df_resumo_silhueta_user.empty:
    def _perc_diff(row):
        su = row.get('silhueta_usada', np.nan)
        ss = row.get('silhueta_sugerida', np.nan)
        if ss is None or np.isnan(ss) or ss == 0:
            return np.nan
        return 100.0 * (su - ss) / ss
    df_resumo_silhueta_user['perc_diff_silhueta_usada_vs_sugerida'] = df_resumo_silhueta_user.apply(_perc_diff, axis=1)

df_resumo_silhueta = pd.concat([
    df_resumo_silhueta_best.assign(tipo='recomendado'),
    df_resumo_silhueta_user.assign(tipo='usuario')
], ignore_index=True)

# salvar CSVs de resumo
if not df_resumo_silhueta_best.empty:
    df_resumo_silhueta_best.to_csv(os.path.join(RESULTS_DIR, 'resumo_silhueta_usado_vs_sugerido_recomendado.csv'), index=False, encoding='utf-8')
if not df_resumo_silhueta_user.empty:
    df_resumo_silhueta_user.to_csv(os.path.join(RESULTS_DIR, 'resumo_silhueta_usado_vs_sugerido_usuario.csv'), index=False, encoding='utf-8')
if not df_resumo_silhueta.empty:
    df_resumo_silhueta.to_csv(os.path.join(RESULTS_DIR, 'resumo_silhueta_usado_vs_sugerido.csv'), index=False, encoding='utf-8')

# feedback
if 'display' in globals():
    if not df_resumo_silhueta_best.empty:
        display(df_resumo_silhueta_best)
    if not df_resumo_silhueta_user.empty:
        display(df_resumo_silhueta_user)

print('Silhueta concluída. Arquivos de curvas e resumos salvos em:', RESULTS_DIR)
print('Imagens salvas em:', IMG_DIR)


In [ ]:
# ==== Análise Elbow: clusters por Elbow (usuário e recomendado) ====
# Suporta 2 ou 3 classes (0,1,2), com cor fixa por classe.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from sklearn_extra.cluster import KMedoids

RESULTS_DIR = str(OUT_CSV)
IMG_DIR = str(OUT_IMG) if 'OUT_IMG' in globals() else os.path.join(RESULTS_DIR, 'images')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

# -------------------- PREP: garantir splits por classe normalizados --------------------
def _pick_label_column(df):
    """Escolhe a coluna de classe na ordem: TARGET_COLUMN -> y_pred -> y -> y_true."""
    candidates = []
    if 'TARGET_COLUMN' in globals() and isinstance(TARGET_COLUMN, str):
        candidates.append(TARGET_COLUMN)
    candidates += ['y_pred', 'y', 'y_true']
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(
        f"Não encontrei coluna-alvo em {candidates}. Colunas disponíveis: {list(df.columns)[:30]} ..."
    )

def _get_global_norm_df():
    """Obtém DF_GLOBAL_NORM; se ausente, monta a partir de X_TRAIN_NORM + X_TEST_NORM."""
    if 'DF_GLOBAL_NORM' in globals() and isinstance(DF_GLOBAL_NORM, pd.DataFrame) and not DF_GLOBAL_NORM.empty:
        return DF_GLOBAL_NORM.copy()
    if ('X_TRAIN_NORM' in globals() and isinstance(X_TRAIN_NORM, pd.DataFrame) and not X_TRAIN_NORM.empty and
        'X_TEST_NORM'  in globals() and isinstance(X_TEST_NORM,  pd.DataFrame) and not X_TEST_NORM.empty):
        return pd.concat([X_TRAIN_NORM, X_TEST_NORM], axis=0, ignore_index=True)
    raise RuntimeError("DF_GLOBAL_NORM não encontrado e não foi possível montar a partir de X_TRAIN_NORM/X_TEST_NORM.")

# Se os DF_GLOBAL_NORM_C* não existirem, crie-os agora
need_build = False
for _c in [0, 1, 2]:
    if globals().get(f'DF_GLOBAL_NORM_C{_c}', None) is None:
        need_build = True
        break

if need_build:
    base_df_norm = _get_global_norm_df()
    label_col = _pick_label_column(base_df_norm)

    present = pd.unique(base_df_norm[label_col].dropna())
    try:
        present = np.array(present, dtype=int)
    except Exception:
        raise RuntimeError(f"A coluna de classe '{label_col}' deve ser numérica (0/1/2). Valores encontrados: {present}")

    present = [c for c in sorted(present) if c in (0, 1, 2)]
    if len(present) == 0:
        raise RuntimeError(f"Nenhuma classe 0/1/2 encontrada na coluna '{label_col}'.")

    for _c in present:
        globals()[f'DF_GLOBAL_NORM_C{_c}'] = base_df_norm[base_df_norm[label_col] == _c].reset_index(drop=True)

    for _c in [0, 1, 2]:
        if globals().get(f'DF_GLOBAL_NORM_C{_c}', None) is None:
            globals()[f'DF_GLOBAL_NORM_C{_c}'] = pd.DataFrame()

# -------------------- Utils --------------------
def run_kmedoids_exemplars(X, k, metric="euclidean", random_state=0):
    if len(X) < 3 or k is None or k < 2 or k > len(X) - 1:
        return np.array([]), np.array([]), np.nan, None
    model = KMedoids(n_clusters=int(k), metric=metric, random_state=random_state)
    labels = model.fit_predict(X)
    meds = model.medoid_indices_
    cost = getattr(model, "inertia_", np.nan)  # custo total do KMedoids
    return meds, labels, float(cost), model

def compute_elbow_df(X, k_min=2, k_max=15, metric="euclidean", random_state=0):
    n = len(X)
    if n < 3:
        return pd.DataFrame(columns=["k", "cost", "sse"])
    k_min_eff = max(2, int(k_min))
    k_max_eff = min(int(k_max), n - 1)
    if k_min_eff > k_max_eff:
        return pd.DataFrame(columns=["k", "cost", "sse"])
    rows = []
    for k in range(k_min_eff, k_max_eff + 1):
        meds, labels, cost, _ = run_kmedoids_exemplars(X, k, metric=metric, random_state=random_state)
        if metric == "euclidean" and labels.size and np.size(meds) > 0:
            D = pairwise_distances(X, X[meds], metric=metric)
            sse = float(np.sum((D[np.arange(len(X)), labels]) ** 2))
        else:
            sse = np.nan
        rows.append({"k": int(k), "cost": cost, "sse": sse})
    return pd.DataFrame(rows)

def _knee_point_k(df, y_col="cost"):
    if df is None or df.empty or df.shape[0] < 3:
        return None
    xs = df["k"].to_numpy(dtype=float)
    ys = df[y_col].to_numpy(dtype=float)
    x0, x1 = xs.min(), xs.max()
    y0, y1 = np.nanmin(ys), np.nanmax(ys)
    if x1 == x0 or not np.isfinite(y0) or not np.isfinite(y1) or y1 == y0:
        return None
    xn = (xs - x0) / (x1 - x0)
    yn = (ys - y0) / (y1 - y0)
    a = np.array([0.0, 0.0]); b = np.array([1.0, 1.0])
    ba = b - a
    ba_norm = np.linalg.norm(ba)
    pts = np.stack([xn, yn], axis=1)
    cross = np.abs(pts[:, 0] * ba[1] - pts[:, 1] * ba[0])
    dists = cross / (ba_norm + 1e-12)
    idx = int(np.nanargmax(dists))
    return int(xs[idx])

def recommend_k_elbow(df, y_col="cost", low_gain_alpha=0.5):
    if df is None or df.empty:
        return {"k_knee": None, "k_lowgain": None, "recommended": None}
    k_knee = _knee_point_k(df, y_col=y_col)
    ks = df["k"].to_numpy()
    ys = df[y_col].to_numpy(dtype=float)
    gains = -np.diff(ys)
    ks_g = ks[1:]
    if gains.size > 0 and np.isfinite(np.nanmean(gains)):
        mean_gain = np.nanmean(gains)
        thresh = low_gain_alpha * mean_gain
        idxs = np.where(gains < thresh)[0]
        k_lowgain = int(ks_g[idxs[0]]) if idxs.size > 0 else None
    else:
        k_lowgain = None
    if k_knee is not None:
        k_rec = k_knee
    elif k_lowgain is not None:
        k_rec = k_lowgain
    else:
        k_rec = int(ks[-1])
    return {"k_knee": k_knee, "k_lowgain": k_lowgain, "recommended": k_rec}

def _sanitize_k(X_np, k_desired):
    n = len(X_np)
    if n < 3 or k_desired is None:
        return None
    return max(2, min(int(k_desired), n - 1))

def _cluster_counts(labels, k):
    labels = np.asarray(labels).ravel()
    if labels.size == 0 or k is None:
        return []
    return [int(x) for x in np.bincount(labels.astype(int), minlength=int(k))[:int(k)]]

def _cost_at_k(df_elb_var: pd.DataFrame, k: int):
    if df_elb_var is None or df_elb_var.empty or k is None:
        return np.nan
    row = df_elb_var.loc[df_elb_var["k"] == int(k), "cost"]
    if row.empty:
        return np.nan
    return float(row.iloc[0])

# -------------------- dados por classe (dinâmico para 0/1/2) --------------------
def _build_X_np(df_c):
    if df_c is None or not isinstance(df_c, pd.DataFrame) or df_c.empty:
        return np.empty((0, 0)), []
    feats = [col for col in df_c.columns
             if col not in EXCLUDE_COLUMNS
             and col != TARGET_COLUMN
             and pd.api.types.is_numeric_dtype(df_c[col])]
    X_np = df_c[feats].to_numpy() if len(feats) > 0 else np.empty((len(df_c), 0))
    return X_np, feats

classes = [c for c in [0,1,2] if (f'DF_GLOBAL_NORM_C{c}' in globals())]
if len(classes) == 0:
    raise RuntimeError('Nenhum DF_GLOBAL_NORM_C{c} encontrado. Verifique a etapa anterior de preparação por classe.')

X_np_by_class = {}
features_by_class = {}
for c in classes:
    dfc = globals()[f'DF_GLOBAL_NORM_C{c}']
    X_np_by_class[c], features_by_class[c] = _build_X_np(dfc)

# -------------------- curvas de Elbow por classe --------------------
K_MIN, K_MAX = 2, 10
df_elbow_curve = {}
for c in classes:
    Xc = X_np_by_class[c]
    df_elbow_curve[c] = compute_elbow_df(Xc, k_min=K_MIN, k_max=K_MAX, metric="euclidean", random_state=RANDOM_STATE) if Xc.size else pd.DataFrame(columns=["k","cost","sse"])
    if not df_elbow_curve[c].empty:
        df_elbow_curve[c].to_csv(os.path.join(RESULTS_DIR, f"elbow_curve_class_{c}.csv"), index=False)

# -------------------- k recomendado e k do usuário por classe --------------------
k_rec_by_class = {c: (recommend_k_elbow(df_elbow_curve[c], y_col="cost")["recommended"] if not df_elbow_curve[c].empty else None) for c in classes}
k_rec_by_class = {c: _sanitize_k(X_np_by_class[c], k_rec_by_class[c]) if k_rec_by_class[c] is not None else None for c in classes}

# Exporta recomendação
N_CLUSTERS_ELBOW_RECOMENDED = {c: k_rec_by_class[c] for c in classes}

# N_CLUSTERS_ELBOW do usuário (mantém lógica original; preenche com rec se faltar)
if 'N_CLUSTERS_ELBOW' not in globals() or not isinstance(N_CLUSTERS_ELBOW, dict):
    N_CLUSTERS_ELBOW = {c: (k_rec_by_class[c] if k_rec_by_class[c] is not None else 2) for c in classes}

k_user_by_class = {c: _sanitize_k(X_np_by_class[c], N_CLUSTERS_ELBOW.get(c)) for c in classes}

# -------------------- KMedoids (user e recomendado) --------------------
labels_rec_by_class = {}
labels_user_by_class = {}
cost_rec_by_class = {}
cost_user_by_class = {}

for c in classes:
    Xc = X_np_by_class[c]
    # recomendado
    if k_rec_by_class[c] is not None and Xc.size:
        _, labels_rec_by_class[c], cost_rec_by_class[c], _ = run_kmedoids_exemplars(Xc, k_rec_by_class[c], metric="euclidean", random_state=RANDOM_STATE)
    else:
        labels_rec_by_class[c], cost_rec_by_class[c] = np.array([]), np.nan
    # usuário
    if k_user_by_class[c] is not None and Xc.size:
        _, labels_user_by_class[c], cost_user_by_class[c], _ = run_kmedoids_exemplars(Xc, k_user_by_class[c], metric="euclidean", random_state=RANDOM_STATE)
    else:
        labels_user_by_class[c], cost_user_by_class[c] = np.array([]), np.nan

# -------------------- cores por classe --------------------
CLASS_COLORS = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:green'}

# -------------------- plotting helpers --------------------
def plot_elbow_vs_k_axis(ax, df, titulo, k_line=None, line_label="k usado", color='tab:blue'):
    if df is None or df.empty:
        ax.text(0.5, 0.5, "Sem dados suficientes", ha="center", va="center")
        ax.set_axis_off()
        return
    ax.plot(df["k"], df["cost"], marker="o", label="Custo (Elbow)", color=color)
    ax.set_title(titulo)
    ax.set_xlabel("k")
    ax.set_ylabel("Custo total")
    ax.grid(alpha=0.3)
    k_star = _knee_point_k(df, y_col="cost")
    if k_star is not None:
        ax.axvline(k_star, linestyle=":", alpha=0.8, label=f"k knee={k_star}", color=color)
    if k_line is not None:
        ax.axvline(int(k_line), linestyle="--", alpha=0.8, label=f"{line_label}={k_line}", color=color)
    ax.set_xticks(df["k"].tolist())
    ax.legend(fontsize=8)

# -------------------- Figuras (recomendado e usuário) em grade n_classes x 2 --------------------
def _plot_grid(kind='recomendado', fname='elbow_grid_rec.png'):
    fig_rows = len(classes)
    fig, axes = plt.subplots(fig_rows, 2, figsize=(12, 4.0 * fig_rows), constrained_layout=True)
    if fig_rows == 1:
        axes = np.array([axes])  # garante 2D

    for i, c in enumerate(classes):
        color = CLASS_COLORS.get(c, 'tab:blue')
        # Histogram
        ax_hist = axes[i, 0]
        if kind == 'recomendado':
            labels = labels_rec_by_class[c]
            kshow  = k_rec_by_class[c]
            title_l = f"Classe {c} (k_rec={kshow}) - Distribuição"
        else:
            labels = labels_user_by_class[c]
            kshow  = k_user_by_class[c]
            title_l = f"Classe {c} (k_user={kshow}) - Distribuição"

        if labels.size and kshow is not None:
            ax_hist.hist(labels, bins=np.arange(kshow + 1) - 0.5, rwidth=0.8, color=color)
            ax_hist.set_title(title_l)
        else:
            ax_hist.text(0.5, 0.5, 'Sem clusters', ha='center')
        # Curva Elbow
        ax_curve = axes[i, 1]
        df_curve = df_elbow_curve[c]
        k_line = k_rec_by_class[c] if kind == 'recomendado' else k_user_by_class[c]
        label_line = "k elbow rec" if kind == 'recomendado' else "k elbow usuário"
        plot_elbow_vs_k_axis(ax_curve, df_curve, f'Classe {c} — elbow × k', k_line=k_line, line_label=label_line, color=color)

    fig.savefig(os.path.join(IMG_DIR, fname), dpi=DPI, bbox_inches='tight', pad_inches=0)
    plt.show()

# Plota e salva
_plot_grid(kind='recomendado', fname='elbow_best_grid.png')
_plot_grid(kind='usuario',    fname='elbow_user_grid.png')

# -------------------- Resumos por classe --------------------
def _resumo_classe_elbow(class_id, X_np, k_usado, df_elb_var, labels_usados):
    custo_usado = _cost_at_k(df_elb_var, k_usado)
    if not np.isfinite(custo_usado) and (k_usado is not None):
        _, _, custo_usado, _ = run_kmedoids_exemplars(X_np, k_usado, metric="euclidean", random_state=RANDOM_STATE)
    tam_usado = _cluster_counts(labels_usados, k_usado) if labels_usados.size else []
    k_sug = recommend_k_elbow(df_elb_var, y_col="cost")["recommended"] if (df_elb_var is not None and not df_elb_var.empty) else None
    custo_sugerido = _cost_at_k(df_elb_var, k_sug) if k_sug is not None else np.nan
    if k_sug is not None and len(X_np) >= (k_sug + 1):
        _, labels_sug, _, _ = run_kmedoids_exemplars(X_np, k_sug, metric="euclidean", random_state=RANDOM_STATE)
        tam_sug = _cluster_counts(labels_sug, k_sug)
    else:
        tam_sug = []
    return {
        "classe": int(class_id),
        "k_usado_diagrama": int(k_usado) if k_usado is not None else None,
        "custo_usado": float(custo_usado) if np.isfinite(custo_usado) else np.nan,
        "k_sugerido_curva": int(k_sug) if k_sug is not None else None,
        "custo_sugerido": float(custo_sugerido) if np.isfinite(custo_sugerido) else np.nan,
        "tam_clusters_usado": tam_usado,
        "tam_clusters_k_sugerido": tam_sug,
    }

# Tabelas (recomendado e usuário)
res_l_rec, res_l_user = [], []
for c in classes:
    Xc = X_np_by_class[c]
    # recomendado
    if k_rec_by_class[c] is not None:
        res_l_rec.append(_resumo_classe_elbow(c, Xc, k_rec_by_class[c], df_elbow_curve[c], labels_rec_by_class[c]))
    # usuário
    if k_user_by_class[c] is not None:
        res_l_user.append(_resumo_classe_elbow(c, Xc, k_user_by_class[c], df_elbow_curve[c], labels_user_by_class[c]))

df_resumo_elbow_rec  = pd.DataFrame(res_l_rec)
df_resumo_elbow_user = pd.DataFrame(res_l_user)

# % diferença custo (usado vs sugerido)
def _calc_perc_diff_elbow(row):
    c_usado = row.get("custo_usado", np.nan)
    c_sugerido = row.get("custo_sugerido", np.nan)
    if c_sugerido is None or not np.isfinite(c_sugerido) or c_sugerido == 0:
        return np.nan
    return 100.0 * (c_usado - c_sugerido) / c_sugerido

if not df_resumo_elbow_user.empty:
    df_resumo_elbow_user["perc_diff_custo_usado_vs_sugerido"] = df_resumo_elbow_user.apply(_calc_perc_diff_elbow, axis=1)

# Salva CSVs
if not df_resumo_elbow_rec.empty:
    df_resumo_elbow_rec.to_csv(os.path.join(RESULTS_DIR, "resumo_elbow_usado_vs_sugerido_recomendado.csv"), index=False, encoding="utf-8")
if not df_resumo_elbow_user.empty:
    df_resumo_elbow_user.to_csv(os.path.join(RESULTS_DIR, "resumo_elbow_usado_vs_sugerido_usuario.csv"), index=False, encoding="utf-8")

# Feedback visual (opcional)
if 'display' in globals():
    if not df_resumo_elbow_rec.empty:
        display(df_resumo_elbow_rec)
    if not df_resumo_elbow_user.empty:
        display(df_resumo_elbow_user)

print('Elbow concluído. Curvas e resumos salvos em:', RESULTS_DIR)
print('Imagens salvas em:', IMG_DIR)


In [ ]:
# === PCA 2D + Voronoi (medoids) com Fallback — K=2..8 (Held-out Gap) — SEPARADO POR CLASSE (y_pred) ===
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import pairwise_distances
from matplotlib.colors import ListedColormap
from matplotlib.patches import Polygon

try:
    from scipy.spatial import Voronoi
    HAS_SCIPY = True
except Exception:
    Voronoi = None
    HAS_SCIPY = False

RESULTS_DIR = str(OUT_IMG) if 'OUT_IMG' in globals() else 'resultados_explainable_clustering'
os.makedirs(RESULTS_DIR, exist_ok=True)

CMAP_CLUSTERS      = plt.cm.tab20
CLUSTER_BG_ALPHA   = 0.18
CLUSTER_LINE_COLOR = "#056559"
MEDOID_COLOR       = "#056559"
# Mantém cores fixas por classe; adiciona classe 2
POINT_COLORS       = {0: "#0B900F", 1: "#0C78EC", 2: "#E67E22"}

# ------------------ Helpers para garantir splits por classe (0/1/2) ------------------
def _pick_label_column(df):
    """Escolhe a coluna de classe na ordem: TARGET_COLUMN -> y_pred -> y -> y_true."""
    candidates = []
    if 'TARGET_COLUMN' in globals() and isinstance(TARGET_COLUMN, str):
        candidates.append(TARGET_COLUMN)
    candidates += ['y_pred', 'y', 'y_true']
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(
        f"Não encontrei coluna-alvo em {candidates}. Colunas disponíveis: {list(df.columns)[:30]} ..."
    )

def _get_global_norm_df():
    """Obtém DF_GLOBAL_NORM; se ausente, monta de X_TRAIN_NORM + X_TEST_NORM."""
    if 'DF_GLOBAL_NORM' in globals() and isinstance(DF_GLOBAL_NORM, pd.DataFrame) and not DF_GLOBAL_NORM.empty:
        return DF_GLOBAL_NORM.copy()
    if ('X_TRAIN_NORM' in globals() and isinstance(X_TRAIN_NORM, pd.DataFrame) and not X_TRAIN_NORM.empty and
        'X_TEST_NORM'  in globals() and isinstance(X_TEST_NORM,  pd.DataFrame) and not X_TEST_NORM.empty):
        return pd.concat([X_TRAIN_NORM, X_TEST_NORM], axis=0, ignore_index=True)
    raise RuntimeError("DF_GLOBAL_NORM não encontrado e não foi possível montar a partir de X_TRAIN_NORM/X_TEST_NORM.")

# Se os DF_GLOBAL_NORM_C* não existirem, crie-os agora (0/1/2)
_need_build = False
for _c in [0, 1, 2]:
    if globals().get(f'DF_GLOBAL_NORM_C{_c}', None) is None:
        _need_build = True
        break

if _need_build:
    _base_df_norm = _get_global_norm_df()
    _label_col = _pick_label_column(_base_df_norm)

    _present = pd.unique(_base_df_norm[_label_col].dropna())
    try:
        _present = np.array(_present, dtype=int)
    except Exception:
        raise RuntimeError(f"A coluna de classe '{_label_col}' deve ser numérica (0/1/2). Valores encontrados: {_present}")

    _present = [c for c in sorted(_present) if c in (0, 1, 2)]
    if len(_present) == 0:
        raise RuntimeError(f"Nenhuma classe 0/1/2 encontrada na coluna '{_label_col}'.")

    for _c in _present:
        globals()[f'DF_GLOBAL_NORM_C{_c}'] = _base_df_norm[_base_df_norm[_label_col] == _c].reset_index(drop=True)

    # Garante objetos vazios para ausentes
    for _c in [0, 1, 2]:
        if globals().get(f'DF_GLOBAL_NORM_C{_c}', None) is None:
            globals()[f'DF_GLOBAL_NORM_C{_c}'] = pd.DataFrame()

# ------------------ Funções do Voronoi/plot (inalteradas, só reaproveitadas) ------------------
def _voronoi_finite_polygons_2d(vor, radius: float = None):
    if vor.points.shape[1] != 2:
        raise ValueError("Suporta apenas Voronoi 2D.")
    new_regions = []
    new_vertices = vor.vertices.tolist()
    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max() * 2
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))
    for p1, region_idx in enumerate(vor.point_region):
        region = vor.regions[region_idx]
        if -1 not in region and len(region) > 0:
            new_regions.append(region)
            continue
        ridges = all_ridges.get(p1, [])
        new_region = [v for v in region if v >= 0]
        for p2, v1, v2 in ridges:
            if v2 < 0 or v1 < 0:
                t = vor.points[p2] - vor.points[p1]
                nrm = np.linalg.norm(t)
                if nrm == 0:
                    continue
                t /= nrm
                n = np.array([-t[1], t[0]])
                midpoint = (vor.points[p1] + vor.points[p2]) / 2
                direction = np.sign(np.dot(midpoint - center, n)) * n
                base = vor.vertices[v1 if v1 >= 0 else v2]
                far_point = base + direction * radius
                new_vertices.append(far_point.tolist())
                new_region.append(len(new_vertices) - 1)
        vs = np.asarray([new_vertices[v] for v in new_region])
        if vs.size == 0:
            new_regions.append([])
            continue
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = [v for _, v in sorted(zip(angles, new_region))]
        new_regions.append(new_region)
    return new_regions, np.asarray(new_vertices)

def _clip_poly_to_bbox(poly_xy: np.ndarray, bbox):
    x_min, x_max, y_min, y_max = bbox
    def clip_edge(points, inside_fn, intersect_fn):
        if len(points) == 0:
            return []
        out = []
        S = points[-1]
        for E in points:
            if inside_fn(E):
                if inside_fn(S):
                    out.append(E)
                else:
                    out.append(intersect_fn(S, E)); out.append(E)
            else:
                if inside_fn(S):
                    out.append(intersect_fn(S, E))
            S = E
        return out
    def inside_left(p):   return p[0] >= x_min
    def inside_right(p):  return p[0] <= x_max
    def inside_bottom(p): return p[1] >= y_min
    def inside_top(p):    return p[1] <= y_max
    def intersect_left(S, E):
        dx, dy = E[0] - S[0], E[1] - S[1]
        if dx == 0: return np.array([x_min, S[1]])
        t = (x_min - S[0]) / dx
        return np.array([x_min, S[1] + t * dy])
    def intersect_right(S, E):
        dx, dy = E[0] - S[0], E[1] - S[1]
        if dx == 0: return np.array([x_max, S[1]])
        t = (x_max - S[0]) / dx
        return np.array([x_max, S[1] + t * dy])
    def intersect_bottom(S, E):
        dx, dy = E[0] - S[0], E[1] - S[1]
        if dy == 0: return np.array([S[0], y_min])
        t = (y_min - S[1]) / dy
        return np.array([S[0] + t * dx, y_min])
    def intersect_top(S, E):
        dx, dy = E[0] - S[0], E[1] - S[1]
        if dy == 0: return np.array([S[0], y_max])
        t = (y_max - S[1]) / dy
        return np.array([S[0] + t * dx, y_max])
    pts = [np.array(p) for p in poly_xy]
    pts = clip_edge(pts, inside_left,   intersect_left)
    pts = clip_edge(pts, inside_right,  intersect_right)
    pts = clip_edge(pts, inside_bottom, intersect_bottom)
    pts = clip_edge(pts, inside_top,    intersect_top)
    return np.array(pts)

def _is_degenerate(points2d: np.ndarray) -> bool:
    if not HAS_SCIPY:
        return True
    if points2d.shape[0] < 3:
        return True
    if np.unique(points2d, axis=0).shape[0] < 3:
        return True
    return np.linalg.matrix_rank(points2d - points2d.mean(axis=0)) < 2

def _draw_voronoi_or_nn(ax, med2d: np.ndarray, bbox, colors, line_color, alpha, radius=None):
    x_min, x_max, y_min, y_max = bbox
    nK = med2d.shape[0]
    cmap_disc = ListedColormap(colors)
    diag = np.hypot(x_max - x_min, y_max - y_min)
    radius_eff = (10.0 * diag) if (radius is None) else float(radius)
    try:
        if _is_degenerate(med2d):
            raise RuntimeError("Degenerate or SciPy unavailable")
        vor = Voronoi(med2d)
        regions, vertices = _voronoi_finite_polygons_2d(vor, radius=radius_eff)
        for i, region in enumerate(regions):
            if not region:
                continue
            poly_xy = vertices[region]
            poly_clip = _clip_poly_to_bbox(poly_xy, (x_min, x_max, y_min, y_max))
            if isinstance(poly_clip, np.ndarray) and len(poly_clip) >= 3:
                poly = Polygon(poly_clip, closed=True,
                               facecolor=colors[i % len(colors)],
                               edgecolor=line_color, linewidth=1.2, alpha=alpha)
                ax.add_patch(poly)
    except Exception:
        RES = 300
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, RES), np.linspace(y_min, y_max, RES))
        grid2d = np.c_[xx.ravel(), yy.ravel()]
        Zk = pairwise_distances(grid2d, med2d, metric="euclidean").argmin(axis=1).reshape(xx.shape)
        levels = np.arange(-0.5, nK + 0.5, 1.0)
        ax.contourf(xx, yy, Zk, levels=levels, cmap=cmap_disc, alpha=alpha)
        ax.contour(xx, yy, Zk, levels=levels, colors=line_color, linewidths=1.2, alpha=0.9)

from sklearn_extra.cluster import KMedoids

def run_kmedoids_exemplars(X, k, metric="euclidean", random_state=0):
    if len(X) < 3 or k < 2 or k > len(X)-1:
        return np.array([]), np.array([]), np.nan, None
    model = KMedoids(n_clusters=k, metric=metric, random_state=random_state)
    labels = model.fit_predict(X)
    meds = model.medoid_indices_
    cost = model.inertia_
    return meds, labels, cost, model

def heldout_gap_kmedoids(X_np, K, metric="euclidean", test_size=0.3, n_repeats=3, random_state=42):
    rng = np.random.default_rng(random_state)
    gaps = []
    n = len(X_np)
    if n < 4:
        return np.nan
    for r in range(n_repeats):
        idx = np.arange(n)
        rng.shuffle(idx)
        split = max(1, int(n * (1 - test_size)))
        tr_idx, va_idx = idx[:split], idx[split:]
        if len(va_idx) == 0 or len(tr_idx) < 2:
            continue
        Xtr, Xva = X_np[tr_idx], X_np[va_idx]
        meds_tr, labels_tr, cost_tr, D_tr = run_kmedoids_exemplars(
            Xtr, K, metric="euclidean", random_state=int(random_state + r)
        )
        P = Xtr[meds_tr]
        train_cost = pairwise_distances(Xtr, P, metric=metric).min(axis=1).mean()
        val_cost   = pairwise_distances(Xva, P, metric=metric).min(axis=1).mean()
        gaps.append(val_cost - train_cost)
    return float(np.mean(gaps)) if gaps else np.nan

def annotate_metrics_inside(ax, sil, db, ch, gap):
    txt = f"Sil={sil:.3f} | DB={db:.3f} | CH={ch:.1f} | Gap={gap:.3f}"
    ax.text(0.5, 0.02, txt, transform=ax.transAxes, ha="center", va="bottom",
            fontsize=9, bbox=dict(boxstyle="round,pad=0.2",
                                  facecolor="white", alpha=0.65, edgecolor="none"))

def draw_panel(ax, title, X_2d, labels_k=None, medoids_idx_k=None, plot_color="tab:blue", add_legend=False):
    ax.scatter(X_2d[:,0], X_2d[:,1], s=18, alpha=0.75, c=plot_color,
               label="amostras" if add_legend else None)
    if (medoids_idx_k is not None) and (len(medoids_idx_k) > 0):
        meds_idx = np.asarray(medoids_idx_k, dtype=int)
        med2d = X_2d[meds_idx]
        if np.unique(med2d, axis=0).shape[0] < med2d.shape[0]:
            rng = np.random.default_rng(0)
            med2d = med2d + 1e-9 * rng.standard_normal(med2d.shape)
        PAD = 0.5
        x_min, x_max = X_2d[:,0].min()-PAD, X_2d[:,0].max()+PAD
        y_min, y_max = X_2d[:,1].min()-PAD, X_2d[:,1].max()+PAD
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
        nK = med2d.shape[0]
        colors = [CMAP_CLUSTERS(i / max(1, nK-1)) for i in range(nK)]
        diag = np.hypot(x_max - x_min, y_max - y_min)
        bbox = (x_min, x_max, y_min, y_max)
        _draw_voronoi_or_nn(ax, med2d, bbox, colors, CLUSTER_LINE_COLOR, CLUSTER_BG_ALPHA, radius=10.0 * diag)
        ax.scatter(med2d[:,0], med2d[:,1], s=120, marker="X", c=MEDOID_COLOR,
                   label="medoids" if add_legend else None)
    ax.set_title(title)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.grid(True, linestyle="--", alpha=0.35)
    if add_legend:
        ax.legend(ncol=1, fontsize=8, framealpha=0.9)

def painel_por_classe(X_sub: pd.DataFrame, class_id: int, filename: str, point_color="tab:blue"):
    if len(X_sub) < 3:
        print(f"[Classe {class_id}] poucos dados ({len(X_sub)}); painel não gerado.")
        return
    # remove colunas excluídas apenas aqui
    X_sub = X_sub.drop(columns=EXCLUDE_COLUMNS, errors='ignore')
    # PCA em features numéricas
    X_vals = X_sub.select_dtypes(include=[np.number]).values
    if X_vals.shape[1] < 1 or X_vals.shape[0] < 3:
        print(f"[Classe {class_id}] dados insuficientes após seleção numérica; painel não gerado.")
        return
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_vals)
    X_np = X_vals

    # Ks
    K_row1 = [2, 3, 4]
    K_row2 = [5, 6, 7, 8]
    maxK = max(2, len(X_sub) - 1)
    K_row1 = [k for k in K_row1 if k <= maxK]
    K_row2 = [k for k in K_row2 if k <= maxK]

    # roda PAM e métricas
    pam_runs = {}
    metrics_by_K = {}
    for K in (K_row1 + K_row2):
        meds_k, labels_k, cost_k, D_k = run_kmedoids_exemplars(X_np, K, metric="euclidean", random_state=42)
        pam_runs[K] = (labels_k, meds_k)
        sil = silhouette_score(X_np, labels_k, metric="euclidean")
        db  = davies_bouldin_score(X_np, labels_k)
        ch  = calinski_harabasz_score(X_np, labels_k)
        gap = heldout_gap_kmedoids(X_np, K, metric="euclidean", test_size=0.3, n_repeats=3, random_state=123)
        metrics_by_K[K] = (sil, db, ch, gap)

    # figura 2x4
    fig, axes = plt.subplots(2, 4, figsize=(24, 12), squeeze=False)
    ax00 = axes[0,0]
    ax00.scatter(X_2d[:,0], X_2d[:,1], s=18, alpha=0.75, c=point_color, label=f"classe {class_id}")
    ax00.set_title(f"PCA 2D — pontos (classe {class_id})")
    ax00.set_xlabel("PC1"); ax00.set_ylabel("PC2"); ax00.grid(True, linestyle="--", alpha=0.35)
    ax00.legend(ncol=1, fontsize=8, framealpha=0.9)

    for j, K in enumerate(K_row1, start=1):
        ax = axes[0, j]
        labels_k, meds_k = pam_runs[K]
        draw_panel(ax, f"PCA 2D — PAM K={K}", X_2d, labels_k=labels_k, medoids_idx_k=meds_k, plot_color=point_color)
        sil, db, ch, gap = metrics_by_K[K]
        annotate_metrics_inside(ax, sil, db, ch, gap)

    for j, K in enumerate(K_row2):
        ax = axes[1, j]
        labels_k, meds_k = pam_runs[K]
        draw_panel(ax, f"PCA 2D — PAM K={K}", X_2d, labels_k=labels_k, medoids_idx_k=meds_k, plot_color=point_color)
        sil, db, ch, gap = metrics_by_K[K]
        annotate_metrics_inside(ax, sil, db, ch, gap)

    for j in range(len(K_row2), 4):
        axes[1, j].axis("off")

    plt.suptitle(f"PCA 2D + Voronoi (medoids) — Classe {class_id} — K=2..8 com métricas e Held-out Gap", fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    outpath = os.path.join(RESULTS_DIR, filename)
    plt.savefig(outpath, dpi=DPI)
    plt.show()
    print(f"[Classe {class_id}] Figura salva em: {outpath}")

# ------------------ Seleção dinâmica de classes e execução ------------------
# Usa DF_GLOBAL_NORM_C0/1/2 (se existirem e não vazios)
_classes = [c for c in [0,1,2] if (f'DF_GLOBAL_NORM_C{c}' in globals())]
if len(_classes) == 0:
    raise RuntimeError('Nenhum DF_GLOBAL_NORM_C{c} encontrado. Verifique a etapa anterior de preparação por classe.')

for _c in _classes:
    _dfc = globals()[f'DF_GLOBAL_NORM_C{_c}']
    if isinstance(_dfc, pd.DataFrame) and not _dfc.empty:
        _color = POINT_COLORS.get(_c, "tab:blue")
        painel_por_classe(_dfc.copy(), class_id=_c, filename=f"pca2d_voronoi_k2a8_classe{_c}.png", point_color=_color)
    else:
        print(f"[Classe {_c}] DataFrame vazio — painel não gerado.")


In [ ]:
# === PCA 3D Voronoi para 2 ou 3 classes (0,1,2) — gera novo HTML ===
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display

# ---------- coleta das classes disponíveis ----------
classes_presentes = []
for c in [0, 1, 2]:
    if f'DF_GLOBAL_NORM_C{c}' in globals():
        df_tmp = globals()[f'DF_GLOBAL_NORM_C{c}']
        if isinstance(df_tmp, pd.DataFrame) and not df_tmp.empty:
            classes_presentes.append(c)

if len(classes_presentes) < 2:
    raise RuntimeError("São necessárias ao menos 2 classes com dados para gerar o PCA 3D Voronoi.")

# dicionário de DataFrames por classe
dfs_por_classe = {c: globals()[f'DF_GLOBAL_NORM_C{c}'].copy() for c in classes_presentes}

# ---------- seleção de features numéricas comuns (excluindo colunas de controle) ----------
exclude = set((EXCLUDE_COLUMNS if 'EXCLUDE_COLUMNS' in globals() else []) + [TARGET_COLUMN] if 'TARGET_COLUMN' in globals() else [])
cols_num_por_classe = {}
for c, dfc in dfs_por_classe.items():
    cols_num_por_classe[c] = [col for col in dfc.select_dtypes(include=[np.number]).columns if col not in exclude]

# interseção de colunas numéricas comuns a TODAS as classes presentes
cols_common = list(set.intersection(*[set(cols_num_por_classe[c]) for c in classes_presentes]))
if len(cols_common) == 0:
    raise RuntimeError("Não há colunas numéricas comuns entre as classes (após exclusões).")

# DataFrames processados só com colunas comuns
dfs_proc = {c: dfs_por_classe[c][cols_common].copy() for c in classes_presentes}

# ---------- PCA 3D global sobre a concatenação (mantendo ordem das classes) ----------
df_all = pd.concat([dfs_proc[c] for c in classes_presentes], axis=0, ignore_index=True)

X_np = df_all.values
scaler = StandardScaler().fit(X_np)  # mantém a lógica original (escalonamento para PCA)
X_scaled = scaler.transform(X_np)
pca = PCA(n_components=3, random_state=42).fit(X_scaled)
X_3d = pca.transform(X_scaled)

# fatia projeções por classe, na mesma ordem usada para concatenação
sizes = [len(dfs_proc[c]) for c in classes_presentes]
splits = np.cumsum(sizes)
X3_por_classe = {}
ini = 0
for c, end in zip(classes_presentes, splits):
    X3_por_classe[c] = X_3d[ini:end]
    ini = end

# ---------- util: obter k por classe, respeitando SILHUET / N_CLUSTERS ----------
def get_k_for_class(cid: int) -> int:
    if 'SILHUET' in globals() and SILHUET is not None:
        if isinstance(SILHUET, dict):
            if str(cid) in SILHUET:
                try: return max(1, int(SILHUET[str(cid)]))
                except: pass
            if cid in SILHUET:
                try: return max(1, int(SILHUET[cid]))
                except: pass
    if 'N_CLUSTERS' in globals() and isinstance(N_CLUSTERS, dict):
        try: return max(1, int(N_CLUSTERS.get(cid, 1)))
        except: pass
    return 1

# ---------- fallback mínimo para run_kmedoids_exemplars, se necessário ----------
if 'run_kmedoids_exemplars' not in globals():
    from sklearn_extra.cluster import KMedoids
    def run_kmedoids_exemplars(X, k, metric="euclidean", random_state=0):
        if len(X) < 3 or k < 2 or k > len(X)-1:
            return np.array([]), np.array([]), np.nan, None
        model = KMedoids(n_clusters=int(k), metric=metric, random_state=random_state)
        labels = model.fit_predict(X)
        meds = model.medoid_indices_
        cost = getattr(model, "inertia_", np.nan)
        return meds, labels, float(cost), model

# ---------- paleta de cores por classe (consistente com outros blocos) ----------
POINT_COLORS = {0: '#0B900F', 1: '#0C78EC', 2: '#E67E22'}  # 0=verde, 1=azul, 2=laranja

# ---------- helpers de visual e voronoi 3D via planos paralelos ----------
GRID_RES     = 28
PLANE_PCTS   = (5, 95)
POINT_SIZE   = 3
MEDOID_SIZE  = 5
SURF_OPACITY     = 0.6
SURF_OPACITY_K1  = 0.5
PALETTE = ["#e41a1c","#377eb8","#4daf4a","#984ea3","#ff7f00",
           "#ffff33","#a65628","#f781bf","#999999","#66c2a5"]

def colorscale_discrete(colors):
    n = len(colors)
    if n == 1:
        return [[0.0, colors[0]], [1.0, colors[0]]]
    return [[i/(n-1), c] for i, c in enumerate(colors)]

def add_classe_voronoi(fig, X_cls_vals, X3_cls, class_id: int, col: int, point_color='red'):
    Kc = get_k_for_class(class_id)
    npts = len(X_cls_vals)
    if npts < 1:
        print(f"[Classe {class_id}] sem pontos. Pulando.")
        return
    if Kc >= 2 and npts < Kc:
        print(f"[Classe {class_id}] poucos dados ({npts}) para k={Kc}. Pulando.")
        return

    meds_k, labels_k, _, _ = run_kmedoids_exemplars(X_cls_vals, Kc, metric="euclidean", random_state=42)
    labels_k = np.asarray(labels_k).astype(int)
    med3d = X3_cls[np.asarray(meds_k, int)]

    # pontos
    fig.add_trace(
        go.Scatter3d(
            x=X3_cls[:,0], y=X3_cls[:,1], z=X3_cls[:,2],
            mode='markers', name=f'classe {class_id}',
            marker=dict(size=POINT_SIZE, color=point_color, opacity=0.85)
        ),
        row=1, col=col
    )
    # medoids
    fig.add_trace(
        go.Scatter3d(
            x=med3d[:,0], y=med3d[:,1], z=med3d[:,2],
            mode='markers+text', name=f'medoids c{class_id}',
            marker=dict(size=MEDOID_SIZE, symbol='x', color='black',
                        opacity=1.0, line=dict(width=1, color='white')),
            text=[f"M{class_id}-{i}" for i in range(len(med3d))],
            textposition='top center', textfont=dict(size=10, color='black')
        ),
        row=1, col=col
    )

    # planos "voronoi" por vizinho mais próximo aos medoids
    PAD = 0.5
    xs = np.linspace(X3_cls[:,0].min()-PAD, X3_cls[:,0].max()+PAD, GRID_RES)
    ys = np.linspace(X3_cls[:,1].min()-PAD, X3_cls[:,1].max()+PAD, GRID_RES)
    z0, z1 = np.percentile(X3_cls[:,2], PLANE_PCTS)
    if np.isclose(z0, z1):  # evita plano degenerado
        z1 = z1 + 1e-6

    CS = colorscale_discrete(PALETTE[:max(1, Kc)])

    def surface_voronoi(zplane, surf_name):
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        grid3 = np.c_[XX.ravel(), YY.ravel(), np.full(XX.size, zplane)]
        diff = grid3[:, None, :] - med3d[None, :, :]
        d2 = np.sum(diff*diff, axis=2)
        pred = np.argmin(d2, axis=1)
        pred_img = pred.reshape(XX.shape).astype(float)
        return go.Surface(
            x=XX, y=YY, z=np.full_like(XX, zplane),
            surfacecolor=pred_img, cmin=0, cmax=max(1, Kc-1),
            colorscale=CS, showscale=False, opacity=SURF_OPACITY, name=surf_name
        )

    if Kc >= 2:
        fig.add_trace(surface_voronoi(z0, f"plano z@{PLANE_PCTS[0]}%"), row=1, col=col)
        fig.add_trace(surface_voronoi(z1, f"plano z@{PLANE_PCTS[1]}%"), row=1, col=col)
    else:
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        const0 = np.zeros_like(XX, dtype=float)
        for zplane, name in [(z0, f"plano z@{PLANE_PCTS[0]}%"), (z1, f"plano z@{PLANE_PCTS[1]}%")]:
            fig.add_trace(
                go.Surface(
                    x=XX, y=YY, z=np.full_like(XX, zplane),
                    surfacecolor=const0, cmin=0, cmax=1,
                    colorscale=CS, showscale=False, opacity=SURF_OPACITY_K1, name=name
                ),
                row=1, col=col
            )

# ---------- construção dinâmica do subplot (1 x N) ----------
ncols = len(classes_presentes)
subplot_titles = tuple([f"Classe {c} · k={get_k_for_class(c)} · Voronoi (medoides)" for c in classes_presentes])

fig = make_subplots(
    rows=1, cols=ncols,
    specs=[[{'type': 'scene'} for _ in range(ncols)]],
    horizontal_spacing=0.0 if ncols == 2 else 0.02,
    column_widths=[1.0/ncols]*ncols,
    subplot_titles=subplot_titles
)

# adicionar cada classe
for idx, c in enumerate(classes_presentes, start=1):
    add_classe_voronoi(
        fig,
        dfs_proc[c].values,
        X3_por_classe[c],
        class_id=c,
        col=idx,
        point_color=POINT_COLORS.get(c, 'gray')
    )

# layout básico
fig.update_layout(
    title="PCA 3D · Regiões por Voronoi (medoides) — classes lado a lado",
    autosize=True, width=None, height=560,
    margin=dict(l=0, r=0, t=34, b=0, pad=0),
    legend=dict(bgcolor="rgba(255,255,255,0.85)")
)

# configura cada cena (scene, scene2, scene3, ...)
for i in range(1, ncols+1):
    scene_key = 'scene' if i == 1 else f'scene{i}'
    fig.update_layout(**{
        scene_key: dict(
            domain=dict(x=[(i-1)/ncols, i/ncols], y=[0, 1]),
            xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3",
            xaxis=dict(backgroundcolor="rgb(245,245,245)"),
            yaxis=dict(backgroundcolor="rgb(245,245,245)"),
            zaxis=dict(backgroundcolor="rgb(245,245,245)")
        )
    })

# saída
out_name = f"pca3d_voronoi_{ncols}colunas.html"
out_html = str(OUT_IMG / out_name)
fig.write_html(out_html, include_plotlyjs="cdn", full_html=True, config={"responsive": True})
print(f"✅ HTML salvo em: {out_html}")

# exibe
try:
    pio.renderers.default = "plotly_mimetype"
    fig.show()
except Exception:
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


In [ ]:
# COTOVELO (por cluster) — usa a variável **SILHUET** para definir K por classe
# Suporta 2 ou 3 classes (0,1,2) e faz fallback se RF_PER_CLASS_PATHS não existir.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances

# --- Globais esperadas (usa valores existentes; senão usa defaults seguros) ---
RESULTS_DIR = str(OUT_CSV) if 'OUT_CSV' in globals() else 'resultados_explainable_clustering'
OUT_IMG_SAFE = str(OUT_IMG) if 'OUT_IMG' in globals() else RESULTS_DIR
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUT_IMG_SAFE, exist_ok=True)

TARGET_FRACS = [0.95, 0.75, 0.65, 0.50, 0.40, 0.30, 0.25, 0.05]
PALETTE = plt.cm.tab10
RANDOM_STATE = globals().get('RANDOM_STATE', 42)
DPI = globals().get('DPI', 300)
EXCLUDE_COLUMNS = globals().get('EXCLUDE_COLUMNS', [])
TARGET_COLUMN = globals().get('TARGET_COLUMN', 'y_pred')  # usado no fallback

assert 'run_kmedoids_exemplars' in globals(), 'Defina run_kmedoids_exemplars antes.'
if 'SILHUET' not in globals() or not isinstance(SILHUET, dict) or not SILHUET:
    raise RuntimeError("A variável SILHUET deve ser um dict não-vazio, ex.: {0:5, 1:3, 2:4}.")

# ----------------- Fallback para RF_PER_CLASS_PATHS -----------------
def _bootstrap_rf_per_class_paths():
    """
    Se RF_PER_CLASS_PATHS não existir, tenta construí-lo a partir de DF_GLOBAL_NORM (preferido) ou DF_GLOBAL,
    filtrando por TARGET_COLUMN (ex.: y_pred) e salvando CSVs em OUT_CSV/RESULTS_DIR.
    Retorna dict {classe: caminho_csv}.
    """
    # fonte: DF_GLOBAL_NORM -> DF_GLOBAL
    source_df = None
    if 'DF_GLOBAL_NORM' in globals() and isinstance(DF_GLOBAL_NORM, pd.DataFrame) and not DF_GLOBAL_NORM.empty:
        source_df = DF_GLOBAL_NORM.copy()
    elif 'DF_GLOBAL' in globals() and isinstance(DF_GLOBAL, pd.DataFrame) and not DF_GLOBAL.empty:
        source_df = DF_GLOBAL.copy()

    if source_df is None:
        raise RuntimeError(
            "RF_PER_CLASS_PATHS não encontrado e não há DF_GLOBAL_NORM/DF_GLOBAL disponíveis para o fallback."
        )

    if TARGET_COLUMN not in source_df.columns:
        # tenta usar 'y_pred' ou 'y' como alternativas padrão
        if 'y_pred' in source_df.columns:
            col = 'y_pred'
        elif 'y' in source_df.columns:
            col = 'y'
        else:
            raise RuntimeError(
                f"RF_PER_CLASS_PATHS não encontrado e a coluna alvo '{TARGET_COLUMN}' "
                "também não existe em DF_GLOBAL_NORM/DF_GLOBAL (nem 'y_pred'/'y')."
            )
    else:
        col = TARGET_COLUMN

    # filtra classes presentes (0/1/2)
    cls_presentes = sorted([int(c) for c in pd.unique(source_df[col]) if pd.notna(c)])
    cls_presentes = [c for c in cls_presentes if c in (0, 1, 2)]
    if not cls_presentes:
        raise RuntimeError(
            f"Nenhuma classe 0/1/2 encontrada na coluna '{col}' para montar RF_PER_CLASS_PATHS."
        )

    per_class_paths = {}
    for c in cls_presentes:
        df_c = source_df[source_df[col] == c].copy()
        if df_c.empty:
            continue
        # garante exclusão de colunas auxiliares irrelevantes no cluster (não obrigatório)
        out_path = os.path.join(RESULTS_DIR, f'rf_per_class_c{c}.csv')
        df_c.to_csv(out_path, index=False)
        per_class_paths[c] = out_path
    if not per_class_paths:
        raise RuntimeError("Falha ao criar CSVs por classe no fallback para RF_PER_CLASS_PATHS.")
    return per_class_paths

if 'RF_PER_CLASS_PATHS' not in globals() or not RF_PER_CLASS_PATHS:
    RF_PER_CLASS_PATHS = _bootstrap_rf_per_class_paths()
# -------------------------------------------------------------------

# ----------------- Preparar dados base / classes -----------------
frames = []
classes_lidas = []
for cls_val, path in RF_PER_CLASS_PATHS.items():
    df_tmp = pd.read_csv(path)
    df_tmp['__cls_tmp__'] = int(cls_val)  # força classe
    frames.append(df_tmp)
    classes_lidas.append(int(cls_val))

if not frames:
    raise RuntimeError('Nenhum CSV por classe encontrado em RF_PER_CLASS_PATHS.')

classes_lidas = sorted(list(set(classes_lidas)))
X_test = pd.concat(frames, axis=0, ignore_index=True)
y_pred = X_test['__cls_tmp__'].to_numpy().astype(int)

# Filtrar somente colunas numéricas válidas para distância, excluindo EXCLUDE_COLUMNS
exclude = set(EXCLUDE_COLUMNS) if isinstance(EXCLUDE_COLUMNS, (list, tuple, set)) else set()
num_cols = [c for c in X_test.columns if c not in exclude and pd.api.types.is_numeric_dtype(X_test[c])]
if '__cls_tmp__' in num_cols:
    num_cols.remove('__cls_tmp__')
if not num_cols:
    raise RuntimeError("Nenhuma feature numérica disponível após exclusões para calcular distâncias.")

X_num = X_test[num_cols].to_numpy(dtype=float, copy=True)

# ----------------- K_BY_CLASS vindo de SILHUET -----------------
def _sanitize_k_for_class(n_pts_class: int, k_desired: int):
    """Garante 2 <= k <= n-1 quando possível; caso contrário retorna None."""
    if n_pts_class < 3 or k_desired is None:
        return None
    k_desired = int(k_desired)
    return max(2, min(k_desired, n_pts_class - 1))

K_BY_CLASS = {}
for cls in sorted(np.unique(y_pred).tolist()):
    n_cls = int((y_pred == cls).sum())
    k_raw = SILHUET.get(int(cls), None)
    K_BY_CLASS[int(cls)] = _sanitize_k_for_class(n_cls, k_raw)

# ----------------- Helpers -----------------
def _r_sequence_kcenter_farthest(D_cluster: np.ndarray, seed_pos: int):
    """Sequência de raios r(m) (m=0..n-1) por farthest-first (k-center). m=0 usa o medoid como semente."""
    n = D_cluster.shape[0]
    radii = []
    centers = [int(seed_pos)]
    dist_to_centers = D_cluster[seed_pos].copy()
    radii.append(float(dist_to_centers.max()))
    for m in range(1, n):
        cand = int(np.argmax(dist_to_centers))
        centers.append(cand)
        dist_to_centers = np.minimum(dist_to_centers, D_cluster[cand])
        radii.append(float(dist_to_centers.max()))
    return np.array(radii, dtype=float)

# ----------------- Construir curvas por cluster -----------------
classes = sorted(np.unique(y_pred).tolist())  # p.ex. [0,1] ou [0,1,2]
per_class_cluster_curves = {}
summary_rows = []

for cls in classes:
    mask = (y_pred == cls)
    if not np.any(mask):
        continue

    Xc = X_num[mask]
    Kc = K_BY_CLASS.get(int(cls), None)

    if (Kc is None) or (Xc.shape[0] < max(3, Kc)):
        print(f"[Classe {cls}] poucos pontos para K={Kc}. Pulando.")
        continue

    meds_loc, labels_loc, _, _ = run_kmedoids_exemplars(Xc, Kc, metric='euclidean', random_state=RANDOM_STATE)
    meds_loc   = np.asarray(meds_loc, dtype=int)
    labels_loc = np.asarray(labels_loc, dtype=int)
    D_cls = pairwise_distances(Xc, metric='euclidean')

    per_class_cluster_curves[cls] = {}
    for c in np.sort(np.unique(labels_loc)):
        idxs_loc = np.where(labels_loc == c)[0]
        if len(idxs_loc) <= 1:
            continue

        # medoid local (fallback para o verdadeiro medoid do subcluster, se índice exceder)
        if c < len(meds_loc):
            med_loc = int(meds_loc[c])
        else:
            sub = D_cls[np.ix_(idxs_loc, idxs_loc)]
            med_loc = int(idxs_loc[np.argmin(sub.sum(axis=1))])

        # seed dentro do subcluster (mapeia para posição em idxs_loc)
        if med_loc in idxs_loc:
            seed_pos = int(np.where(idxs_loc == med_loc)[0][0])
        else:
            seed_pos = 0  # fallback

        # distância interna do cluster
        D_sub = D_cls[np.ix_(idxs_loc, idxs_loc)]

        # sequência de raios por farthest-first
        radii = _r_sequence_kcenter_farthest(D_sub, seed_pos=seed_pos)

        r_medoid = radii[0]
        r_norm = (radii / r_medoid) if r_medoid > 0 else np.zeros_like(radii)
        m_vals = np.arange(len(radii))
        n_pts = int(len(idxs_loc))

        # metas de redução do raio
        m_hits = {}
        for frac in TARGET_FRACS:
            thr = frac * r_medoid
            ok = np.where(radii <= thr)[0]
            m_star = int(ok[0]) if ok.size > 0 else int(m_vals[-1])
            m_hits[frac] = m_star

            pct_protos = 100.0 * m_star / n_pts               # exclui medoid
            pct_repr   = 100.0 * (m_star + 1) / n_pts         # inclui medoid

            summary_rows.append({
                'classe': int(cls),
                'cluster': int(c),
                'n_pontos_cluster': n_pts,
                'r_medoid': float(r_medoid),
                'alvo_frac': float(frac),
                'm_necessario_excl_medoid': int(m_star),
                'representantes_total_incl_medoid': int(m_star + 1),
                'raio_resultante': float(radii[m_star]),
                'raio_resultante_rel_medoid': float(r_norm[m_star]),
                'pct_prototipos_necessarios_%': float(pct_protos),
                'pct_representantes_incl_medoid_%': float(pct_repr),
            })

        per_class_cluster_curves[cls][int(c)] = dict(
            m_vals=m_vals, r_norm=r_norm, r_abs=radii, m_hits=m_hits
        )

# ----------------- Plot -----------------
if per_class_cluster_curves:
    n_rows = len(classes)
    ncols = max([len(per_class_cluster_curves.get(c, {})) for c in classes]) or 1
    fig, axes = plt.subplots(n_rows, ncols, figsize=(5*ncols, 4.2*n_rows), squeeze=False)

    for r, cls in enumerate(classes):
        clusters = sorted(per_class_cluster_curves.get(cls, {}).keys())
        for j in range(ncols):
            ax = axes[r, j]
            if j >= len(clusters):
                ax.axis('off')
                continue
            c = clusters[j]
            data = per_class_cluster_curves[cls][c]
            m_vals = data['m_vals']; r_norm = data['r_norm']
            color = PALETTE(j % PALETTE.N)

            ax.plot(m_vals, r_norm*100.0, marker='o', linewidth=2.0, color=color, label=f'c{c}')

            for frac in TARGET_FRACS:
                ax.axhline(y=100.0*frac, linestyle='--', linewidth=1.2, color='gray', alpha=0.7)
                m_star = data['m_hits'][frac]
                ax.scatter([m_star], [100.0*(data['r_abs'][m_star]/data['r_abs'][0])],
                           s=55, zorder=5, color=color, edgecolor='white')
                ax.annotate(f"m*={m_star}",
                            (m_star, 100.0*(data['r_abs'][m_star]/data['r_abs'][0])),
                            textcoords='offset points', xytext=(6,6), fontsize=9, color=color)

            ax.set_xlabel('m (protótipos, exclui medoid)')
            ax.set_ylabel('raio r(m) / r_medoid  (%)')
            ax.set_title(f'Classe {cls} · Cluster c{c}')
            ax.set_ylim(0, 105); ax.set_xlim(0, max(m_vals))
            ax.grid(True, linestyle='--', alpha=0.35)

    labels_targets = [f"meta: r ≤ {int(frac*100)}% do r_medoid" for frac in TARGET_FRACS]
    fig.legend(labels_targets, loc='upper center', ncol=len(TARGET_FRACS), framealpha=0.9)
    plt.tight_layout(rect=[0,0,1,0.93])

    out_png = os.path.join(OUT_IMG_SAFE, f'kcenter_curva_raio_vs_m_por_cluster_{n_rows}linhas.png')
    plt.savefig(out_png, dpi=DPI, bbox_inches='tight')
    plt.show()
    print(f'✅ Figura salva em: {out_png}')
else:
    print('Nenhuma curva gerada (verifique dados / clusters).')

# ----------------- Tabela resumo -----------------
df_m_needed = pd.DataFrame(summary_rows).sort_values(['classe','cluster','alvo_frac']).reset_index(drop=True)
out_csv = os.path.join(RESULTS_DIR, 'kcenter_m_necessario_por_cluster_por_meta.csv')
df_m_needed.to_csv(out_csv, index=False)
print(f'✅ Tabela salva em: {out_csv}')
print(df_m_needed.head())

# Estruturas auxiliares (inalteradas, agora suportam 3 classes naturalmente)
cols_keep = ['classe','cluster','n_pontos_cluster','m_necessario_excl_medoid',
             'representantes_total_incl_medoid','pct_prototipos_necessarios_%',
             'pct_representantes_incl_medoid_%','raio_resultante_rel_medoid']

requirements_by_target = {}
for frac in TARGET_FRACS:
    sub = df_m_needed[df_m_needed['alvo_frac'] == float(frac)].copy()
    sub = sub[cols_keep].sort_values(['classe','cluster']).reset_index(drop=True)
    requirements_by_target[float(frac)] = sub

requirements_by_target_compact = {}
for frac in TARGET_FRACS:
    sub = requirements_by_target[float(frac)]
    by_class = {}
    for cls in sorted(sub['classe'].unique()):
        sub_cls = sub[sub['classe'] == cls]
        by_class[int(cls)] = {int(r.cluster): int(r.m_necessario_excl_medoid) for r in sub_cls.itertuples(index=False)}
    requirements_by_target_compact[float(frac)] = by_class

wide_dfs_by_class = {}
df_tmp = df_m_needed.copy()
df_tmp['target_label'] = (df_tmp['alvo_frac'] * 100).round(0).astype(int).astype(str) + '%'
for cls in sorted(df_tmp['classe'].unique()):
    pivot = (df_tmp[df_tmp['classe'] == cls]
             .pivot(index='cluster', columns='target_label', values='m_necessario_excl_medoid')
             .sort_index(axis=0).sort_index(axis=1))
    wide_dfs_by_class[int(cls)] = pivot

# Exemplos de inspeção rápida
if 0.25 in requirements_by_target:
    print('\n▶ requirements_by_target[0.25]')
    print(requirements_by_target[0.25].to_string(index=False))
if 0.25 in requirements_by_target_compact:
    print('\n▶ requirements_by_target_compact[0.25]')
    print(requirements_by_target_compact[0.25])
if 1 in wide_dfs_by_class:
    print('\n▶ wide_dfs_by_class[classe=1]')
    print(wide_dfs_by_class[1].to_string())


In [ ]:
# Heatmap de m* por cluster/meta para cada classe (usando SILHUET para K_BY_CLASS se disponível)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# --- Fallbacks de caminhos e parâmetros visuais ---
OUT_IMG_SAFE = str(OUT_IMG) if 'OUT_IMG' in globals() else 'resultados_explainable_clustering/images'
os.makedirs(OUT_IMG_SAFE, exist_ok=True)
DPI_SAFE = globals().get('DPI', 300)

# --- Garantir estruturas que vêm de etapas anteriores ---
if 'wide_dfs_by_class' not in globals():
    wide_dfs_by_class = {}
if 'df_m_needed' not in globals():
    # cria vazio para não quebrar; sem df_m_needed, rótulos percentuais por cluster/meta não serão anotados com %
    df_m_needed = pd.DataFrame(columns=[
        'classe','cluster','n_pontos_cluster','alvo_frac',
        'm_necessario_excl_medoid','representantes_total_incl_medoid',
        'raio_resultante','raio_resultante_rel_medoid',
        'pct_prototipos_necessarios_%','pct_representantes_incl_medoid_%'
    ])

# --- Atualizar K_BY_CLASS a partir de SILHUET (se disponível) ---
if 'SILHUET' in globals() and isinstance(SILHUET, dict) and SILHUET:
    # usa apenas classes com k definido
    K_BY_CLASS = {int(k): int(v) for k, v in SILHUET.items() if v is not None}
else:
    # Se SILHUET não existe, tenta inferir k pela quantidade de clusters em wide_dfs_by_class
    K_BY_CLASS = {}
    for cls_key, dfw in wide_dfs_by_class.items():
        try:
            n_clusters_inferido = int(dfw.index.max()) + 1 if len(dfw.index) > 0 else 0
        except Exception:
            n_clusters_inferido = len(dfw.index)
        if n_clusters_inferido and n_clusters_inferido >= 1:
            # Para compatibilidade de layout de heatmap, exigimos no mínimo 1
            K_BY_CLASS[int(cls_key)] = int(n_clusters_inferido)

# Se ainda estiver vazio, tenta olhar df_m_needed
if not K_BY_CLASS:
    if not df_m_needed.empty:
        for cls in sorted(df_m_needed['classe'].dropna().unique().tolist()):
            sub = df_m_needed[df_m_needed['classe'] == cls]
            if not sub.empty:
                max_cluster = int(sub['cluster'].max())
                K_BY_CLASS[int(cls)] = max(1, max_cluster + 1)

# Se continuar vazio mesmo assim, não há nada a plotar
if not K_BY_CLASS:
    print("Nada a plotar: não há K_BY_CLASS nem dados de clusters para inferir.")
else:
    # --- Map (classe, cluster) -> n_pontos_cluster para cálculo do percentual ---
    cluster_sizes = {}
    if not df_m_needed.empty:
        tmp_sizes = df_m_needed[['classe','cluster','n_pontos_cluster']].drop_duplicates()
        for r in tmp_sizes.itertuples(index=False):
            try:
                cluster_sizes[(int(r.classe), int(r.cluster))] = int(r.n_pontos_cluster)
            except Exception:
                continue

    # --- Classes alvo: 0,1,2 se existirem; senão, as do K_BY_CLASS/wide_dfs_by_class ---
    classes_to_plot = [c for c in [0, 1, 2] if c in K_BY_CLASS or c in wide_dfs_by_class]
    if not classes_to_plot:
        classes_to_plot = sorted(set(list(K_BY_CLASS.keys()) + list(wide_dfs_by_class.keys())))

    # --- Loop por classe ---
    for cls in classes_to_plot:
        # df_wide = tabela (clusters x metas) com m* (criada na etapa anterior)
        df_wide = wide_dfs_by_class.get(cls)

        # número de clusters alvo: 1) SILHUET/K_BY_CLASS; 2) inferido de df_wide; 3) fallback 1
        if cls in K_BY_CLASS:
            n_clusters = int(K_BY_CLASS[cls])
        else:
            if df_wide is not None and not df_wide.empty:
                try:
                    n_clusters = int(df_wide.index.max()) + 1
                except Exception:
                    n_clusters = len(df_wide.index)
                n_clusters = max(1, n_clusters)
            else:
                n_clusters = 1

        # metas (colunas): vem de df_wide; se vazio, tenta extrair do df_m_needed (labels %)
        if df_wide is not None and not df_wide.empty:
            metas = list(df_wide.columns)
        else:
            # cria tabela vazia inicialmente
            metas = []
            if not df_m_needed.empty:
                sub = df_m_needed[df_m_needed['classe'] == cls].copy()
                if not sub.empty and 'alvo_frac' in sub.columns:
                    # converter em rótulos "NN%" como foi feito na etapa anterior
                    metas = (sub['alvo_frac'].dropna().unique() * 100)
                    metas = sorted([f"{int(round(x))}%" for x in metas])
            # se ainda vazio, segue vazio; heatmap mostrará nada com NaN
            df_wide = pd.DataFrame(np.nan, index=list(range(n_clusters)), columns=metas)

        # reindex para (0..n_clusters-1) nas linhas e metas detectadas nas colunas
        cluster_idx = list(range(n_clusters))
        df_wide = df_wide.reindex(index=cluster_idx, columns=metas, fill_value=np.nan)

        # --- Plot ---
        fig, ax = plt.subplots(figsize=(1.2 * max(1, len(metas)) + 2, 0.7 * n_clusters + 2))
        im = ax.imshow(df_wide.values, cmap='viridis', aspect='auto', interpolation='nearest')

        # ticks/labels
        ax.set_xticks(np.arange(len(metas)))
        ax.set_xticklabels(metas, rotation=45, ha='right')
        ax.set_yticks(np.arange(n_clusters))
        ax.set_yticklabels(cluster_idx)

        # faixa de cor p/ decidir cor do texto
        vmax = np.nanmax(df_wide.values) if np.isfinite(np.nanmax(df_wide.values)) else 0
        half = vmax / 2 if vmax > 0 else 0

        # anotações com valor absoluto e % (quando soubermos o tamanho do cluster)
        for i in range(n_clusters):
            for j in range(len(metas)):
                val = df_wide.values[i, j]
                if not np.isnan(val):
                    n_pts = cluster_sizes.get((int(cls), int(i)))
                    if n_pts and n_pts > 0:
                        pct = 100.0 * float(val) / float(n_pts)
                        label_txt = f'{int(val)} - {pct:.0f}%'
                    else:
                        label_txt = f'{int(val)}'
                    ax.text(
                        j, i, label_txt,
                        ha='center', va='center',
                        color='white' if (vmax > 0 and val > half) else 'black',
                        fontsize=10
                    )

        ax.set_xlabel('Meta: raio (%) do medoid')
        ax.set_ylabel('Cluster (definido em SILHUET)')
        ax.set_title(f'Classe {cls} — m* necessário por cluster/meta (SILHUET)')
        fig.colorbar(im, ax=ax, label='m* necessário')
        plt.tight_layout()

        out_png = os.path.join(OUT_IMG_SAFE, f'heatmap_mstar_por_cluster_meta_classe{cls}.png')
        plt.savefig(out_png, dpi=DPI_SAFE, bbox_inches='tight')
        plt.show()
        print(f'✅ Heatmap salvo em: {out_png}')


In [ ]:
# Parametrização da Compactação por CCluster 

REDUCAO_TARGET_POR_CLASSE_CLUSTER = {
    0: { 0: 0.50, 1: 0.30, 2:0.30, 3:0.40 },   # classe 0, cluster 0 -> 25%; classe 0, cluster 1 -> 35%
    1: { 0: 0.65, 1: 0.65, 2:0.65},         # classe 1, cluster 0 -> 20%
    2: { 0: 0.50, 1: 0.50, 2:0.50, 3:0.50 }    # classe 2, cluster 0 -> 30%; classe 2, cluster 2 -> 40%
}

In [ ]:
# === [MEDOID SEED] PAM + K-CENTER — DIAGNÓSTICO DE PROTÓTIPOS POR CLUSTER (0/1/2) ===
# Usa o mesmo dataset do COTOVELO e calcula internamente a curva r_percent(m) por cluster.
# Mantida a lógica original; apenas robustecido para 3 classes e com fallbacks seguros.

import os
import json
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances, silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ---------------- Parâmetros principais ----------------
TARGET_FRAC_FOR_REQUIREMENTS = 0.75  # ex.: 0.75 -> metas de 75% do raio do medoid

# NOVO: alvo opcional por classe/cluster, na MESMA lógica:
#   valor = parte do raio mantida (ex: 0.75 ou 75 -> 75% do raio)
REDUCAO_TARGET_POR_CLASSE_CLUSTER = globals().get("REDUCAO_TARGET_POR_CLASSE_CLUSTER", {})  # ### NOVO

# Pega m* por classe/cluster para a meta pedida; trata ausência com fallback vazio
if 'requirements_by_target_compact' not in globals() or TARGET_FRAC_FOR_REQUIREMENTS not in requirements_by_target_compact:
    M_PER_CLUSTER_INPUT = {}
else:
    M_PER_CLUSTER_INPUT = requirements_by_target_compact[TARGET_FRAC_FOR_REQUIREMENTS]

# K por classe (usa SILHUET; trata ausências)
if 'SILHUET' in globals() and isinstance(SILHUET, dict) and SILHUET:
    K_BY_CLASS = {int(k): int(v) for k, v in SILHUET.items() if v is not None}
else:
    raise RuntimeError("SILHUET não encontrado/sem conteúdo. Defina, ex.: SILHUET = {0:5, 1:3, 2:4}")

RANDOM_STATE = 42
PCA_COMPONENTS = 2
RESTRICT_TO_PCA_CELL = False
ALLOW_CROSS_CELL_ON_SHORTAGE = True

RESULTS_DIR = globals().get("RESULTS_DIR", str(OUT_CSV) if 'OUT_CSV' in globals() else "resultados_explainable_clustering")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------- Pré-requisitos ----------------
if 'X_test' not in globals():
    raise RuntimeError('X_test não encontrado. Execute o bloco COTOVELO (que monta X_test) antes.')

if 'EXCLUDE_COLUMNS' not in globals():
    EXCLUDE_COLUMNS = set()

if 'run_kmedoids_exemplars' not in globals():
    # Define um fallback mínimo (mesma assinatura) se não existir
    from sklearn_extra.cluster import KMedoids
    def run_kmedoids_exemplars(X, k, metric="euclidean", random_state=0):
        if len(X) < 3 or k < 2 or k > len(X) - 1:
            return np.array([]), np.array([]), np.nan, None
        model = KMedoids(n_clusters=int(k), metric=metric, random_state=random_state)
        labels = model.fit_predict(X)
        meds = model.medoid_indices_
        cost = getattr(model, "inertia_", np.nan)
        return meds, labels, cost, model

# ---------------- Funções auxiliares ----------------
def get_feature_matrix(df, exclude_cols):
    num_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
    if '__cls_tmp__' in num_cols:
        num_cols.remove('__cls_tmp__')
    return df[num_cols].to_numpy(dtype=float, copy=True), num_cols

def select_prototypes_farthest(X_np, m, seed_idx=None, random_state=RANDOM_STATE):
    """Farthest-first sobre a matriz de features X_np. Retorna índices locais (ordem de seleção)."""
    n = X_np.shape[0]
    if n <= 0:
        return []
    m = min(m, n)
    D = pairwise_distances(X_np, metric="euclidean")
    rng = np.random.default_rng(random_state)
    if seed_idx is None:
        seed_idx = int(rng.integers(low=0, high=n))
    selected = [int(seed_idx)]
    dist_min = D[selected[0], :].copy()
    while len(selected) < m:
        cand = int(np.argmax(dist_min))
        selected.append(cand)
        dist_min = np.minimum(dist_min, D[cand, :])
    return selected

def make_region_predicate(med2d, cluster_index):
    """Predicado: pertence à célula de Voronoi (em PCA-2D) do medoid do cluster 'cluster_index'."""
    med2d = np.asarray(med2d)
    c = int(cluster_index)
    def _predicate(P):
        Dp = pairwise_distances(np.asarray(P), med2d, metric="euclidean")
        return np.argmin(Dp, axis=1) == c
    return _predicate

# ---------------- Dados base ----------------
X_df_all = X_test.copy()
X_np_all, USED_FEATURE_COLS = get_feature_matrix(X_df_all, EXCLUDE_COLUMNS)

# Determina a classe de cada amostra (0/1/2). Prioriza y_pred, cai para __cls_tmp__ se existir.
if 'y_pred' in globals():
    y_color = np.asarray(y_pred).astype(int)
elif '__cls_tmp__' in X_test.columns:
    y_color = X_test['__cls_tmp__'].to_numpy(dtype=int, copy=True)
else:
    raise RuntimeError("Não encontrei y_pred nem __cls_tmp__ em X_test para saber as classes (0/1/2).")

# ---------------- Loop por classe (suporta 0,1,2) ----------------
results_by_class = {}
resumo_rows = []
diag_rows = []

# classes que iremos processar: interseção entre K_BY_CLASS e y_color; se vazio, usa K_BY_CLASS
classes_in_data = sorted(np.unique(y_color).tolist())
classes_to_process = sorted(set(K_BY_CLASS.keys()) & set(classes_in_data))
if not classes_to_process:
    classes_to_process = sorted(K_BY_CLASS.keys())

for cls in classes_to_process:
    cls = int(cls)
    if cls not in K_BY_CLASS:
        print(f"[Classe {cls}] sem K definido em SILHUET -> pulando.")
        continue

    mask = (y_color == cls)
    idx_global = np.where(mask)[0]
    X_cls = X_np_all[mask]
    Kc = int(K_BY_CLASS[cls])

    if len(X_cls) < max(3, Kc):
        print(f"[Classe {cls}] poucos dados ({len(X_cls)}) para k={Kc}. Pulando.")
        continue

    # Clustering PAM
    medoids_local, labels_local, cost_k, _ = run_kmedoids_exemplars(
        X_cls, Kc, metric="euclidean", random_state=RANDOM_STATE
    )
    medoids_local = np.asarray(medoids_local, dtype=int)
    labels_local  = np.asarray(labels_local,  dtype=int)

    # Distâncias no espaço original
    D_norm = pairwise_distances(X_cls, metric="euclidean")

    # PCA 2D (para métricas/plot auxiliares)
    pca_cls = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    X2d_cls = pca_cls.fit_transform(X_cls)
    D_pca = pairwise_distances(X2d_cls, metric="euclidean")
    med2d = X2d_cls[medoids_local]

    protos_only_by_cluster = {}
    centers_by_cluster = {}
    protos_locais_flat = []
    clusters = np.sort(np.unique(labels_local))

    # m* desejado (exclui medoid) vindo da tabela requirements_by_target_compact (se existir)
    m_target_by_cluster = M_PER_CLUSTER_INPUT.get(cls, {}) if isinstance(M_PER_CLUSTER_INPUT, dict) else {}

    for c in clusters:
        c = int(c)
        idxs_loc = np.where(labels_local == c)[0]
        if len(idxs_loc) == 0:
            protos_only_by_cluster[c] = []
            centers_by_cluster[c] = []
            continue

        # --- alvo efetivo de fração do raio (por classe/cluster ou global) ---  ### NOVO
        target_frac_eff = float(TARGET_FRAC_FOR_REQUIREMENTS)
        if isinstance(REDUCAO_TARGET_POR_CLASSE_CLUSTER, dict):
            frac_cls = REDUCAO_TARGET_POR_CLASSE_CLUSTER.get(cls, {})
            if isinstance(frac_cls, dict):
                frac_val = frac_cls.get(c, None)
                if frac_val is not None:
                    frac_val = float(frac_val)
                    # aceita 0–1 ou 0–100
                    if frac_val > 1.0:
                        frac_val = frac_val / 100.0
                    target_frac_eff = frac_val
        # ---------------------------------------------------------------

        # medoid local (garante medoid válido dentro do subcluster)
        if c < len(medoids_local):
            med_loc_global = int(medoids_local[c])
            if med_loc_global not in idxs_loc:
                # recalcula o medoid dentro do subcluster, por segurança
                sub = D_norm[np.ix_(idxs_loc, idxs_loc)]
                med_loc_global = int(idxs_loc[np.argmin(sub.sum(axis=1))])
        else:
            sub = D_norm[np.ix_(idxs_loc, idxs_loc)]
            med_loc_global = int(idxs_loc[np.argmin(sub.sum(axis=1))])

        # Candidatos: restrito à célula Voronoi (PCA) ou todo cluster
        if RESTRICT_TO_PCA_CELL:
            region_pred = make_region_predicate(med2d, c)
            X2d_cluster = X2d_cls[idxs_loc]
            inside_mask = region_pred(X2d_cluster)
            cand_loc_in_cluster = np.where(inside_mask)[0]
            cand_loc_in_Xcls = idxs_loc[cand_loc_in_cluster].tolist()
            obs = "modo=cell"
        else:
            cand_loc_in_Xcls = idxs_loc.tolist()
            obs = "modo=cluster"

        # Garante que o medoid esteja entre os candidatos
        if med_loc_global not in cand_loc_in_Xcls:
            cand_loc_in_Xcls = [med_loc_global] + cand_loc_in_Xcls

        # m* desejado p/ este cluster (exclui medoid). Se ausente, 0.
        m_target = int(m_target_by_cluster.get(c, 0))

        # Se restrito à célula e faltar candidato suficiente, amplia para o cluster
        if RESTRICT_TO_PCA_CELL:
            falta = max(0, m_target - max(0, len(cand_loc_in_Xcls) - 1))
            if falta > 0 and ALLOW_CROSS_CELL_ON_SHORTAGE:
                cand_loc_in_Xcls = idxs_loc.tolist()
                if med_loc_global not in cand_loc_in_Xcls:
                    cand_loc_in_Xcls = [med_loc_global] + cand_loc_in_Xcls
                obs = f"modo=cell -> ampliado p/ cluster (faltavam {falta} cand.)"
            elif falta > 0:
                obs = f"modo=cell (restrito; faltaram {falta} cand.)"

        # m efetivo não pode exceder qtd de candidatos (excluindo medoid)
        m_eff = min(m_target, max(0, len(cand_loc_in_Xcls) - 1))

        # Ordena candidatos pelo farthest-first (sendo o seed o medoid)
        cand_array = np.asarray(cand_loc_in_Xcls, dtype=int)
        seed_pos = int(np.where(cand_array == med_loc_global)[0][0])
        ordem_full = select_prototypes_farthest(
            X_cls[cand_array],
            m=len(cand_array),
            seed_idx=seed_pos,
            random_state=RANDOM_STATE
        )
        ordem_sem_seed_subset = [j for j in ordem_full if j != seed_pos]
        ordem_protos_locais = [int(cand_array[j]) for j in ordem_sem_seed_subset]

        # Curva r_percent(m): raio máximo coberto / raio no medoid
        r_percent_curve = []
        r100_norm = float(D_norm[idxs_loc, med_loc_global].max()) if len(idxs_loc) else 0.0
        for m_tmp in range(m_eff + 1):
            centers_tmp = [med_loc_global] + ordem_protos_locais[:m_tmp]
            dmin_tmp = D_norm[np.ix_(idxs_loc, centers_tmp)].min(axis=1) if len(idxs_loc) else np.array([0.0])
            r_cover_tmp = float(dmin_tmp.max())
            r_percent_tmp = (100.0 * r_cover_tmp / r100_norm) if r100_norm > 0 else 100.0
            r_percent_curve.append(r_percent_tmp)
        r_percent_ref = r_percent_curve[m_eff] if r_percent_curve else np.nan  # "cotovelo" p/ m usado

        # Selecionados finais para este cluster: medoid + m_eff protótipos
        protos_only_local = ordem_protos_locais[:m_eff]
        centers_by_cluster[c] = [med_loc_global] + protos_only_local
        protos_only_by_cluster[c] = protos_only_local
        protos_locais_flat.extend(protos_only_local)

        # Métricas de cobertura no espaço original
        centers_loc = centers_by_cluster[c]
        dmin_norm = D_norm[np.ix_(idxs_loc, centers_loc)].min(axis=1) if len(idxs_loc) else np.array([0.0])
        r_cover_norm = float(dmin_norm.max())
        r_percent = (100.0 * r_cover_norm / r100_norm) if r100_norm > 0 else 100.0

        # Info adicional (PCA)
        dmin_pca = D_pca[np.ix_(idxs_loc, centers_loc)].min(axis=1) if len(idxs_loc) else np.array([0.0])
        r100_pca = float(D_pca[idxs_loc, med_loc_global].max()) if len(idxs_loc) else 0.0
        r_cover_pca = float(dmin_pca.max())

        total_pontos = int(len(idxs_loc))
        total_selecionados = int(len(centers_loc))
        pct_sel = 100.0 * total_selecionados / total_pontos if total_pontos > 0 else np.nan

        resumo_rows.append({
            "classe": cls,
            "cluster": c,
            "total_pontos": total_pontos,
            "total_selecionados_incl_medoid": total_selecionados,
            "percentual_selecionados_%": pct_sel,
            "r100_radius": r100_norm,
            "r_protos_radius": r_cover_norm,
            "r_percent_%": r_percent,
            "r100_radius_pca": r100_pca,
            "r_protos_radius_pca": r_cover_pca,
            "m_target_excl_medoid": int(m_target),
            "m_usado_excl_medoid": int(m_eff),
            "observacao": obs,
            "alvo_frac": target_frac_eff    # ### NOVO: alvo efetivo por classe/cluster
        })

        pct_m_usado_mais_medoid = (100.0 * (m_eff + 1) / total_pontos) if total_pontos > 0 else np.nan
        ganho_compactacao = 100.0 - float(r_percent)
        diff_ref = float(r_percent) - float(r_percent_ref) if not np.isnan(r_percent_ref) else np.nan

        diag_rows.append({
            "classe": cls,
            "cluster": c,
            "alvo_percent_%": float(100.0 * target_frac_eff),  # ### NOVO: usa alvo efetivo
            "m_necessario_alvo": int(m_target),
            "m_usado": int(m_eff),
            "n_pontos_cluster": int(total_pontos),
            "pct_m_usado_mais_medoid_%": float(pct_m_usado_mais_medoid),
            "r_percent_realizado_%": float(r_percent),
            "ganho_compactacao_%": float(ganho_compactacao),
            "r_percent_cotovelo_ref_%": float(r_percent_ref) if not np.isnan(r_percent_ref) else np.nan,
            "diff_r_percent_vs_cotovelo": float(diff_ref) if not np.isnan(diff_ref) else np.nan,
            "ok(r<=alvo?)": bool(r_percent <= 100.0 * float(target_frac_eff)),  # ### NOVO
            "observacao": obs,
            "alvo_frac": target_frac_eff    # ### NOVO: também registrado no diag
        })

    # Métricas globais da partição por classe
    uniq = np.unique(labels_local)
    if len(uniq) > 1:
        sil = silhouette_score(X_cls, labels_local, metric="euclidean")
        db  = davies_bouldin_score(X_cls, labels_local)
        ch  = calinski_harabasz_score(X_cls, labels_local)
    else:
        sil = db = ch = np.nan

    results_by_class[cls] = dict(
        K=int(Kc),
        labels=labels_local,
        medoids_local=medoids_local,
        prototipos_local=np.asarray(protos_locais_flat, dtype=int),
        protos_by_cluster=protos_only_by_cluster,
        centers_by_cluster=centers_by_cluster,
        idx_global=idx_global,
        metrics=dict(silhouette=sil, davies_bouldin=db, calinski_harabasz=ch),
        pca_model=pca_cls,
        X2d=X2d_cls
    )

# ---------------- Salva DFs ----------------
df_resumo_cobertura = pd.DataFrame(resumo_rows).sort_values(["classe", "cluster"]).reset_index(drop=True)
df_diag = pd.DataFrame(diag_rows).sort_values(["classe", "cluster"]).reset_index(drop=True)

# Mantém compat com código antigo: se por algum motivo 'alvo_frac' não existir, usa global
if 'alvo_frac' not in df_resumo_cobertura.columns:
    df_resumo_cobertura['alvo_frac'] = TARGET_FRAC_FOR_REQUIREMENTS
if 'alvo_frac' not in df_diag.columns:
    df_diag['alvo_frac'] = TARGET_FRAC_FOR_REQUIREMENTS

cols_order = [
    'classe', 'cluster', 'alvo_percent_%', 'm_necessario_alvo', 'm_usado', 'n_pontos_cluster',
    'pct_m_usado_mais_medoid_%', 'r_percent_realizado_%', 'ganho_compactacao_%',
    'r_percent_cotovelo_ref_%', 'diff_r_percent_vs_cotovelo',
    'ok(r<=alvo?)', 'observacao', 'alvo_frac'
]
for col in cols_order:
    if col not in df_diag.columns:
        df_diag[col] = np.nan
df_diag = df_diag[cols_order]

df_resumo_path = os.path.join(RESULTS_DIR, "resumo_cobertura_requirements.csv")
df_diag_path   = os.path.join(RESULTS_DIR, "diagnostico_requirements.csv")
df_resumo_cobertura.to_csv(df_resumo_path, index=False)
df_diag.to_csv(df_diag_path, index=False)
print(f"✅ DF resumo salvo em: {df_resumo_path}")
print(f"✅ DF diagnóstico salvo em: {df_diag_path}")
print("\nPrévia do diagnóstico (primeiras linhas):")
print(df_diag.head().to_string(index=False))

# ---------------- Arrays de seleção ----------------
def _to_positional_indices(ids_iter, X_df):
    ids_list = list(ids_iter)
    if not ids_list:
        return np.array([], dtype=int)
    ids_arr = np.asarray(ids_list)
    n = len(X_df)
    if np.issubdtype(ids_arr.dtype, np.integer) and ids_arr.min() >= 0 and ids_arr.max() < n:
        return np.sort(np.unique(ids_arr.astype(int)))
    pos = X_df.index.get_indexer(ids_list)
    pos = pos[pos >= 0]
    return np.sort(np.unique(pos.astype(int)))

selected_labels = set()
ENTRADAS_SELECIONADAS_POR_CLASSE_CLUSTER = {}
iloc_to_class = {}
iloc_to_cluster = {}

for cls, res in results_by_class.items():
    idx_global = np.asarray(res["idx_global"], dtype=int)
    per_cluster = {}
    for c, centers_loc in (res.get("centers_by_cluster", {}) or {}).items():
        centers_loc = [int(u) for u in centers_loc if 0 <= int(u) < len(idx_global)]
        glob_ids = [int(idx_global[u]) for u in centers_loc]
        glob_ids = sorted(set(glob_ids))
        per_cluster[int(c)] = glob_ids
        selected_labels.update(glob_ids)
        for gid in glob_ids:
            iloc_to_class[int(gid)] = int(cls)
            iloc_to_cluster[int(gid)] = int(c)
    ENTRADAS_SELECIONADAS_POR_CLASSE_CLUSTER[int(cls)] = dict(sorted(per_cluster.items()))

ENTRADAS_SELECIONADAS_KCENTER = _to_positional_indices(selected_labels, X_test)
ENTRADAS_SELECIONADAS_POR_CLASSE_CLUSTER_POS = {
    int(cls): {int(c): _to_positional_indices(ids, X_test)
               for c, ids in clusters.items()}
    for cls, clusters in ENTRADAS_SELECIONADAS_POR_CLASSE_CLUSTER.items()
}
ENTRADAS_SELECIONADAS_KCENTER_LABELS = np.array(sorted(set(selected_labels)), dtype=int)

print(f"\n✅ Total selecionados (únicos): {len(ENTRADAS_SELECIONADAS_KCENTER)}")
print("📌 Prévia (posições p/ .iloc):", ENTRADAS_SELECIONADAS_KCENTER[:20])

entradas_por_cluster_path = os.path.join(RESULTS_DIR, "ENTRADAS_SELECIONADAS_POR_CLASSE_CLUSTER_POS.json")
with open(entradas_por_cluster_path, "w", encoding="utf-8") as f_out:
    json.dump({str(cls): {str(c): ids.tolist() for c, ids in clusters.items()}
               for cls, clusters in ENTRADAS_SELECIONADAS_POR_CLASSE_CLUSTER_POS.items()},
              f_out, ensure_ascii=False, indent=2)
print(f"✅ ENTRADAS_SELECIONADAS_POR_CLASSE_CLUSTER_POS salvos em: {entradas_por_cluster_path}")

ENTRADAS_SELECIONADAS_KCENTER_ID = (
    X_test.iloc[ENTRADAS_SELECIONADAS_KCENTER].index.values
    if len(ENTRADAS_SELECIONADAS_KCENTER) > 0
    else np.array([], dtype=int)
)
print("📌 ENTRADAS_SELECIONADAS_KCENTER_ID (primeiros 20):", ENTRADAS_SELECIONADAS_KCENTER_ID[:20])

if len(ENTRADAS_SELECIONADAS_KCENTER) > 0:
    selected_ilocs = ENTRADAS_SELECIONADAS_KCENTER
    selected_subset = X_test.iloc[selected_ilocs]
    ids_values = (
        selected_subset['id'].to_numpy()
        if 'id' in selected_subset.columns
        else selected_subset.index.to_numpy()
    )
    df_kcenter = pd.DataFrame({
        'ID': ids_values,
        'ILOC': selected_ilocs,
        'CLASSE': [iloc_to_class.get(int(iloc), np.nan) for iloc in selected_ilocs],
        'CLUSTER': [iloc_to_cluster.get(int(iloc), np.nan) for iloc in selected_ilocs]
    })
    entradas_kcenter_path = os.path.join(RESULTS_DIR, "entradas_selecionadas_kcenter.csv")
    df_kcenter.to_csv(entradas_kcenter_path, index=False)
    print(f"✅ IDs, iloc, classe e cluster salvos em: {entradas_kcenter_path}")
    print(df_kcenter.head())
else:
    df_kcenter = pd.DataFrame(columns=['ID', 'ILOC', 'CLASSE', 'CLUSTER'])
    print("Nenhuma entrada selecionada para complementar.")

CAMINHO_ARQUIVO = CSV_TEST_PATH if 'CSV_TEST_PATH' in globals() else 'N/A'
if not df_kcenter.empty:
    df_kcenter_full = df_kcenter.copy()
    df_kcenter_full['CSV_PATH'] = CAMINHO_ARQUIVO

    # marca quais selecionados são medoids (o primeiro de cada centers_by_cluster)
    is_medoid = np.zeros(len(df_kcenter_full), dtype=bool)
    iloc_array = df_kcenter_full['ILOC'].to_numpy(dtype=int)
    for cls, res in results_by_class.items():
        idx_global = np.asarray(res["idx_global"], dtype=int)
        for centers_loc in (res.get("centers_by_cluster", {}) or {}).values():
            centers_loc = [int(u) for u in centers_loc if 0 <= int(u) < len(idx_global)]
            if not centers_loc:
                continue
            medoid_global = int(idx_global[centers_loc[0]])
            medoid_mask = (iloc_array == medoid_global)
            if medoid_mask.any():
                is_medoid[medoid_mask] = True
    df_kcenter_full['IS_MEDOID'] = is_medoid

    entradas_kcenter_full_path = os.path.join(RESULTS_DIR, "entradas_selecionadas_kcenter_completo.csv")
    df_kcenter_full.to_csv(entradas_kcenter_full_path, index=False)
    print(f"✅ Arquivo completo salvo em: {entradas_kcenter_full_path}")
    print(df_kcenter_full.head())


In [ ]:
# ProtoDash + MMD por (CLASSE, CLUSTER) — pronto para 0/1/2/3 classes (versão corrigida)
import os, math, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, pairwise_distances
from sklearn.neighbors import LocalOutlierFactor

# ----------------------------
# Diretórios de saída
# ----------------------------
RESULTS_DIR = globals().get("RESULTS_DIR", str(globals().get('OUT_CSV', 'resultados_explainable_clustering')))
CSV_DIR = str(globals().get('OUT_CSV', RESULTS_DIR))
IMG_DIR = str(globals().get('OUT_IMG', os.path.join(RESULTS_DIR, 'images')))
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

CSV_OUT       = os.path.join(CSV_DIR, "protodash_MMD_prototipos_por_classe_e_cluster.csv")
CSV_OUT_FULL  = os.path.join(CSV_DIR, "protodash_MMD_dataset_selecionado_completo_classe_cluster.csv")
CSV_MMD_CURVE = os.path.join(CSV_DIR, "protodash_MMD_curve_por_cluster.csv")
PNG_MMD_CURVE = os.path.join(IMG_DIR, "protodash_MMD_curvas_por_cluster.png")

# ----------------------------
# Parâmetros (podem ser sobrescritos via PROTO_*)
# ----------------------------
KERNEL = globals().get("PROTO_KERNEL", "rbf")           # "rbf" ou "linear"
GAMMA  = globals().get("PROTO_GAMMA", "scale")          # "scale", "auto" ou float
MMD_EPS       = globals().get("PROTO_MMD_EPS", 0.05)
MMD_PATIENCE  = globals().get("PROTO_MMD_PATIENCE", 1)

MMD_M_MAX_ABS = globals().get("PROTO_MMD_M_MAX_ABS", 6)  # teto global padrão

# <<< NOVO: teto M máximo por classe (sobrescreve o global, se existir)
# Exemplo antes da célula:
PROTO_MMD_M_MAX_ABS_BY_CLASS = {0: 20, 1: 15, 2: 40}
MMD_M_MAX_ABS_BY_CLASS = globals().get("PROTO_MMD_M_MAX_ABS_BY_CLASS", {})
# >>>

M_MAX_MODE    = globals().get("PROTO_M_MAX_MODE", "sqrt")  # "sqrt" ou "proportional"
M_MAX_ALPHA   = globals().get("PROTO_M_MAX_ALPHA", 0.6)
ENSURE_ONE_RARE = globals().get("PROTO_ENSURE_ONE_RARE", True)
RARE_PCT_DIST   = globals().get("PROTO_RARE_PCT_DIST", 95.0)
RARE_Q_MU       = globals().get("PROTO_RARE_Q_MU", 0.10)
RARE_Q_LOF      = globals().get("PROTO_RARE_Q_LOF", 0.90)
LOF_K_MAX       = globals().get("PROTO_LOF_K_MAX", 20)
LOF_K_MIN       = globals().get("PROTO_LOF_K_MIN", 5)
REDUNDANCY_SIM_TAU = globals().get("PROTO_REDUNDANCY_SIM_TAU", 0.85)
ID_COLS = globals().get("PROTO_ID_COLS", [])  # ex.: ["patient_id"]

# ----------------------------
# Pré-condições
# ----------------------------
if 'results_by_class' not in globals():
    raise RuntimeError("results_by_class não encontrado. Execute a célula de diagnóstico anterior.")
if 'X_test' not in globals():
    raise RuntimeError("X_test não encontrado no ambiente.")

# Se ainda não existir, gere a matriz numérica global a partir do X_test (mesma ordem de linhas)
def get_feature_matrix(df, exclude_cols):
    num_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
    if '__cls_tmp__' in num_cols:
        num_cols.remove('__cls_tmp__')
    return df[num_cols].to_numpy(dtype=float, copy=True), num_cols

if 'EXCLUDE_COLUMNS' not in globals():
    EXCLUDE_COLUMNS = set()
if 'X_np_all' not in globals() or 'USED_FEATURE_COLS' not in globals():
    X_np_all, USED_FEATURE_COLS = get_feature_matrix(X_test, EXCLUDE_COLUMNS)

# ----------------------------
# Helpers de kernel / gamma
# ----------------------------
def _resolve_gamma_value(XA):
    XA = np.asarray(XA, dtype=float)  # garante numérico
    if KERNEL != "rbf":
        return np.nan
    if isinstance(GAMMA, (int, float)):
        return float(GAMMA)
    n_features = XA.shape[1]
    if GAMMA == "auto":
        return 1.0 / max(n_features, 1)
    var = np.nanmean(np.var(XA, axis=0))  # robusto a NaN
    denom = (n_features * var) if (var is not None and np.isfinite(var) and var > 0) else n_features
    denom = max(float(denom), 1.0)
    return 1.0 / denom

def _compute_kernel_fixed(XA, XB, gamma_value):
    XA = np.asarray(XA, dtype=float); XB = np.asarray(XB, dtype=float)
    if KERNEL == "linear":
        return linear_kernel(XA, XB)
    return rbf_kernel(XA, XB, gamma=float(gamma_value))

# ----------------------------
# MMD² empírica
# ----------------------------
def _mmd2_ref_vs_weighted(X_ref, Z, alpha, gamma_value):
    if len(Z) == 0:
        return np.inf
    X_ref = np.asarray(X_ref, dtype=float)
    Z     = np.asarray(Z, dtype=float)
    alpha = np.asarray(alpha, dtype=float)
    s = alpha.sum()
    if s > 1e-12:
        alpha = alpha / s
    K_xx = _compute_kernel_fixed(X_ref, X_ref, gamma_value).mean()
    K_xz = _compute_kernel_fixed(X_ref, Z, gamma_value).mean(axis=0)
    K_zz = _compute_kernel_fixed(Z, Z, gamma_value)
    return float(K_xx - 2.0 * np.dot(alpha, K_xz) + (alpha[:,None]*alpha[None,:]*K_zz).sum())

# ----------------------------
# Disponibilidade do AIX360
# ----------------------------
try:
    from aix360.algorithms.protodash import ProtodashSelection
    _AIX_AVAILABLE = True
except Exception:
    _AIX_AVAILABLE = False

# ----------------------------
# NNLS simples (gradiente projetado)
# ----------------------------
def _nnls_weights_for_selected(K_cand_cand, mu_cand, selected_idx, iters=300):
    if len(selected_idx) == 0:
        return np.array([], dtype=float)
    Ks = K_cand_cand[np.ix_(selected_idx, selected_idx)] + 1e-8*np.eye(len(selected_idx))
    bs = mu_cand[selected_idx]
    w = np.zeros_like(bs)
    try:
        L = np.linalg.norm(Ks, 2)
    except Exception:
        L = np.linalg.norm(Ks) or 1.0
    lr = 0.1 / max(L, 1.0)
    for _ in range(iters):
        grad = Ks @ w - bs
        w -= lr * grad
        w = np.maximum(w, 0.0)
    return w

# ----------------------------
# Fallback greedy (se AIX360 indisponível)
# ----------------------------
def _protodash_indices_fallback(X_ref, X_cand, m, gamma_value):
    X_ref = np.asarray(X_ref, dtype=float)
    X_cand = np.asarray(X_cand, dtype=float)
    n_cand = X_cand.shape[0]
    m = min(int(m), n_cand)
    if m <= 0:
        return np.array([], dtype=int), np.array([], dtype=float)
    K_ref_cand   = _compute_kernel_fixed(X_ref, X_cand, gamma_value)
    K_cand_cand  = _compute_kernel_fixed(X_cand, X_cand, gamma_value)
    mu = K_ref_cand.mean(axis=0)
    selected = []
    gains = mu.copy()
    for _ in range(m):
        gm = gains.copy()
        if selected:
            gm[selected] = -np.inf
        j = int(np.argmax(gm))
        if not np.isfinite(gm[j]): break
        selected.append(j)
        gains = mu - np.maximum.reduce([K_cand_cand[:, s] for s in selected])
    selected = np.array(selected, dtype=int)
    weights = _nnls_weights_for_selected(K_cand_cand, mu, selected)
    return selected, weights

# ----------------------------
# Wrapper ProtoDash
# ----------------------------
def _run_protodash_block(X_block, m, gamma_value):
    X_block = np.asarray(X_block, dtype=float)
    if m <= 0 or X_block.shape[0] == 0:
        return np.array([], dtype=int), np.array([], dtype=float), "fallback"
    if _AIX_AVAILABLE:
        proto = ProtodashSelection()
        S, W = proto.learn_prototypes(X_block, X_block, m)
        return np.array(S, dtype=int), np.array(W, dtype=float), "aix360"
    idx, w = _protodash_indices_fallback(X_block, X_block, m, gamma_value)
    return idx, w, "fallback"

# ----------------------------
# Teto m_max (agora depende da CLASSE)
# ----------------------------
def _cluster_m_cap(n_c, cls=None):  # <<< NOVO: recebe cls
    base = math.ceil(math.sqrt(n_c)) if M_MAX_MODE == "sqrt" else math.ceil(M_MAX_ALPHA * n_c)
    # teto absoluto por classe (se existir), senão usa o global
    if cls is not None and int(cls) in MMD_M_MAX_ABS_BY_CLASS:
        cap_abs = int(MMD_M_MAX_ABS_BY_CLASS[int(cls)])
    else:
        cap_abs = int(MMD_M_MAX_ABS)
    return max(1, min(base, n_c, cap_abs))

# ----------------------------
# Escolha de m pela curva MMD
# ----------------------------
def _choose_m_by_mmd(X_block, gamma_value, eps=MMD_EPS, patience=MMD_PATIENCE, cls=None):  # <<< NOVO: cls
    X_block = np.asarray(X_block, dtype=float)
    n = X_block.shape[0]
    m_cap = _cluster_m_cap(n, cls=cls)  # <<< NOVO: passa cls
    best = (None, None, None, np.inf, None)
    prev_mmd, stalls = None, 0
    curve = []
    for m in range(1, m_cap + 1):
        sel, w, backend = _run_protodash_block(X_block, m, gamma_value)
        Z = X_block[sel]
        mmd = _mmd2_ref_vs_weighted(X_block, Z, w, gamma_value)
        curve.append((m, mmd))
        if best[0] is None or mmd < best[3]:
            best = (m, sel.copy(), w.copy(), mmd, backend)
        if prev_mmd is not None:
            rel = (prev_mmd - mmd) / max(prev_mmd, 1e-12)
            stalls = stalls + 1 if rel < eps else 0
            if stalls >= patience:
                break
        prev_mmd = mmd
    return best[0], best[1], best[2], best[3], best[4], curve

# ----------------------------
# Estatísticas locais + raridade
# ----------------------------
def _cluster_local_stats_and_rare(X_c, res_cls, idxs_loc_c, gamma_value):
    X_c = np.asarray(X_c, dtype=float)
    K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
    mu = K_cc.mean(axis=0)
    # distância ao medoid no subcluster (fallback robusto)
    try:
        labels_loc = np.asarray(res_cls["labels"], dtype=int)
        c = int(np.unique(labels_loc[idxs_loc_c])[0])
        med_loc_in_cls = int(res_cls["medoids_local"][c])
        pos_in_cluster = np.where(idxs_loc_c == med_loc_in_cls)[0]
        if pos_in_cluster.size:
            med_pos = int(pos_in_cluster[0])
        else:
            raise RuntimeError
    except Exception:
        D_c0 = pairwise_distances(X_c)
        med_pos = int(np.argmin(D_c0.sum(axis=1)))
    D_c = pairwise_distances(X_c)
    dist_medoid_all = D_c[:, med_pos]
    dist_pct_all = np.array([(dist_medoid_all <= d).mean()*100.0 for d in dist_medoid_all])

    n = X_c.shape[0]
    if n >= LOF_K_MIN + 1:
        k = min(max(LOF_K_MIN, min(LOF_K_MAX, n - 1)), LOF_K_MAX)
        lof = LocalOutlierFactor(n_neighbors=k)
        lof.fit(X_c)
        lof_out_all = -lof.negative_outlier_factor_
    else:
        lof_out_all = np.full(n, np.nan)

    mu_q10  = float(np.nanquantile(mu, RARE_Q_MU))
    lof_q90 = float(np.nanquantile(lof_out_all, RARE_Q_LOF)) if np.isfinite(lof_out_all).any() else np.nan

    rare_mask_all = (
        (dist_pct_all >= RARE_PCT_DIST) |
        (mu <= mu_q10) |
        (np.where(np.isnan(lof_out_all), -np.inf, lof_out_all) >= (lof_q90 if np.isfinite(lof_q90) else np.inf))
    )
    return dict(mu=mu, dist_medoid_all=dist_medoid_all, dist_pct_all=dist_pct_all,
                lof_all=lof_out_all, mu_q10=mu_q10, lof_q90=lof_q90, rare_mask_all=rare_mask_all)

# ----------------------------
# Thinning por similaridade
# ----------------------------
def _thin_by_similarity(X_c, selected_idxs, gamma_value, sim_tau=REDUNDANCY_SIM_TAU, keep_rare_idx=None):
    X_c = np.asarray(X_c, dtype=float)
    if len(selected_idxs) <= 1:
        return np.asarray(selected_idxs, dtype=int)
    Z = X_c[selected_idxs]
    K_sel = _compute_kernel_fixed(Z, Z, gamma_value)
    keep = []
    order = list(range(len(selected_idxs)))
    if keep_rare_idx is not None:
        try:
            pos_rare = selected_idxs.tolist().index(int(keep_rare_idx))
            order.insert(0, order.pop(pos_rare))
        except Exception:
            pass
    for pos in order:
        if pos in keep:
            continue
        if not keep:
            keep.append(pos)
            continue
        sim_to_kept = np.max(K_sel[pos, keep])
        if sim_to_kept > sim_tau:
            continue
        keep.append(pos)
    keep = np.array(sorted(keep), dtype=int)
    return selected_idxs[keep]

# ----------------------------
# Loop principal (classe, cluster) — usa matriz numérica X_np_all
# ----------------------------
rows = []
mmd_curve_rows = []

for cls in sorted(results_by_class.keys()):
    res = results_by_class[cls]
    idx_global_cls = np.asarray(res["idx_global"], dtype=int)
    labels_loc = np.asarray(res["labels"], dtype=int)

    # >>> USAR APENAS FEATURES NUMÉRICAS <<<
    X_cls = X_np_all[idx_global_cls]

    clusters = np.sort(np.unique(labels_loc))
    for c in clusters:
        idxs_loc_c = np.where(labels_loc == c)[0]
        if idxs_loc_c.size == 0:
            continue
        X_c = X_cls[idxs_loc_c]
        n_c = X_c.shape[0]
        if n_c == 0:
            continue

        gamma_value = _resolve_gamma_value(X_c)
        # <<< NOVO: passa cls para respeitar M_MAX por classe
        m_star, sel_loc_c, w_sel, mmd_star, backend, curve = _choose_m_by_mmd(
            X_c, gamma_value, eps=MMD_EPS, patience=MMD_PATIENCE, cls=cls
        )
        # >>>
        for m_val, mmd_val in curve:
            mmd_curve_rows.append({
                "classe": int(cls), "cluster": int(c), "m": int(m_val), "mmd2": float(mmd_val),
                "kernel": KERNEL, "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
                "protodash_backend": backend
            })

        stats = _cluster_local_stats_and_rare(X_c, res, idxs_loc_c, gamma_value)
        weights_recomputed = False

        keep_rare_idx = None
        if ENSURE_ONE_RARE and sel_loc_c.size > 0:
            rare_mask_sel = stats["rare_mask_all"][sel_loc_c]
            rare_selected = np.where(rare_mask_sel)[0]
            if rare_selected.size == 0:
                rare_candidates = np.where(stats["rare_mask_all"])[0]
                if rare_candidates.size > 0:
                    def _rare_score(i):
                        return (
                            float(stats["dist_pct_all"][i]),
                            float(stats["lof_all"][i]) if np.isfinite(stats["lof_all"][i]) else -np.inf,
                            float(-stats["mu"][i])
                        )
                    add_idx = max(rare_candidates.tolist(), key=_rare_score)
                    keep_rare_idx = add_idx
                    if add_idx not in sel_loc_c.tolist():
                        sel_loc_c = np.append(sel_loc_c, add_idx)
                        K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
                        mu_all = K_cc.mean(axis=0)
                        w_sel = _nnls_weights_for_selected(K_cc, mu_all, sel_loc_c)
                        weights_recomputed = True
            else:
                rare_global_idxs = sel_loc_c[rare_selected]
                def _rare_score(i):
                    return (
                        float(stats["dist_pct_all"][i]),
                        float(stats["lof_all"][i]) if np.isfinite(stats["lof_all"][i]) else -np.inf,
                        float(-stats["mu"][i])
                    )
                keep_rare_idx = max(rare_global_idxs.tolist(), key=_rare_score)
                to_keep = [i for i in sel_loc_c.tolist() if not stats["rare_mask_all"][i] or i == keep_rare_idx]
                if len(to_keep) != len(sel_loc_c):
                    sel_loc_c = np.array(to_keep, dtype=int)
                    K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
                    mu_all = K_cc.mean(axis=0)
                    w_sel = _nnls_weights_for_selected(K_cc, mu_all, sel_loc_c)
                    weights_recomputed = True

        if w_sel.size:
            order = np.argsort(-w_sel); sel_loc_c = sel_loc_c[order]; w_sel = w_sel[order]

        sel_before = sel_loc_c.copy()
        sel_loc_c = _thin_by_similarity(X_c, sel_loc_c, gamma_value, sim_tau=REDUNDANCY_SIM_TAU, keep_rare_idx=keep_rare_idx)
        if sel_loc_c.shape[0] != sel_before.shape[0]:
            K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
            mu_all = K_cc.mean(axis=0)
            w_sel = _nnls_weights_for_selected(K_cc, mu_all, sel_loc_c)
            weights_recomputed = True
        if w_sel.size:
            order = np.argsort(-w_sel); sel_loc_c = sel_loc_c[order]; w_sel = w_sel[order]

        Z_final = X_c[sel_loc_c]
        mmd_final = _mmd2_ref_vs_weighted(X_c, Z_final, w_sel, gamma_value)
        m_final = int(len(sel_loc_c))

        # mapear para índices globais do X_test (para salvar IDs/iloc)
        sel_loc_in_class = idxs_loc_c[sel_loc_c] if sel_loc_c.size else np.array([], dtype=int)
        sel_global = res["idx_global"][sel_loc_in_class] if sel_loc_in_class.size else np.array([], dtype=int)

        for rk, (loc_c_i, g, w) in enumerate(zip(sel_loc_c, sel_global, w_sel), start=1):
            is_rare = bool(stats["rare_mask_all"][loc_c_i])
            reasons = []
            if stats["dist_pct_all"][loc_c_i] >= RARE_PCT_DIST: reasons.append("dist_pct>=95")
            if stats["mu"][loc_c_i] <= stats["mu_q10"]: reasons.append("mu<=Q10")
            lof_q90 = stats["lof_q90"]; lof_val = stats["lof_all"][loc_c_i]
            if np.isfinite(lof_q90) and np.isfinite(lof_val) and (lof_val >= lof_q90): reasons.append("LOF>=Q90")
            rows.append({
                "classe": int(cls), "cluster": int(c), "rank_no_cluster": int(rk),
                "idx_global_no_X_test": int(g), "peso_protodash": float(w),
                "mu_witness_cluster": float(stats["mu"][loc_c_i]),
                "dist_ao_medoid": float(stats["dist_medoid_all"][loc_c_i]),
                "dist_ao_medoid_pct": float(stats["dist_pct_all"][loc_c_i]),
                "lof_outlier_score": float(lof_val) if np.isfinite(lof_val) else np.nan,
                "mu_q10_cluster": float(stats["mu_q10"]),
                "lof_q90_cluster": float(lof_q90) if np.isfinite(lof_q90) else np.nan,
                "rare_candidate": bool(is_rare),
                "rare_reason": "|".join(reasons),
                "kernel": KERNEL, "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
                "m_selected_cluster": int(m_final), "mmd2_cluster": float(mmd_final),
                "protodash_backend": backend, "weights_recomputed_nnls": bool(weights_recomputed),
                "sim_thinning_tau": float(REDUNDANCY_SIM_TAU)
            })

# ----------------------------
# Consolidação
# ----------------------------
df_protos = pd.DataFrame(rows).sort_values(["classe","cluster","rank_no_cluster"]).reset_index(drop=True)

if not df_protos.empty and {'classe','cluster','idx_global_no_X_test'}.issubset(df_protos.columns):
    entradas_protodash = {}
    for (cls, clu), subdf in df_protos.groupby(['classe','cluster']):
        entradas_protodash[(int(cls), int(clu))] = subdf['idx_global_no_X_test'].astype(int).tolist()
    globals()['ENTRADAS_SELECIONADAS_PROTODASH'] = entradas_protodash
    print(f'ENTRADAS_SELECIONADAS_PROTODASH preenchida: {len(entradas_protodash)} chaves.')
else:
    print('⚠️ Não foi possível preencher ENTRADAS_SELECIONADAS_PROTODASH.')

if not df_protos.empty:
    df_protos["rank_na_classe"] = (
        df_protos.groupby("classe")["peso_protodash"].rank(ascending=False, method="first").astype(int)
    )

if ID_COLS and not df_protos.empty:
    try:
        ids_df = X_test.loc[df_protos["idx_global_no_X_test"], ID_COLS].reset_index(drop=True)
        df_protos = pd.concat([df_protos, ids_df], axis=1)
    except Exception as e:
        print(f"[Aviso] Falha ao anexar ID_COLS: {e}")

if not df_protos.empty:
    X_selected = X_test.iloc[df_protos["idx_global_no_X_test"].values].copy()
    meta_cols = ["classe","cluster","rank_no_cluster","peso_protodash","rare_candidate","dist_ao_medoid_pct",
                 "mu_witness_cluster","kernel","gamma_value","m_selected_cluster","mmd2_cluster",
                 "protodash_backend","weights_recomputed_nnls","sim_thinning_tau"]
    for i, col in enumerate(meta_cols):
        X_selected.insert(i, col, df_protos[col].values)
else:
    X_selected = pd.DataFrame()

# ----------------------------
# Salva CSVs
# ----------------------------
pd.DataFrame(mmd_curve_rows).to_csv(CSV_MMD_CURVE, index=False)
df_protos.to_csv(CSV_OUT, index=False)
X_selected.to_csv(CSV_OUT_FULL, index=False)

# ----------------------------
# Plot consolidado das curvas MMD (n subplots = nº de classes)
# ----------------------------
try:
    if mmd_curve_rows:
        df_curve = pd.DataFrame(mmd_curve_rows)
        classes = sorted(df_curve['classe'].unique())
        n_cls = len(classes)
        fig, axes = plt.subplots(1, n_cls, figsize=(5*n_cls, 4), squeeze=False)
        axes = axes[0]
        for ax, cls in zip(axes, classes):
            sub = df_curve[df_curve['classe'] == cls].copy()
            for cluster_id, g in sub.groupby('cluster'):
                g_sorted = g.sort_values('m')
                ax.plot(g_sorted['m'], g_sorted['mmd2'], marker='o', label=f"c{int(cluster_id)}")
                best_row = g_sorted.loc[g_sorted['mmd2'].idxmin()]
                ax.scatter([best_row['m']], [best_row['mmd2']], color='red', zorder=5, s=30)
            ax.set_title(f"Curva MMD - Classe {int(cls)}")
            ax.set_xlabel('m (nº protótipos)')
            ax.set_ylabel('MMD²')
            ax.grid(True, ls='--', alpha=0.4)
            ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(PNG_MMD_CURVE, dpi=globals().get('DPI', 300), bbox_inches='tight')
        plt.show()
        print(f"📈 Figura curvas MMD salva em: {PNG_MMD_CURVE}")
except Exception as e:
    print(f"[Aviso] Falha ao gerar figura de curvas MMD: {e}")

print(f"✅ ProtoDash+MMD concluído. Protos: {len(df_protos)} | Arquivos:")
print(" - Prototipos:      ", CSV_OUT)
print(" - Dataset sel.:    ", CSV_OUT_FULL)
print(" - Curvas MMD CSV:  ", CSV_MMD_CURVE)
print(" - Curvas MMD PNG:  ", PNG_MMD_CURVE if os.path.exists(PNG_MMD_CURVE) else '(não gerado)')
print("Prévia protótipos:")
print(df_protos.head().to_string(index=False))
X_selected


In [ ]:
# ProtoDash + MMD por (CLASSE, CLUSTER) — pronto para 3 classes
# - EPS=5%, PATIENCE=1 (cotovelo por MMD)
# - m_max = ceil(sqrt(n_c)) (ou proporcional via M_MAX_MODE/M_MAX_ALPHA)
# - Garante exatamente 1 "raro" por cluster
# - Thinning por similaridade (kernel) mantendo o raro
# - Salva CSVs com kernel, gamma, MMD², backend, etc.

import os, math
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, pairwise_distances
from sklearn.neighbors import LocalOutlierFactor

# ----------------------------
# Parâmetros
# ----------------------------
RESULTS_DIR = globals().get("RESULTS_DIR", "resultados_explainable_clustering")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Kernel/MMD
KERNEL = globals().get("PROTO_KERNEL", "rbf")     # "rbf" ou "linear"
GAMMA  = globals().get("PROTO_GAMMA", "scale")    # "scale", "auto" ou float
MMD_EPS       = globals().get("PROTO_MMD_EPS", 0.05)
MMD_PATIENCE  = globals().get("PROTO_MMD_PATIENCE", 1)
MMD_M_MAX_ABS = globals().get("PROTO_MMD_M_MAX_ABS", 6)

# Teto m por cluster
M_MAX_MODE    = globals().get("PROTO_M_MAX_MODE", "sqrt")   # "sqrt" ou "proportional"
M_MAX_ALPHA   = globals().get("PROTO_M_MAX_ALPHA", 0.6)

# Raridade
ENSURE_ONE_RARE = globals().get("PROTO_ENSURE_ONE_RARE", True)
RARE_PCT_DIST   = globals().get("PROTO_RARE_PCT_DIST", 95.0)
RARE_Q_MU       = globals().get("PROTO_RARE_Q_MU", 0.10)
RARE_Q_LOF      = globals().get("PROTO_RARE_Q_LOF", 0.90)
LOF_K_MAX       = globals().get("PROTO_LOF_K_MAX", 20)
LOF_K_MIN       = globals().get("PROTO_LOF_K_MIN", 5)

# Thinning
REDUNDANCY_SIM_TAU = globals().get("PROTO_REDUNDANCY_SIM_TAU", 0.85)

# IDs opcionais
ID_COLS = globals().get("PROTO_ID_COLS", [])

# Saídas
CSV_OUT          = os.path.join(RESULTS_DIR, "protodash_MMD_prototipos_por_classe_e_cluster.csv")
CSV_OUT_FULL     = os.path.join(RESULTS_DIR, "protodash_MMD_dataset_selecionado_completo_classe_cluster.csv")
CSV_MMD_CURVE    = os.path.join(RESULTS_DIR, "protodash_MMD_curve_por_cluster.csv")

# ----------------------------
# Pré-condições
# ----------------------------
if 'results_by_class' not in globals():
    raise RuntimeError("results_by_class não encontrado. Execute o diagnóstico de protótipos anterior.")
if 'X_test' not in globals():
    raise RuntimeError("X_test não encontrado no ambiente.")

# Apenas features numéricas (mesma abordagem dos blocos anteriores)
if 'EXCLUDE_COLUMNS' not in globals():
    EXCLUDE_COLUMNS = set()

def _get_numeric_matrix(df, exclude_cols):
    num_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
    if '__cls_tmp__' in num_cols:
        num_cols.remove('__cls_tmp__')
    return df[num_cols].to_numpy(dtype=float, copy=True), num_cols

# Reusa se já existir (dos blocos anteriores); senão cria agora
if 'X_np_all' not in globals() or 'USED_FEATURE_COLS' not in globals():
    X_np_all, USED_FEATURE_COLS = _get_numeric_matrix(X_test, EXCLUDE_COLUMNS)

# ----------------------------
# Helpers de kernel/gamma
# ----------------------------
def _resolve_gamma_value(XA):
    XA = np.asarray(XA, dtype=float)
    if KERNEL != "rbf":
        return np.nan
    if isinstance(GAMMA, (int, float)):
        return float(GAMMA)
    n_features = XA.shape[1]
    if n_features <= 0:
        return 1.0
    if GAMMA == "auto":
        return 1.0 / n_features
    # "scale"
    var = np.nanmean(np.var(XA, axis=0))
    denom = (n_features * var) if (np.isfinite(var) and var > 0) else n_features
    return 1.0 / max(float(denom), 1.0)

def _compute_kernel_fixed(XA, XB, gamma_value):
    XA = np.asarray(XA, dtype=float); XB = np.asarray(XB, dtype=float)
    if KERNEL == "linear":
        return linear_kernel(XA, XB)
    return rbf_kernel(XA, XB, gamma=float(gamma_value))

# ----------------------------
# MMD² (empírica)
# ----------------------------
def _mmd2_ref_vs_weighted(X_ref, Z, alpha, gamma_value):
    if len(Z) == 0:
        return np.inf
    X_ref = np.asarray(X_ref, dtype=float)
    Z     = np.asarray(Z, dtype=float)
    alpha = np.asarray(alpha, dtype=float)
    s = float(alpha.sum())
    if s > 1e-12:
        alpha = alpha / s
    K_xx = _compute_kernel_fixed(X_ref, X_ref, gamma_value).mean()
    K_xz = _compute_kernel_fixed(X_ref, Z, gamma_value).mean(axis=0)
    K_zz = _compute_kernel_fixed(Z, Z, gamma_value)
    return float(K_xx - 2.0 * np.dot(alpha, K_xz) + (alpha[:,None]*alpha[None,:]*K_zz).sum())

# ----------------------------
# ProtoDash (AIX360 se houver)
# ----------------------------
try:
    from aix360.algorithms.protodash import ProtodashSelection
    _AIX_AVAILABLE = True
except Exception:
    _AIX_AVAILABLE = False

def _nnls_weights_for_selected(K_cand_cand, mu_cand, selected_idx, iters=300):
    if len(selected_idx) == 0:
        return np.array([], dtype=float)
    Ks = K_cand_cand[np.ix_(selected_idx, selected_idx)] + 1e-8*np.eye(len(selected_idx))
    bs = mu_cand[selected_idx]
    w = np.zeros_like(bs, dtype=float)
    try:
        L = np.linalg.norm(Ks, 2)
    except Exception:
        L = np.linalg.norm(Ks) or 1.0
    lr = 0.1 / max(L, 1.0)
    for _ in range(iters):
        grad = Ks @ w - bs
        w -= lr * grad
        w = np.maximum(w, 0.0)
    return w

def _protodash_indices_fallback(X_ref, X_cand, m, gamma_value):
    X_ref = np.asarray(X_ref, dtype=float)
    X_cand = np.asarray(X_cand, dtype=float)
    n_cand = X_cand.shape[0]
    m = min(int(m), n_cand)
    if m <= 0:
        return np.array([], dtype=int), np.array([], dtype=float)
    K_ref_cand  = _compute_kernel_fixed(X_ref, X_cand, gamma_value)
    K_cand_cand = _compute_kernel_fixed(X_cand, X_cand, gamma_value)
    mu = K_ref_cand.mean(axis=0)
    selected = []
    gains = mu.copy()
    for _ in range(m):
        gm = gains.copy()
        if selected:
            gm[selected] = -np.inf
        j = int(np.argmax(gm))
        if not np.isfinite(gm[j]): break
        selected.append(j)
        gains = mu - np.maximum.reduce([K_cand_cand[:, s] for s in selected])
    selected = np.array(selected, dtype=int)
    weights = _nnls_weights_for_selected(K_cand_cand, mu, selected)
    return selected, weights

def _run_protodash_block(X_block, m, gamma_value):
    X_block = np.asarray(X_block, dtype=float)
    if m <= 0 or X_block.shape[0] == 0:
        return np.array([], dtype=int), np.array([], dtype=float), "fallback"
    if _AIX_AVAILABLE:
        proto = ProtodashSelection()
        S, W = proto.learn_prototypes(X_block, X_block, m)
        return np.array(S, dtype=int), np.array(W, dtype=float), "aix360"
    idx, w = _protodash_indices_fallback(X_block, X_block, m, gamma_value)
    return idx, w, "fallback"

# ----------------------------
# Cotovelo MMD por cluster
# ----------------------------
def _cluster_m_cap(n_c):
    base = math.ceil(math.sqrt(n_c)) if M_MAX_MODE == "sqrt" else math.ceil(M_MAX_ALPHA * n_c)
    return max(1, min(base, n_c, MMD_M_MAX_ABS))

def _choose_m_by_mmd(X_block, gamma_value, eps=MMD_EPS, patience=MMD_PATIENCE):
    X_block = np.asarray(X_block, dtype=float)
    n = X_block.shape[0]
    m_cap = _cluster_m_cap(n)
    best = (None, None, None, np.inf, None)
    prev_mmd, stalls = None, 0
    curve = []
    for m in range(1, m_cap + 1):
        sel, w, backend = _run_protodash_block(X_block, m, gamma_value)
        Z = X_block[sel]
        mmd = _mmd2_ref_vs_weighted(X_block, Z, w, gamma_value)
        curve.append((m, mmd))
        if best[0] is None or mmd < best[3]:
            best = (m, sel.copy(), w.copy(), mmd, backend)
        if prev_mmd is not None:
            rel = (prev_mmd - mmd) / max(prev_mmd, 1e-12)
            stalls = stalls + 1 if rel < eps else 0
            if stalls >= patience:
                break
        prev_mmd = mmd
    return best[0], best[1], best[2], best[3], best[4], curve

# ----------------------------
# Métricas locais + raridade
# ----------------------------
def _cluster_local_stats_and_rare(X_c, res_cls, idxs_loc_c, gamma_value):
    X_c = np.asarray(X_c, dtype=float)
    K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
    mu = K_cc.mean(axis=0)

    # distância ao medoid (percentil)
    try:
        labels_loc = np.asarray(res_cls["labels"], dtype=int)
        c = int(np.unique(labels_loc[idxs_loc_c])[0])
        med_loc_in_cls = int(res_cls["medoids_local"][c])
        pos_in_cluster = np.where(idxs_loc_c == med_loc_in_cls)[0]
        if pos_in_cluster.size:
            med_pos = int(pos_in_cluster[0])
        else:
            raise RuntimeError
    except Exception:
        D_tmp = pairwise_distances(X_c)
        med_pos = int(np.argmin(D_tmp.sum(axis=1)))

    D_c = pairwise_distances(X_c)
    dist_medoid_all = D_c[:, med_pos]
    dist_pct_all = np.array([(dist_medoid_all <= d).mean()*100.0 for d in dist_medoid_all], dtype=float)

    n = X_c.shape[0]
    if n >= LOF_K_MIN + 1:
        k = min(max(LOF_K_MIN, min(LOF_K_MAX, n - 1)), LOF_K_MAX)
        lof = LocalOutlierFactor(n_neighbors=k, novelty=False)
        lof.fit(X_c)
        lof_out_all = -lof.negative_outlier_factor_
    else:
        lof_out_all = np.full(n, np.nan, dtype=float)

    mu_q10  = float(np.nanquantile(mu, RARE_Q_MU))
    lof_q90 = float(np.nanquantile(lof_out_all, RARE_Q_LOF)) if np.isfinite(lof_out_all).any() else np.nan
    rare_mask_all = (
        (dist_pct_all >= RARE_PCT_DIST) |
        (mu <= mu_q10) |
        (np.where(np.isnan(lof_out_all), -np.inf, lof_out_all) >= (lof_q90 if np.isfinite(lof_q90) else np.inf))
    )
    return dict(mu=mu, dist_medoid_all=dist_medoid_all, dist_pct_all=dist_pct_all,
                lof_all=lof_out_all, mu_q10=mu_q10, lof_q90=lof_q90, rare_mask_all=rare_mask_all)

# ----------------------------
# Thinning por similaridade (preserva raro)
# ----------------------------
def _thin_by_similarity(X_c, selected_idxs, gamma_value, sim_tau=REDUNDANCY_SIM_TAU, keep_rare_idx=None):
    X_c = np.asarray(X_c, dtype=float)
    if len(selected_idxs) <= 1:
        return np.asarray(selected_idxs, dtype=int)
    Z = X_c[selected_idxs]
    K_sel = _compute_kernel_fixed(Z, Z, gamma_value)
    keep = []
    order = list(range(len(selected_idxs)))
    if keep_rare_idx is not None:
        try:
            pos_rare = selected_idxs.tolist().index(int(keep_rare_idx))
            order.insert(0, order.pop(pos_rare))
        except Exception:
            pass
    for pos in order:
        if pos in keep:
            continue
        if not keep:
            keep.append(pos); continue
        sim_to_kept = np.max(K_sel[pos, keep])
        if sim_to_kept > sim_tau:
            continue
        keep.append(pos)
    keep = np.array(sorted(keep), dtype=int)
    return selected_idxs[keep]

# ----------------------------
# Loop principal por (classe, cluster) — funciona para 3 classes
# ----------------------------
rows = []
mmd_curve_rows = []

for cls in sorted(results_by_class.keys()):
    res = results_by_class[cls]
    idx_global_cls = np.asarray(res["idx_global"], dtype=int)
    labels_loc = np.asarray(res["labels"], dtype=int)

    # >>> usar somente matriz numérica <<<
    X_cls = X_np_all[idx_global_cls]

    clusters = np.sort(np.unique(labels_loc))
    for c in clusters:
        idxs_loc_c = np.where(labels_loc == c)[0]
        if idxs_loc_c.size == 0:
            continue

        X_c = X_cls[idxs_loc_c]
        n_c = X_c.shape[0]
        if n_c == 0:
            continue

        gamma_value = _resolve_gamma_value(X_c)

        # 1) Cotovelo MMD
        m_star, sel_loc_c, w_sel, mmd_star, backend, curve = _choose_m_by_mmd(X_c, gamma_value, eps=MMD_EPS, patience=MMD_PATIENCE)

        for m_val, mmd_val in curve:
            mmd_curve_rows.append({
                "classe": int(cls), "cluster": int(c),
                "m": int(m_val), "mmd2": float(mmd_val),
                "kernel": KERNEL, "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
                "protodash_backend": backend
            })

        # 2) Estatísticas + raro
        stats = _cluster_local_stats_and_rare(X_c, res, idxs_loc_c, gamma_value)

        # 3) Garante exatamente 1 raro
        weights_recomputed = False
        if ENSURE_ONE_RARE and sel_loc_c.size > 0:
            rare_mask_sel = stats["rare_mask_all"][sel_loc_c]
            rare_selected = np.where(rare_mask_sel)[0]

            if rare_selected.size == 0:
                rare_candidates = np.where(stats["rare_mask_all"])[0]
                keep_rare_idx = None
                if rare_candidates.size > 0:
                    def _rare_score(i):
                        d = stats["dist_pct_all"][i]
                        l = stats["lof_all"][i] if np.isfinite(stats["lof_all"][i]) else -np.inf
                        mu_i = stats["mu"][i]
                        return (float(d), float(l), float(-mu_i))
                    add_idx = max(rare_candidates.tolist(), key=_rare_score)
                    keep_rare_idx = add_idx
                    if add_idx not in sel_loc_c.tolist():
                        sel_loc_c = np.append(sel_loc_c, add_idx)
                        K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
                        mu_all = K_cc.mean(axis=0)
                        w_sel = _nnls_weights_for_selected(K_cc, mu_all, sel_loc_c)
                        weights_recomputed = True
            else:
                rare_global_idxs = sel_loc_c[rare_selected]
                def _rare_score(i):
                    d = stats["dist_pct_all"][i]
                    l = stats["lof_all"][i] if np.isfinite(stats["lof_all"][i]) else -np.inf
                    mu_i = stats["mu"][i]
                    return (float(d), float(l), float(-mu_i))
                keep_rare_idx = max(rare_global_idxs.tolist(), key=_rare_score)
                to_keep = [i for i in sel_loc_c.tolist() if (not stats["rare_mask_all"][i]) or (i == keep_rare_idx)]
                if len(to_keep) != len(sel_loc_c):
                    sel_loc_c = np.array(to_keep, dtype=int)
                    K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
                    mu_all = K_cc.mean(axis=0)
                    w_sel = _nnls_weights_for_selected(K_cc, mu_all, sel_loc_c)
                    weights_recomputed = True
        else:
            keep_rare_idx = None

        # 4) Ordena por peso > thinning > reordena
        if w_sel.size:
            order = np.argsort(-w_sel)
            sel_loc_c = sel_loc_c[order]
            w_sel = w_sel[order]

        sel_before = sel_loc_c.copy()
        sel_loc_c = _thin_by_similarity(X_c, sel_loc_c, gamma_value, sim_tau=REDUNDANCY_SIM_TAU, keep_rare_idx=keep_rare_idx)
        if sel_loc_c.shape[0] != sel_before.shape[0]:
            K_cc = _compute_kernel_fixed(X_c, X_c, gamma_value)
            mu_all = K_cc.mean(axis=0)
            w_sel = _nnls_weights_for_selected(K_cc, mu_all, sel_loc_c)
            weights_recomputed = True

        if w_sel.size:
            order = np.argsort(-w_sel)
            sel_loc_c = sel_loc_c[order]
            w_sel = w_sel[order]

        # 5) MMD final e m_final
        Z_final = X_c[sel_loc_c]
        mmd_final = _mmd2_ref_vs_weighted(X_c, Z_final, w_sel, gamma_value)
        m_final = int(len(sel_loc_c))

        # 6) Map para índices globais do X_test (para salvar)
        sel_loc_in_class = idxs_loc_c[sel_loc_c] if sel_loc_c.size else np.array([], dtype=int)
        sel_global = res["idx_global"][sel_loc_in_class] if sel_loc_in_class.size else np.array([], dtype=int)

        # 7) Registrar
        for rk, (loc_c, g, w) in enumerate(zip(sel_loc_c, sel_global, w_sel), start=1):
            is_rare = bool(stats["rare_mask_all"][loc_c])
            reasons = []
            if stats["dist_pct_all"][loc_c] >= RARE_PCT_DIST: reasons.append("dist_pct>=95")
            if stats["mu"][loc_c] <= stats["mu_q10"]: reasons.append("mu<=Q10")
            lof_q90 = stats["lof_q90"]; lof_val = stats["lof_all"][loc_c]
            if np.isfinite(lof_q90) and np.isfinite(lof_val) and (lof_val >= lof_q90): reasons.append("LOF>=Q90")
            rows.append({
                "classe": int(cls),
                "cluster": int(c),
                "rank_no_cluster": int(rk),
                "idx_global_no_X_test": int(g),
                "peso_protodash": float(w),
                "mu_witness_cluster": float(stats["mu"][loc_c]),
                "dist_ao_medoid": float(stats["dist_medoid_all"][loc_c]),
                "dist_ao_medoid_pct": float(stats["dist_pct_all"][loc_c]),
                "lof_outlier_score": float(lof_val) if np.isfinite(lof_val) else np.nan,
                "mu_q10_cluster": float(stats["mu_q10"]),
                "lof_q90_cluster": float(lof_q90) if np.isfinite(lof_q90) else np.nan,
                "rare_candidate": bool(is_rare),
                "rare_reason": "|".join(reasons),
                "kernel": KERNEL,
                "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
                "m_selected_cluster": int(m_final),
                "mmd2_cluster": float(mmd_final),
                "protodash_backend": backend,
                "weights_recomputed_nnls": bool(weights_recomputed),
                "sim_thinning_tau": float(REDUNDANCY_SIM_TAU)
            })

# ----------------------------
# Consolidação + ranks + IDs
# ----------------------------
df_protos = pd.DataFrame(rows).sort_values(["classe","cluster","rank_no_cluster"]).reset_index(drop=True)

if not df_protos.empty:
    df_protos["rank_na_classe"] = (
        df_protos.groupby("classe")["peso_protodash"].rank(ascending=False, method="first").astype(int)
    )

if ID_COLS and not df_protos.empty:
    try:
        ids_df = X_test.loc[df_protos["idx_global_no_X_test"], ID_COLS].reset_index(drop=True)
        df_protos = pd.concat([df_protos, ids_df], axis=1)
    except Exception as e:
        print(f"[Aviso] Falha ao anexar ID_COLS: {e}")

# Dataset completo selecionado
if not df_protos.empty:
    X_selected = X_test.iloc[df_protos["idx_global_no_X_test"].values].copy()
    meta_cols = [
        "classe","cluster","rank_no_cluster","peso_protodash","rare_candidate","dist_ao_medoid_pct",
        "mu_witness_cluster","kernel","gamma_value","m_selected_cluster","mmd2_cluster",
        "protodash_backend","weights_recomputed_nnls","sim_thinning_tau"
    ]
    for i, col in enumerate(meta_cols):
        X_selected.insert(i, col, df_protos[col].values)
else:
    X_selected = pd.DataFrame()

# ----------------------------
# Salva CSVs
# ----------------------------
df_protos.to_csv(CSV_OUT, index=False)
X_selected.to_csv(CSV_OUT_FULL, index=False)
pd.DataFrame(mmd_curve_rows).to_csv(CSV_MMD_CURVE, index=False)

print(f"✅ ProtoDash + MMD concluído para {len(sorted(results_by_class.keys()))} classe(s).")
print("• Prototipos:", CSV_OUT)
print("• Dataset selecionado:", CSV_OUT_FULL)
print("• Curvas MMD:", CSV_MMD_CURVE)

X_selected


In [ ]:
# COTOVELO (por cluster) — mostra EPS_def e EPS_obs na anotação + salva CSV/PNG
# Funciona para N classes (inclui 3 classes). Mantém a lógica original.

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt

# --- Carrega curvas ---
if "CSV_MMD_CURVE" in globals() and os.path.exists(CSV_MMD_CURVE):
    df_curve = pd.read_csv(CSV_MMD_CURVE)
elif "mmd_curve_rows" in globals():
    df_curve = pd.DataFrame(mmd_curve_rows)
else:
    raise RuntimeError("Não encontrei CSV_MMD_CURVE nem mmd_curve_rows.")

need = {"classe","cluster","m","mmd2"}
if not need.issubset(df_curve.columns):
    raise ValueError(f"Esperava colunas {need}, recebi {df_curve.columns.tolist()}")

# tipar para evitar problemas de dtype vindos do CSV
df_curve["classe"] = df_curve["classe"].astype(int)
df_curve["cluster"] = df_curve["cluster"].astype(int)
df_curve["m"] = df_curve["m"].astype(int)
df_curve["mmd2"] = pd.to_numeric(df_curve["mmd2"], errors="coerce")

# --- Parâmetros ---
EPS = float(globals().get("MMD_EPS", 0.06))
PATIENCE = int(globals().get("MMD_PATIENCE", 0))
RESULTS_DIR = globals().get("RESULTS_DIR", ".")
CSV_DIR = str(globals().get('OUT_CSV', RESULTS_DIR))
IMG_DIR = str(globals().get('OUT_IMG', os.path.join(RESULTS_DIR, 'images')))
DPI = globals().get("DPI", 300)  # fallback se não existir
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

# --- Helpers ---
def stop_by_eps_with_obs(m_list, y_list, eps=0.06, patience=0):
    prev, stalls = None, 0
    for i, cur in enumerate(y_list):
        if i == 0:
            prev = cur
            continue
        rel = (prev - cur) / max(prev, 1e-12)
        if rel < eps:
            stalls += 1
        else:
            stalls = 0
        if stalls >= patience:
            return int(m_list[i]), float(rel), bool(patience > 0)
        prev = cur
    last_rel = float((y_list[-2]-y_list[-1]) / max(y_list[-2], 1e-12)) if len(y_list) >= 2 else np.nan
    return int(m_list[-1]), last_rel, False

def local_minima_idx(y, tol=1e-12):
    idx=[]
    for i in range(1,len(y)-1):
        if (y[i] < y[i-1]-tol) and (y[i] <= y[i+1]+tol):
            idx.append(i)
    return idx

def cluster_size_from_results(cls, c):
    if "results_by_class" in globals():
        res = results_by_class.get(int(cls))
        if res is not None and "labels" in res:
            labels = np.asarray(res["labels"], dtype=int)
            return int(np.sum(labels == int(c)))
    return None

# --- Estética ---
plt.rcParams.update({
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})
line_color = "#f5a623"
line_w, marker_size = 2.6, 5.8
square_size, tri_size = 58, 46

MAX_M_PLOT = int(df_curve['m'].max()) if df_curve['m'].max() <= 15 else 15
MAX_M_PLOT = max(1, MAX_M_PLOT)
XTICKS = np.arange(1, MAX_M_PLOT + 1, 1)
X_LIM   = (0.9, MAX_M_PLOT + 0.1)

# --- clusters por classe (dinâmico) ---
clusters_by_class = {
    int(cls): sorted(df_curve.loc[df_curve["classe"]==cls, "cluster"].astype(int).unique().tolist())
    for cls in sorted(df_curve['classe'].astype(int).unique())
}
classes_sorted = sorted(clusters_by_class.keys())
rows_fig = len(classes_sorted)
cols_fig = max((len(v) for v in clusters_by_class.values()), default=1)

fig, axes = plt.subplots(rows_fig, cols_fig, figsize=(6.6*cols_fig, 4.1*rows_fig), constrained_layout=False, sharey=False)
if rows_fig == 1 and cols_fig == 1:
    axes = np.array([[axes]])
elif rows_fig == 1:
    axes = np.array([axes])
elif cols_fig == 1:
    axes = np.array([[ax] for ax in axes])

summary_rows = []

for r, cls in enumerate(classes_sorted):
    cls_clusters = clusters_by_class[cls]
    for c_idx in range(cols_fig):
        ax = axes[r, c_idx]
        if c_idx >= len(cls_clusters):
            ax.axis("off"); continue
        c = cls_clusters[c_idx]
        sub = df_curve[(df_curve["classe"] == cls) & (df_curve["cluster"] == c)].sort_values("m")
        sub = sub[sub["m"] <= MAX_M_PLOT]
        if sub.empty:
            ax.axis("off"); continue
        m_vals = sub["m"].to_numpy()
        y_vals = sub["mmd2"].to_numpy()
        m_stop, eps_obs, used_pat = stop_by_eps_with_obs(m_vals, y_vals, eps=EPS, patience=PATIENCE)
        # pega y_stop de forma segura (caso m não seja contínuo)
        try:
            y_stop = float(sub.loc[sub["m"] == m_stop, "mmd2"].iloc[0])
        except Exception:
            y_stop = float(y_vals[-1])
            m_stop = int(m_vals[-1])

        N_c = cluster_size_from_results(cls, c)
        legend_suffix = ""
        pct_m = np.nan
        if N_c and N_c > 0:
            pct_m = 100.0 * m_stop / N_c
            legend_suffix = f"  [Total:{N_c}, m={int(round(pct_m))}%]"

        # curva
        ax.plot(m_vals, y_vals, color=line_color, linewidth=line_w, marker="o", markersize=marker_size,
                alpha=0.98, label=f"MMD²(P, Q_w){legend_suffix}")
        # mínimos locais
        mins = local_minima_idx(y_vals)
        if mins:
            ax.scatter(m_vals[mins], y_vals[mins], marker="v", s=tri_size, c="#6d6d6d", edgecolors="k", linewidths=0.6, alpha=0.9, zorder=3)
        # parada
        ax.scatter([m_stop], [y_stop], marker="s", s=square_size, c=line_color, edgecolors="k", linewidths=0.8, zorder=4)
        ax.axvline(m_stop, linestyle="--", linewidth=1.4, color=line_color, alpha=0.95)
        # anotação
        eps_def_pct = int(round(EPS*100))
        eps_obs_pct = (np.nan if not np.isfinite(eps_obs) else round(100.0*eps_obs, 1))
        ax.annotate(f"EPS_def:{eps_def_pct}%\nEPS_obs:{eps_obs_pct}%", xy=(m_stop, y_stop),
                    xytext=(m_stop+0.18, y_stop + 0.02*(y_vals.max()-y_vals.min()+1e-12)), fontsize=8.3, color="#444444",
                    arrowprops=dict(arrowstyle="-", color="#888888", lw=0.8))
        # eixos
        ax.set_xticks(XTICKS); ax.set_xlim(*X_LIM)
        ax.set_xlabel("nº de protótipos (m)")
        if c_idx == 0:
            ax.set_ylabel("MMD²(P, Q_w)")
        usado = "usado" if PATIENCE > 0 else "não usado"
        ax.set_title(f"classe {cls} — cluster {c} — patience={PATIENCE} ({usado})")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right", frameon=True, fontsize=8)

        summary_rows.append({
            "classe": int(cls), "cluster": int(c), "m_stop": int(m_stop), "eps_obs": float(eps_obs),
            "EPS_def": float(EPS), "patience": int(PATIENCE), "N_cluster": int(N_c) if N_c else np.nan,
            "pct_m_stop_%": float(pct_m) if np.isfinite(pct_m) else np.nan
        })

fig.subplots_adjust(wspace=0.35, hspace=0.65)
fig.suptitle("Cotovelo por ganho mínimo (MMD_EPS)", y=0.995, fontsize=16)

# --- Salva figura e CSV resumo ---
out_png = os.path.join(IMG_DIR, "cotovelo_grid_eps_obs_vs_def.png")
fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
plt.show()
print("📈 Figura salva em:", out_png, "| EPS_def:", EPS, "| patience:", PATIENCE)

cotovelo_summary_df = pd.DataFrame(summary_rows).sort_values(["classe","cluster"]).reset_index(drop=True)
out_csv = os.path.join(CSV_DIR, "cotovelo_resumo_m_stop_eps_obs.csv")
cotovelo_summary_df.to_csv(out_csv, index=False)
print("✅ Resumo cotovelo salvo em:", out_csv)
print("Prévia:")
print(cotovelo_summary_df.head().to_string(index=False))


In [ ]:
# MÉTRICAS ENXUTAS + AVALIAÇÃO COM FOLGA (SLACK)
# Funciona para 3 classes (ou N classes) e usa apenas colunas numéricas
# Gera df_protos_metrics e df_cluster_metrics (+ CSVs) nos diretórios padronizados
# ============================================================
import numpy as np
import pandas as pd
import os
from sklearn.metrics.pairwise import pairwise_distances

# ---------- Pré-checagens ----------
if 'results_by_class' not in globals():
    raise RuntimeError("results_by_class não encontrado. Execute o bloco de seleção/diagnóstico antes.")
if 'df_protos' not in globals():
    raise RuntimeError("df_protos não encontrado. Execute o bloco ProtoDash/MMD antes.")
if 'X_test' not in globals():
    raise RuntimeError("X_test não encontrado no ambiente.")

# ---------- Parâmetros ----------
RANDOM_BASELINE_REPEATS = 21
SEED = 42
rng = np.random.default_rng(SEED)

# Folga global (ex.: 0.05 = 5%, 0.10 = 10%)
SLACK = globals().get('SLACK', 0.01)

RESULTS_DIR = globals().get("RESULTS_DIR", ".")
CSV_DIR = str(globals().get('OUT_CSV', RESULTS_DIR))
os.makedirs(CSV_DIR, exist_ok=True)

CSV_PROTOS  = os.path.join(CSV_DIR, "protodash_repr_por_prototipo.csv")
CSV_CLUSTER = os.path.join(CSV_DIR, "protodash_repr_por_cluster.csv")

# ---------- Escolha de features numéricas coerentes ----------
def _infer_used_numeric_columns(X_df, exclude_cols):
    """
    Se USED_FEATURE_COLS existir, usa exatamente elas.
    Senão, usa todas as numéricas de X_df menos exclude_cols e '__cls_tmp__'.
    """
    if 'USED_FEATURE_COLS' in globals() and isinstance(USED_FEATURE_COLS, (list, tuple)) and len(USED_FEATURE_COLS) > 0:
        cols = [c for c in USED_FEATURE_COLS if c in X_df.columns]
        # mantém só numéricas por segurança
        cols = [c for c in cols if pd.api.types.is_numeric_dtype(X_df[c])]
        return cols
    excl = set(exclude_cols) if exclude_cols is not None else set()
    excl = set(list(excl) + ['__cls_tmp__'])
    return [c for c in X_df.columns if (c not in excl) and pd.api.types.is_numeric_dtype(X_df[c])]

EXCLUDE_COLUMNS = set(globals().get('EXCLUDE_COLUMNS', []))
USED_NUM_COLS = _infer_used_numeric_columns(X_test, EXCLUDE_COLUMNS)
if len(USED_NUM_COLS) == 0:
    raise RuntimeError("Nenhuma coluna numérica disponível para métricas. Verifique EXCLUDE_COLUMNS/USED_FEATURE_COLS.")

# ---------- Helpers já existentes no ambiente (do bloco ProtoDash) ----------
# _compute_kernel_fixed, _nnls_weights_for_selected, _mmd2_ref_vs_weighted, _resolve_gamma_value
missing = [name for name in ['_compute_kernel_fixed','_nnls_weights_for_selected','_mmd2_ref_vs_weighted','_resolve_gamma_value'] if name not in globals()]
if missing:
    raise RuntimeError(f"Funções auxiliares ausentes: {missing}. Execute o bloco ProtoDash/MMD antes.")

def _neff(p):
    p = np.asarray(p, dtype=float)
    s = p.sum()
    if s <= 0 or p.size == 0:
        return np.nan
    p = p / s
    return float(1.0 / np.sum(p * p))

def _mmd2_random_baseline(X_group, m, gamma_value, repeats=21, rng=None):
    if m <= 0 or X_group.shape[0] == 0:
        return np.nan
    if rng is None:
        rng = np.random.default_rng(SEED)
    n = X_group.shape[0]
    K_cc = _compute_kernel_fixed(X_group, X_group, gamma_value)
    mu_all = K_cc.mean(axis=0)
    vals = []
    for _ in range(repeats):
        sel = np.sort(rng.choice(n, size=min(m, n), replace=False))
        w   = _nnls_weights_for_selected(K_cc, mu_all, sel)
        Z   = X_group[sel]
        vals.append(_mmd2_ref_vs_weighted(X_group, Z, w, gamma_value))
    return float(np.median(vals))

# ----------------------------
# Montagem das métricas (cluster e proto)
# ----------------------------
rows_cluster = []
rows_protos  = []

for cls in sorted(results_by_class.keys()):
    res = results_by_class[cls]
    idx_global_cls = np.asarray(res["idx_global"], dtype=int)
    labels_loc     = np.asarray(res["labels"], dtype=int)

    # Subconjunto numérico da classe
    X_cls_df = X_test.iloc[idx_global_cls][USED_NUM_COLS]
    X_cls = X_cls_df.to_numpy(dtype=float, copy=True)

    clusters = np.sort(np.unique(labels_loc))
    for c in clusters:
        idxs_loc_c = np.where(labels_loc == c)[0]
        if idxs_loc_c.size == 0:
            continue

        # Dados do cluster (numéricos)
        X_c = X_cls[idxs_loc_c]
        n_c = int(X_c.shape[0])
        if n_c == 0:
            continue

        gamma_value = _resolve_gamma_value(X_c)

        # Protótipos do (cls, c) — vindos do df_protos (global)
        df_g = df_protos[
            (df_protos["classe"].astype(int) == int(cls)) &
            (df_protos["cluster"].astype(int) == int(c))
        ]

        if df_g.empty:
            rows_cluster.append({
                "classe": int(cls), "cluster": int(c), "n_grupo": n_c, "m_protos": 0,
                "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
                "(i) self-P (constante do grupo)": np.nan,
                "(ii) cross P↔Z (↑ melhor)": np.nan,
                "(iii) self-Q (↓ melhor)": np.nan,
                "MMD2 (↓)": np.nan,
                "MMD2_self_norm (↓)": np.nan,
                "MMD2_rel_random (↓)": np.nan,
                "nn_dist_mean (↓)": np.nan,
                "nn_dist_p90 (↓)": np.nan,
                "nn_dist_p95 (↓)": np.nan,
                "N_eff (↑)": np.nan
            })
            continue

        sel_globals = df_g["idx_global_no_X_test"].astype(int).values
        w_sel       = df_g["peso_protodash"].astype(float).values
        m_final     = int(len(sel_globals))

        # Subconjunto dos protótipos nas MESMAS colunas numéricas
        Z = X_test.iloc[sel_globals][USED_NUM_COLS].to_numpy(dtype=float, copy=True)

        # Kernels e métricas
        K_xx_mean = _compute_kernel_fixed(X_c, X_c, gamma_value).mean()
        K_xz = _compute_kernel_fixed(X_c, Z, gamma_value)
        mu_z = K_xz.mean(axis=0)
        w_norm = w_sel / max(w_sel.sum(), 1e-12)
        cross_PZ = float(np.dot(w_norm, mu_z))

        K_zz = _compute_kernel_fixed(Z, Z, gamma_value)
        self_Q = float(w_norm @ K_zz @ w_norm)
        mmd2 = float(K_xx_mean - 2.0 * cross_PZ + self_Q)
        mmd2_self_norm  = float(mmd2 / max(K_xx_mean, 1e-12))

        mmd2_rand_med   = _mmd2_random_baseline(X_c, m_final, gamma_value, repeats=RANDOM_BASELINE_REPEATS, rng=rng)
        mmd2_rel_random = float(mmd2 / max(mmd2_rand_med, 1e-12)) if np.isfinite(mmd2_rand_med) else np.nan

        # Distâncias
        D = pairwise_distances(X_c, Z, metric="euclidean")
        dmin = D.min(axis=1)
        nn_dist_mean = float(np.mean(dmin))
        nn_dist_p90  = float(np.quantile(dmin, 0.90))
        nn_dist_p95  = float(np.quantile(dmin, 0.95))

        Neff = _neff(w_sel)

        cluster_row = {
            "classe": int(cls), "cluster": int(c), "n_grupo": n_c, "m_protos": m_final,
            "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
            "(i) self-P (constante do grupo)": float(K_xx_mean),
            "(ii) cross P↔Z (↑ melhor)": float(cross_PZ),
            "(iii) self-Q (↓ melhor)": float(self_Q),
            "MMD2 (↓)": float(mmd2),
            "MMD2_self_norm (↓)": float(mmd2_self_norm),
            "MMD2_rel_random (↓)": float(mmd2_rel_random),
            "nn_dist_mean (↓)": nn_dist_mean,
            "nn_dist_p90 (↓)": nn_dist_p90,
            "nn_dist_p95 (↓)": nn_dist_p95,
            "N_eff (↑)": float(Neff)
        }
        rows_cluster.append(cluster_row)

        for idx_g in sel_globals:
            rows_protos.append({
                "idx_global": int(idx_g), "classe": int(cls), "cluster": int(c),
                "(i) self-P (constante do grupo)": cluster_row["(i) self-P (constante do grupo)"],
                "(ii) cross P↔Z (↑ melhor)": cluster_row["(ii) cross P↔Z (↑ melhor)"],
                "(iii) self-Q (↓ melhor)": cluster_row["(iii) self-Q (↓ melhor)"],
                "MMD2 (↓)": cluster_row["MMD2 (↓)"],
                "MMD2_self_norm (↓)": cluster_row["MMD2_self_norm (↓)"],
                "MMD2_rel_random (↓)": cluster_row["MMD2_rel_random (↓)"],
                "nn_dist_mean (↓)": cluster_row["nn_dist_mean (↓)"],
                "nn_dist_p90 (↓)": cluster_row["nn_dist_p90 (↓)"],
                "nn_dist_p95 (↓)": cluster_row["nn_dist_p95 (↓)"],
                "N_eff (↑)": cluster_row["N_eff (↑)"]
            })

# ----------------------------
# DataFrames base
# ----------------------------
df_cluster_metrics = pd.DataFrame(rows_cluster).sort_values(["classe","cluster"]).reset_index(drop=True)
df_protos_metrics  = pd.DataFrame(rows_protos).sort_values(["classe","cluster","idx_global"]).reset_index(drop=True)

# ============================================================
# AVALIAÇÃO: rótulo BOM/OK/RUIM com FOLGA e 2+ problemas => RUIM
# ============================================================
BASE_TH_MMD_REL_GOOD = 0.80
BASE_TH_MMD_REL_OK   = 0.90
BASE_TH_MMD_REL_BAD  = 1.00
BASE_TH_SELF_GOOD    = 0.60
BASE_TH_SELF_OK      = 0.90
BASE_TH_SELF_BAD     = 1.10
BASE_TH_NEFF_OK      = 0.60
BASE_TAIL_PCTL       = 0.75

TH_MMD_REL_GOOD = BASE_TH_MMD_REL_GOOD * (1 + SLACK)
TH_MMD_REL_OK   = BASE_TH_MMD_REL_OK   * (1 + SLACK)
TH_MMD_REL_BAD  = BASE_TH_MMD_REL_BAD  * (1 + SLACK)
TH_SELF_GOOD    = BASE_TH_SELF_GOOD    * (1 + SLACK)
TH_SELF_OK      = BASE_TH_SELF_OK      * (1 + SLACK)
TH_SELF_BAD     = BASE_TH_SELF_BAD     * (1 + SLACK)
TH_NEFF_OK      = BASE_TH_NEFF_OK      * (1 - SLACK)
TAIL_PCTL       = BASE_TAIL_PCTL
TAIL_MARGIN     = (1 + SLACK)

COL_MMD_REL = "MMD2_rel_random (↓)"
COL_SELF    = "MMD2_self_norm (↓)"
COL_P95     = "nn_dist_p95 (↓)"
COL_NEFF    = "N_eff (↑)"
COL_M       = "m_protos"

df_eval = df_cluster_metrics.copy()
# p75 por classe
p75_by_cls = df_eval.groupby("classe")[COL_P95].transform(lambda s: s.quantile(TAIL_PCTL))
df_eval["tail_high"] = df_eval[COL_P95] > (p75_by_cls * TAIL_MARGIN)
df_eval["neff_ratio"] = df_eval[COL_NEFF] / df_eval[COL_M].clip(lower=1)

def _avaliar(row):
    v_rel  = row[COL_MMD_REL]
    v_self = row[COL_SELF]
    tail   = bool(row["tail_high"])
    r_eff  = row["neff_ratio"]
    motivos_pos, motivos_neg = [], []
    problems = 0

    # MMD relativo ao aleatório
    if np.isfinite(v_rel):
        if v_rel < TH_MMD_REL_GOOD:
            motivos_pos.append(f"MMD² melhor que aleatório (<{TH_MMD_REL_GOOD:.2f})")
        elif v_rel < TH_MMD_REL_OK:
            motivos_pos.append(f"MMD² melhor que aleatório (<{TH_MMD_REL_OK:.2f})")
        elif v_rel >= TH_MMD_REL_BAD:
            motivos_neg.append(f"MMD² ≥ aleatório (≥{TH_MMD_REL_BAD:.2f})")
            problems += 1
        else:
            motivos_neg.append(f"MMD² ~ aleatório ({TH_MMD_REL_OK:.2f}–{TH_MMD_REL_BAD:.2f})")

    # MMD normalizado pela coesão
    if np.isfinite(v_self):
        if v_self < TH_SELF_GOOD:
            motivos_pos.append(f"MMD² baixa vs coesão (<{TH_SELF_GOOD:.2f})")
        elif v_self < TH_SELF_OK:
            motivos_pos.append(f"MMD² moderada vs coesão ({TH_SELF_GOOD:.2f}–{TH_SELF_OK:.2f})")
        elif v_self >= TH_SELF_BAD:
            motivos_neg.append(f"MMD² alta vs coesão (≥{TH_SELF_BAD:.2f})")
            problems += 1
        else:
            motivos_neg.append(f"MMD² um pouco alta vs coesão ({TH_SELF_OK:.2f}–{TH_SELF_BAD:.2f})")

    # cauda
    if tail:
        motivos_neg.append(f"cauda distante (p95 > p{int(TAIL_PCTL*100)}·{TAIL_MARGIN:.2f})")
        problems += 1
    else:
        motivos_pos.append("cauda controlada (p95 ok)")

    # N_eff
    if np.isfinite(r_eff) and r_eff >= TH_NEFF_OK:
        motivos_pos.append(f"pesos bem distribuídos (N_eff/m ≥ {TH_NEFF_OK:.2f})")
    else:
        motivos_neg.append(f"pesos concentrados (N_eff/m < {TH_NEFF_OK:.2f})")
        problems += 1

    # decisão
    if (np.isfinite(v_rel) and v_rel < TH_MMD_REL_GOOD) and (np.isfinite(v_self) and v_self < TH_SELF_OK) and (not tail) and (r_eff >= TH_NEFF_OK):
        label = "BOM"
    elif problems >= 2:
        label = "RUIM"
    else:
        label = "OK"

    texto = "; ".join(motivos_pos + (["Problemas: " + ", ".join(motivos_neg)] if motivos_neg else []))
    return pd.Series({"avaliacao": label, "analise": texto})

df_aux = df_eval.apply(_avaliar, axis=1)
df_eval["avaliacao"] = df_aux["avaliacao"]
df_eval["analise"]   = df_aux["analise"]

df_cluster_metrics = df_eval.sort_values(["classe","cluster"]).reset_index(drop=True)
df_protos_metrics = (
    df_protos_metrics
    .merge(df_cluster_metrics[["classe","cluster","avaliacao","analise"]], on=["classe","cluster"], how="left")
    .sort_values(["classe","cluster","idx_global"])
    .reset_index(drop=True)
)

# Salva
df_protos_metrics.to_csv(CSV_PROTOS, index=False, float_format="%.6f")
df_cluster_metrics.to_csv(CSV_CLUSTER, index=False, float_format="%.6f")

df_cluster_metrics_view = df_cluster_metrics.drop(columns=["gamma_value"], errors="ignore")
print(f"✅ SLACK aplicado: {SLACK:.2%}")
print(f"   TH_MMD_REL_GOOD={TH_MMD_REL_GOOD:.2f}, TH_MMD_REL_OK={TH_MMD_REL_OK:.2f}, TH_MMD_REL_BAD={TH_MMD_REL_BAD:.2f}")
print(f"   TH_SELF_GOOD={TH_SELF_GOOD:.2f}, TH_SELF_OK={TH_SELF_OK:.2f}, TH_SELF_BAD={TH_SELF_BAD:.2f}")
print(f"   TH_NEFF_OK={TH_NEFF_OK:.2f}, TAIL_MARGIN={TAIL_MARGIN:.2f}x p75")
print("\n=== df_protos_metrics (head) ===")
display(df_protos_metrics.head())
print("\n=== df_cluster_metrics (head) ===")
display(df_cluster_metrics_view.head())


In [ ]:
# ============================================================
# ADEQUAÇÃO ÀS DEFINIÇÕES DO PRINT:
# - MMD²(P,Q) via kernel
# - MMD²_self(P) (auto-discrepância de P)
# - N_eff/m
# - Tail Ratio por quantil dos pesos
# + gera CSV no formato anexo
# ============================================================

import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel

SEED = 42
rng = np.random.default_rng(SEED)

SLACK = float(globals().get("SLACK", 0.10))       # no print: ±0.1
ALPHA_TAIL = float(globals().get("ALPHA_TAIL", 0.75))  # q_alpha(w)
DATASET_NAME = str(globals().get("DATASET_NAME", "D1"))

RESULTS_DIR = globals().get("RESULTS_DIR", ".")
CSV_DIR = str(globals().get('OUT_CSV', RESULTS_DIR))
os.makedirs(CSV_DIR, exist_ok=True)

CSV_PROTOS  = os.path.join(CSV_DIR, "protodash_repr_por_prototipo.csv")
CSV_CLUSTER = os.path.join(CSV_DIR, "protodash_repr_por_cluster.csv")
CSV_ANEXO   = os.path.join(CSV_DIR, "protodash_tabela_anexo.csv")

# ---------------------- Pré-condições ----------------------
for v in ["results_by_class", "df_protos", "X_test"]:
    if v not in globals():
        raise RuntimeError(f"{v} não encontrado no ambiente.")

# ---------------------- Kernel helpers (caso não existam no ambiente) ----------------------
KERNEL = globals().get("PROTO_KERNEL", "rbf")
GAMMA  = globals().get("PROTO_GAMMA", "scale")

def _resolve_gamma_value(XA):
    # <<< correção mínima: garante float
    XA = np.asarray(XA, dtype=float)

    if KERNEL != "rbf":
        return np.nan
    if isinstance(GAMMA, (int, float)):
        return float(GAMMA)

    n_features = XA.shape[1]
    if GAMMA == "auto":
        return 1.0 / n_features

    # scale
    var = np.var(XA, axis=0).mean()
    return 1.0 / (n_features * var if var > 0 else n_features)

def _compute_kernel_fixed(XA, XB, gamma_value):
    XA = np.asarray(XA, dtype=float)
    XB = np.asarray(XB, dtype=float)
    if KERNEL == "linear":
        return linear_kernel(XA, XB)
    return rbf_kernel(XA, XB, gamma=float(gamma_value))

# ---------------------- Codificação mínima de colunas literais ----------------------
X_test_encoded = X_test.copy()
obj_cols = X_test_encoded.select_dtypes(include=["object"]).columns
if len(obj_cols) > 0:
    print(f"🔤 Codificando colunas categóricas em X_test: {list(obj_cols)}")
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_test_encoded[obj_cols] = enc.fit_transform(X_test_encoded[obj_cols])

# ---------------------- Helpers métricas ----------------------
def _neff(p):
    p = np.asarray(p, dtype=float)
    s = p.sum()
    if s <= 0 or p.size == 0:
        return np.nan
    p = p / s
    return float(1.0 / np.sum(p * p))

def _tail_ratio_quantile(w, alpha=0.75):
    """
    Tail Ratio = |{i: w_i >= q_alpha(w)}| / m
    """
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if w.size == 0 or not np.isfinite(s) or s <= 0:
        return np.nan
    w = w / s
    q = float(np.quantile(w, alpha))
    return float(np.mean(w >= q))

def _mmd2_self_P_from_kernel(X_c, gamma):
    """
    MMD²_self(P) = E[k(x,x)] - E[k(x,x')]
    (medida de dispersão interna no RKHS; ↓ mais coeso)
    """
    if X_c is None or len(X_c) == 0:
        return np.nan
    K = _compute_kernel_fixed(X_c, X_c, gamma)
    if K.size == 0:
        return np.nan
    diag_mean = float(np.mean(np.diag(K)))
    all_mean  = float(np.mean(K))
    return float(max(diag_mean - all_mean, 0.0))

def _wbar_norm(w):
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if w.size == 0 or not np.isfinite(s) or s <= 0:
        return np.nan
    w = w / s
    return float(np.mean(w))

# ---------------------------- Montagem das métricas ----------------------------
rows_cluster = []
rows_protos  = []

for cls in sorted(results_by_class.keys()):
    res = results_by_class[cls]
    idx_global_cls = np.asarray(res["idx_global"], dtype=int)
    labels_loc     = np.asarray(res["labels"], dtype=int)

    # >>> CORREÇÃO MÍNIMA: usar X_test_encoded (sem strings)
    X_cls = X_test_encoded.iloc[idx_global_cls].values
    clusters = np.sort(np.unique(labels_loc))

    for c in clusters:
        idxs_loc_c = np.where(labels_loc == c)[0]
        if idxs_loc_c.size == 0:
            continue

        X_c = X_cls[idxs_loc_c]
        n_c = int(X_c.shape[0])

        gamma_value = _resolve_gamma_value(X_c)

        df_g = df_protos[(df_protos["classe"] == int(cls)) & (df_protos["cluster"] == int(c))]
        if df_g.empty:
            rows_cluster.append({
                "classe": int(cls), "cluster": int(c), "n_grupo": n_c, "m_protos": 0,
                "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
                "MMD2 (↓)": np.nan,
                "MMD2_self (↓)": np.nan,
                "N_eff (↑)": np.nan,
                "N_eff/m (↑)": np.nan,
                "wbar_k": np.nan,
                "Tail_Ratio (↓)": np.nan
            })
            continue

        sel_globals = df_g["idx_global_no_X_test"].astype(int).values
        w_sel       = df_g["peso_protodash"].astype(float).values
        m_final     = int(len(sel_globals))

        # >>> CORREÇÃO MÍNIMA: usar X_test_encoded também para Z
        Z = X_test_encoded.iloc[sel_globals].values

        # ---- MMD²(P,Q) ----
        K_xx_mean = float(_compute_kernel_fixed(X_c, X_c, gamma_value).mean())

        K_xz = _compute_kernel_fixed(X_c, Z, gamma_value)
        mu_z = K_xz.mean(axis=0)

        w_norm = w_sel / max(w_sel.sum(), 1e-12)
        cross_PZ = float(np.dot(w_norm, mu_z))

        K_zz = _compute_kernel_fixed(Z, Z, gamma_value)
        self_Q = float(w_norm @ K_zz @ w_norm)

        mmd2 = float(K_xx_mean - 2.0 * cross_PZ + self_Q)

        # ---- MMD²_self(P) ----
        mmd2_self_P = _mmd2_self_P_from_kernel(X_c, gamma_value)

        # ---- N_eff/m ----
        Neff = _neff(w_sel)
        neff_ratio = float(Neff / max(m_final, 1)) if np.isfinite(Neff) else np.nan

        # ---- Tail Ratio ----
        tail_ratio = _tail_ratio_quantile(w_sel, alpha=ALPHA_TAIL)

        wbar_k = _wbar_norm(w_sel)

        cluster_row = {
            "classe": int(cls), "cluster": int(c),
            "n_grupo": n_c, "m_protos": m_final,
            "gamma_value": float(gamma_value) if np.isfinite(gamma_value) else np.nan,
            "MMD2 (↓)": float(mmd2),
            "MMD2_self (↓)": float(mmd2_self_P),
            "N_eff (↑)": float(Neff),
            "N_eff/m (↑)": float(neff_ratio),
            "wbar_k": float(wbar_k),
            "Tail_Ratio (↓)": float(tail_ratio),
        }
        rows_cluster.append(cluster_row)

        for idx_g in sel_globals:
            rows_protos.append({
                "idx_global": int(idx_g), "classe": int(cls), "cluster": int(c),
                "MMD2 (↓)": cluster_row["MMD2 (↓)"],
                "MMD2_self (↓)": cluster_row["MMD2_self (↓)"],
                "N_eff (↑)": cluster_row["N_eff (↑)"],
                "N_eff/m (↑)": cluster_row["N_eff/m (↑)"],
                "wbar_k": cluster_row["wbar_k"],
                "Tail_Ratio (↓)": cluster_row["Tail_Ratio (↓)"],
            })

df_cluster_metrics = pd.DataFrame(rows_cluster).sort_values(["classe","cluster"]).reset_index(drop=True)
df_protos_metrics  = pd.DataFrame(rows_protos).sort_values(["classe","cluster","idx_global"]).reset_index(drop=True)

# ---------------------------- Avaliação tipo anexo ----------------------------
TH_GOOD_MMD2      = 0.50
TH_GOOD_MMD2_SELF = 0.50
TH_GOOD_NEFFR     = 0.70
TH_GOOD_TAIL      = 0.65

TH_OK_MMD2        = 0.80
TH_OK_MMD2_SELF   = 0.60
TH_OK_NEFFR       = 0.60
TH_OK_TAIL        = 0.75

def _pass_down(v, thr):
    return (np.isfinite(v) and (v < (thr + SLACK)))

def _pass_up(v, thr):
    return (np.isfinite(v) and (v >= (thr - SLACK)))

def _avaliar_anexo(row):
    v_mmd2  = row["MMD2 (↓)"]
    v_self  = row["MMD2_self (↓)"]
    v_neffr = row["N_eff/m (↑)"]
    v_tail  = row["Tail_Ratio (↓)"]

    good_all = (
        _pass_down(v_mmd2,  TH_GOOD_MMD2) and
        _pass_down(v_self,  TH_GOOD_MMD2_SELF) and
        _pass_up  (v_neffr, TH_GOOD_NEFFR) and
        _pass_down(v_tail,  TH_GOOD_TAIL)
    )

    ok_count = int(_pass_down(v_mmd2,  TH_OK_MMD2)) \
             + int(_pass_down(v_self,  TH_OK_MMD2_SELF)) \
             + int(_pass_up  (v_neffr, TH_OK_NEFFR)) \
             + int(_pass_down(v_tail,  TH_OK_TAIL))

    if good_all:
        return "BOM"
    elif ok_count >= 2:
        return "OK"
    else:
        return "RUIM"

df_cluster_metrics["avaliacao"] = df_cluster_metrics.apply(_avaliar_anexo, axis=1)

# ---------------------------- CSVs ----------------------------
df_protos_metrics.to_csv(CSV_PROTOS, index=False, float_format="%.6f")
df_cluster_metrics.to_csv(CSV_CLUSTER, index=False, float_format="%.6f")

def _cluster_label(c):
    return f"C{int(c)}"

df_anexo = pd.DataFrame({
    "Dataset": DATASET_NAME,
    "Classe (y)": df_cluster_metrics["classe"].astype(int),
    "Cluster": df_cluster_metrics["cluster"].apply(_cluster_label),
    "n_p,k": df_cluster_metrics["m_protos"].astype(int),
    "w̄_k": df_cluster_metrics["wbar_k"].astype(float),
    "MMD²": df_cluster_metrics["MMD2 (↓)"].astype(float),
    "MMD²_self": df_cluster_metrics["MMD2_self (↓)"].astype(float),
    "N_eff/m": df_cluster_metrics["N_eff/m (↑)"].astype(float),
    "Tail Ratio": df_cluster_metrics["Tail_Ratio (↓)"].astype(float),
    "Avaliação": df_cluster_metrics["avaliacao"].astype(str),
}).sort_values(["Classe (y)", "Cluster"]).reset_index(drop=True)

df_anexo.to_csv(CSV_ANEXO, index=False, float_format="%.6f")

print("✅ Gerados:")
print(" -", CSV_PROTOS)
print(" -", CSV_CLUSTER)
print(" -", CSV_ANEXO)
print(f"✅ Tail Ratio: fração w_i >= q_{ALPHA_TAIL}(w)")

try:
    display(df_anexo)
except Exception:
    print(df_anexo.head().to_string(index=False))


In [ ]:
# ============================================================
# Gráfico estilizado: barras com IC95% por (classe, cluster)
# (VERSÃO ADEQUADA ÀS MÉTRICAS "FORMAIS" DO PRINT)
# - MMD²(P,Q): kernel (expansão)
# - MMD²_self(P): diag_mean - all_mean
# - N_eff/m: 1/sum(w^2) / m
# - Tail Ratio: |{i: w_i >= q_alpha(w)}| / m
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import os
from sklearn.preprocessing import OrdinalEncoder

# ---------------------- Pré-condições mínimas ----------------------
for v in ["results_by_class", "df_protos", "X_test"]:
    if v not in globals():
        raise RuntimeError(f"{v} não encontrado no ambiente.")

# ---------------------- Codifica X_test (corrige strings tipo 'F') ----------------------
X_test_encoded = X_test.copy()
obj_cols = X_test_encoded.select_dtypes(include=["object"]).columns
if len(obj_cols) > 0:
    print(f"🔤 Codificando colunas categóricas em X_test: {list(obj_cols)}")
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_test_encoded[obj_cols] = enc.fit_transform(X_test_encoded[obj_cols])

# ---------------------- Aparência global (sem seaborn) ----------------------
plt.rcParams.update({
    "figure.facecolor": "#f7f8fa",
    "axes.facecolor":   "#ffffff",
    "axes.edgecolor":   "#e0e3e8",
    "axes.labelcolor":  "#2a2a2a",
    "axes.grid": True,
    "grid.color": "#e9ecf2",
    "grid.linestyle": "-",
    "grid.linewidth": 0.7,
    "axes.titleweight": "bold",
    "xtick.color": "#444",
    "ytick.color": "#444",
})

# ---------------------- Paleta e configs ----------------------
COLOR_BY_LABEL = {"BOM": "#009E73", "OK": "#E69F00", "RUIM": "#D55E00"}

N_BOOT    = int(globals().get("N_BOOT", 50))
SEED_BOOT = int(globals().get("SEED_BOOT", 123))
rng_boot  = np.random.default_rng(SEED_BOOT)

MAX_COLS_PER_ROW = int(globals().get("MAX_COLS_PER_ROW", 4))
DPI = int(globals().get("DPI", 200))

SLACK = float(globals().get("SLACK", 0.10))
ALPHA_TAIL = float(globals().get("ALPHA_TAIL", 0.75))

# ---------------------- LIMIARES ----------------------
TH_GOOD = {"MMD²": 0.50, "MMD²_self": 0.50, "N_eff/m": 0.70, "Tail_ratio": 0.65}
TH_OK   = {"MMD²": 0.80, "MMD²_self": 0.60, "N_eff/m": 0.60, "Tail_ratio": 0.75}

TH_RUIM_BY_LABEL = {
    "MMD²":       TH_OK["MMD²"] + SLACK,
    "MMD²_self":  TH_OK["MMD²_self"] + SLACK,
    "N_eff/m":    TH_OK["N_eff/m"] - SLACK,
    "Tail_ratio": TH_OK["Tail_ratio"] + SLACK,
}
BEST_DIR_BY_LABEL = {"MMD²": "down", "MMD²_self": "down", "N_eff/m": "up", "Tail_ratio": "down"}

metrics_order = [
    ("mmd2_mu",     "mmd2_lo",     "mmd2_hi",     "MMD²"),
    ("mmd2self_mu", "mmd2self_lo", "mmd2self_hi", "MMD²_self"),
    ("neffr_mu",    "neffr_lo",    "neffr_hi",    "N_eff/m"),
    ("tail_mu",     "tail_lo",     "tail_hi",     "Tail_ratio"),
]

# ---------------------- Helpers (dependem do seu kernel) ----------------------
if "_compute_kernel_fixed" not in globals() or "_resolve_gamma_value" not in globals():
    raise RuntimeError("Faltam _compute_kernel_fixed e/ou _resolve_gamma_value no ambiente (use as funções do pipeline formal).")

def _neff_ratio_from_weights(w):
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if w.size == 0 or not np.isfinite(s) or s <= 0:
        return np.nan
    w = w / s
    neff = 1.0 / np.sum(w * w)
    m = w.size
    return float(neff / max(m, 1))

def _tail_ratio_quantile(w, alpha=0.75):
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if w.size == 0 or not np.isfinite(s) or s <= 0:
        return np.nan
    w = w / s
    q = float(np.quantile(w, alpha))
    return float(np.mean(w >= q))

def _mmd2_self_P_from_kernel(X_s, gamma):
    if X_s is None or len(X_s) == 0:
        return np.nan
    K = _compute_kernel_fixed(X_s, X_s, gamma)
    if K.size == 0:
        return np.nan
    diag_mean = float(np.mean(np.diag(K)))
    all_mean  = float(np.mean(K))
    return float(max(diag_mean - all_mean, 0.0))

def _mmd2_PQ_from_kernel(X_s, Z, w_norm, gamma):
    if X_s is None or len(X_s) == 0 or Z is None or len(Z) == 0:
        return np.nan
    K_xx = _compute_kernel_fixed(X_s, X_s, gamma)
    Kxx_mean = float(K_xx.mean()) if K_xx.size else np.nan
    K_xz = _compute_kernel_fixed(X_s, Z, gamma)
    mu_z = K_xz.mean(axis=0)
    K_zz = _compute_kernel_fixed(Z, Z, gamma)
    cross = float(np.dot(w_norm, mu_z))
    selfQ = float(w_norm @ K_zz @ w_norm)
    return float(Kxx_mean - 2.0 * cross + selfQ)

def _metrics_from_sample_formal(X_s, Z, w_norm_fixed, w_raw_for_tail_neff, gamma):
    mmd2 = _mmd2_PQ_from_kernel(X_s, Z, w_norm_fixed, gamma)
    mmd2_self = _mmd2_self_P_from_kernel(X_s, gamma)
    neff_ratio = _neff_ratio_from_weights(w_raw_for_tail_neff)
    tail_ratio = _tail_ratio_quantile(w_raw_for_tail_neff, alpha=ALPHA_TAIL)
    return dict(mmd2=mmd2, mmd2_self=mmd2_self, neff_ratio=neff_ratio, tail_ratio=tail_ratio)

def _bootstrap_cluster_formal(X_c, Z, w_norm_fixed, w_raw, gamma, n_boot=50, rng=None):
    if rng is None:
        rng = np.random.default_rng(123)
    if X_c is None or len(X_c) == 0 or Z is None or len(Z) == 0:
        return {k: (np.nan, np.nan, np.nan) for k in ["mmd2", "mmd2_self", "neff_ratio", "tail_ratio"]}

    vals = {"mmd2": [], "mmd2_self": [], "neff_ratio": [], "tail_ratio": []}
    n = len(X_c)

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        out = _metrics_from_sample_formal(X_c[idx], Z, w_norm_fixed, w_raw, gamma)
        for k in vals:
            vals[k].append(out[k])

    out_ci = {}
    for k, v in vals.items():
        a = np.asarray(v, float)
        mu = float(np.nanmean(a))
        finite = a[np.isfinite(a)]
        lo, hi = (np.nan, np.nan)
        if finite.size > 0:
            lo, hi = np.quantile(finite, [0.025, 0.975])
        out_ci[k] = (mu, float(lo), float(hi))
    return out_ci

# ---------------------- Computa df_boot ----------------------
rows = []

for cls, res in results_by_class.items():
    idx_global_cls = np.asarray(res["idx_global"], int)
    labels_loc     = np.asarray(res["labels"], int)

    X_cls = X_test_encoded.iloc[idx_global_cls].values

    for c in np.sort(np.unique(labels_loc)):
        idxs = np.where(labels_loc == c)[0]
        X_c  = X_cls[idxs]

        df_g = df_protos[(df_protos["classe"] == int(cls)) & (df_protos["cluster"] == int(c))]
        if df_g.empty:
            continue

        sel_g = df_g["idx_global_no_X_test"].astype(int).values
        w_raw = df_g["peso_protodash"].astype(float).values
        if len(sel_g) == 0:
            continue

        Z = X_test_encoded.iloc[sel_g].values
        w_norm = w_raw / max(w_raw.sum(), 1e-12)
        gamma = _resolve_gamma_value(X_c)

        boot = _bootstrap_cluster_formal(X_c, Z, w_norm, w_raw, gamma, n_boot=N_BOOT, rng=rng_boot)

        rows.append({
            "classe": int(cls), "cluster": int(c),

            "mmd2_mu":     boot["mmd2"][0],
            "mmd2_lo":     boot["mmd2"][1],
            "mmd2_hi":     boot["mmd2"][2],

            "mmd2self_mu": boot["mmd2_self"][0],
            "mmd2self_lo": boot["mmd2_self"][1],
            "mmd2self_hi": boot["mmd2_self"][2],

            "neffr_mu":    boot["neff_ratio"][0],
            "neffr_lo":    boot["neff_ratio"][1],
            "neffr_hi":    boot["neff_ratio"][2],

            "tail_mu":     boot["tail_ratio"][0],
            "tail_lo":     boot["tail_ratio"][1],
            "tail_hi":     boot["tail_ratio"][2],
        })

df_boot = pd.DataFrame(rows).sort_values(["classe", "cluster"]).reset_index(drop=True)

# ---------------------- Salva CSV ----------------------
IMG_DIR = str(globals().get('OUT_IMG', globals().get('IMG_DIR', 'images')))
os.makedirs(IMG_DIR, exist_ok=True)
CSV_DIR = str(globals().get('OUT_CSV', globals().get('CSV_DIR', 'csv')))
os.makedirs(CSV_DIR, exist_ok=True)

CSV_BOOT = os.path.join(CSV_DIR, "bootstrap_metrics_por_cluster_formais.csv")
df_boot.to_csv(CSV_BOOT, index=False, float_format="%.6f")
print(f"[Salvo] {CSV_BOOT}")

# ---------------------- Plot — 1 figura por classe ----------------------
def plot_class_panels_pretty_formal(clsid: int, savefig=True):
    dfc = df_boot[df_boot["classe"] == clsid].copy()
    if dfc.empty:
        print(f"(classe {clsid}) — sem dados.")
        return

    clusts = sorted(dfc["cluster"].tolist())
    nC = len(clusts)

    ncols = min(MAX_COLS_PER_ROW, nC)
    nrows = int(math.ceil(nC / ncols))

    fig_w = 4.2 * ncols + 1.0
    fig_h = 3.8 * nrows + 0.8

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)
    fig.suptitle(f"Métricas (formais) com IC95% — classe {clsid}", fontsize=16, y=1.02)

    patches = [Patch(facecolor=COLOR_BY_LABEL[k], edgecolor="#222", label=k)
               for k in ["BOM", "OK", "RUIM"]]
    fig.legend(handles=patches,
               labels=[p.get_label() for p in patches],
               loc="upper right", bbox_to_anchor=(0.99, 1.02),
               ncol=3, frameon=True)

    eval_map = {(int(r.classe), int(r.cluster)): str(r.avaliacao)
                for r in df_cluster_metrics.itertuples(index=False)} if "df_cluster_metrics" in globals() else {}

    for ax, c in zip(axes.ravel(), clusts):
        row = dfc[dfc["cluster"] == c].iloc[0]
        cat = eval_map.get((int(clsid), int(c)), "OK")
        col = COLOR_BY_LABEL.get(cat, "#999999")

        xs   = np.arange(len(metrics_order))
        barw = 0.72

        mu = np.asarray([row[m[0]] for m in metrics_order], float)
        lo = np.asarray([row[m[1]] for m in metrics_order], float)
        hi = np.asarray([row[m[2]] for m in metrics_order], float)

        # ===== CORREÇÃO MÍNIMA: garante IC consistente (evita yerr negativo) =====
        lo = np.minimum(lo, mu)          # lo <= mu
        hi = np.maximum(hi, mu)          # hi >= mu
        err_low = np.where(np.isfinite(mu - lo), mu - lo, 0.0)
        err_hi  = np.where(np.isfinite(hi - mu), hi - mu, 0.0)
        err_low = np.maximum(err_low, 0.0)
        err_hi  = np.maximum(err_hi, 0.0)
        err = np.vstack([err_low, err_hi])
        # ======================================================================

        # --- Faixas de fundo (BOM/OK/RUIM) por métrica ---
        for i, (_, _, _, lab) in enumerate(metrics_order):
            if lab == "MMD²":
                bom, ok = TH_GOOD["MMD²"], TH_OK["MMD²"]
            elif lab == "MMD²_self":
                bom, ok = TH_GOOD["MMD²_self"], TH_OK["MMD²_self"]
            elif lab == "N_eff/m":
                bom, ok = TH_GOOD["N_eff/m"], TH_OK["N_eff/m"]   # ↑ melhor
            elif lab == "Tail_ratio":
                bom, ok = TH_GOOD["Tail_ratio"], TH_OK["Tail_ratio"]
            else:
                continue

            x0, x1 = i - barw/2, i + barw/2
            xmin = (x0 + 0.5) / len(metrics_order)
            xmax = (x1 + 0.5) / len(metrics_order)

            if lab == "N_eff/m":  # ↑ melhor
                ax.axhspan(ok, bom, xmin=xmin, xmax=xmax, color="#c7e9c0", alpha=0.35, zorder=0)  # BOM
                ax.axhspan(0, ok,  xmin=xmin, xmax=xmax, color="#fee391", alpha=0.35, zorder=0)  # OK
            else:  # ↓ melhor
                ax.axhspan(0, bom, xmin=xmin, xmax=xmax, color="#c7e9c0", alpha=0.35, zorder=0)  # BOM
                ax.axhspan(bom, ok, xmin=xmin, xmax=xmax, color="#fee391", alpha=0.35, zorder=0)  # OK

        # barras + IC
        ax.bar(xs, mu, width=barw, color=col, edgecolor="#222", linewidth=0.8, zorder=2)
        ax.errorbar(xs, mu, yerr=err, fmt="none", ecolor="#222",
                    elinewidth=0.9, capsize=3, capthick=0.9, zorder=3)

        # linhas RUIM + seta (sentido do bom)
        thr_list = []
        arrow_tops = []

        for i, (_, _, _, lab) in enumerate(metrics_order):
            thr = TH_RUIM_BY_LABEL.get(lab, None)
            thr_list.append(thr)

            if thr is None or not np.isfinite(thr):
                continue

            x0 = xs[i] - barw / 2.0
            x1 = xs[i] + barw / 2.0
            ax.hlines(thr, x0, x1, linestyles="dashed", linewidth=1.2, colors="#444", zorder=4)
            ax.text(x1 + 0.03, thr, "RUIM", va="center", ha="left", fontsize=8, color="#444")

            y_ref = max([h for h in hi if np.isfinite(h)] + [thr, 1e-6])
            dy  = 0.08 * y_ref
            off = 0.015 * y_ref

            if BEST_DIR_BY_LABEL.get(lab, "down") == "down":
                y_start = max(thr - (dy + off), 0.0)
                y_end   = max(thr - off, 0.0)
            else:
                y_start = thr + (dy + off)
                y_end   = thr + off

            x_arrow = xs[i] - barw * 0.18
            ax.annotate(
                "", xy=(x_arrow, y_end), xytext=(x_arrow, y_start),
                arrowprops=dict(arrowstyle="-|>", lw=1.8, color="#2C14B3", mutation_scale=14),
                zorder=5
            )
            arrow_tops.append(max(y_start, y_end))

        y_top = max([h for h in hi if np.isfinite(h)] +
                    [t for t in thr_list if t is not None and np.isfinite(t)] +
                    arrow_tops + [0])
        ax.set_ylim(0, y_top * 1.10 if np.isfinite(y_top) and y_top > 0 else 1.0)

        # pinta RUIM acima do OK para métricas "↓ melhor"
        ymax = ax.get_ylim()[1]
        for i, (_, _, _, lab) in enumerate(metrics_order):
            if lab == "N_eff/m":
                continue
            ok = TH_OK["MMD²"] if lab == "MMD²" else TH_OK["MMD²_self"] if lab == "MMD²_self" else TH_OK["Tail_ratio"]
            x0, x1 = i - barw/2, i + barw/2
            xmin = (x0 + 0.5) / len(metrics_order)
            xmax = (x1 + 0.5) / len(metrics_order)
            ax.axhspan(ok, ymax, xmin=xmin, xmax=xmax, color="#fcbba1", alpha=0.30, zorder=0)

        # rótulos das métricas
        for i, (_, _, _, lab) in enumerate(metrics_order):
            ax.text(i, 0.02, lab, ha="center", va="bottom",
                    fontsize=9, fontweight="bold", color="white",
                    bbox=dict(boxstyle="round,pad=0.2", fc="#5a6272", ec="none", alpha=0.85),
                    transform=ax.get_xaxis_transform())

        ax.set_xticks(xs)
        ax.set_xticklabels([""] * len(xs))
        #ax.set_ylabel("valor (± IC95%)", fontsize=10)
        ax.set_ylabel("valor", fontsize=10)
        ax.set_title(f"cluster {c}  —  avaliação: {cat}", fontsize=12, pad=6)
        ax.grid(False)

    for ax in axes.ravel()[nC:]:
        ax.axis("off")

    fig.text(
        0.02, -0.02,
        (f"Regras (BOM | OK) com slack ±{SLACK:.2f}:   "
         f"MMD² < {TH_GOOD['MMD²']:.2f} | {TH_OK['MMD²']:.2f}   ·   "
         f"MMD²_self < {TH_GOOD['MMD²_self']:.2f} | {TH_OK['MMD²_self']:.2f}   ·   "
         f"N_eff/m ≥ {TH_GOOD['N_eff/m']:.2f} | {TH_OK['N_eff/m']:.2f}   ·   "
         f"Tail_ratio < {TH_GOOD['Tail_ratio']:.2f} | {TH_OK['Tail_ratio']:.2f}   "
         f"(Tail: |{{w_i ≥ q_{ALPHA_TAIL:.2f}(w)}}|/m)"),
        fontsize=10
    )

    fig.tight_layout()
    if savefig:
        fname = os.path.join(IMG_DIR, f"barras_ic95_classe_{clsid}_formais.png")
        fig.savefig(fname, dpi=DPI, bbox_inches="tight")
        print(f"[Salvo] {fname}")
    plt.show()

# ---------------------- Executa para cada classe ----------------------
for clsid in sorted(df_boot["classe"].unique()):
    plot_class_panels_pretty_formal(int(clsid), savefig=True)

# ---------------------- Mantém salvamento de ids/ilocs ----------------------
if 'ENTRADAS_SELECIONADAS_PROTODASH' in globals() and 'DF_GLOBAL' in globals() and 'ID_COL' in globals():
    ilocs = []
    for v in ENTRADAS_SELECIONADAS_PROTODASH.values():
        ilocs.extend(v)
    ilocs = sorted(set(ilocs))
    ids = DF_GLOBAL.iloc[ilocs][ID_COL].tolist() if len(ilocs) > 0 else []
    pd.DataFrame({'id': ids}).to_csv(os.path.join(CSV_DIR, 'ENTRADAS_SELECIONADAS_PROTODASH_IDDF.csv'), index=False)
    pd.DataFrame({'iloc': ilocs}).to_csv(os.path.join(CSV_DIR, 'ENTRADAS_SELECIONADAS_PROTODASH_ILOC.csv'), index=False)
    print('[Salvo] ENTRADAS_SELECIONADAS_PROTODASH_IDDF.csv e ENTRADAS_SELECIONADAS_PROTODASH_ILOC.csv')
else:
    print('[Aviso] Não foi possível salvar arquivos ENTRADAS_SELECIONADAS_PROTODASH_IDDF/ILOC (variáveis não encontradas)')


In [ ]:
# Gráfico estilizado: barras com IC95% por (classe, cluster) — compatível com 3+ classes
# - Usa somente colunas numéricas (USED_FEATURE_COLS se existir; senão autodetecta)
# - Reutiliza helpers do bloco ProtoDash/MMD (_compute_kernel_fixed, _nnls_weights_for_selected, _resolve_gamma_value)
# ================================================================================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import pairwise_distances
import os

# ---------------------- Checagens e setup numérico ----------------------
if 'results_by_class' not in globals():
    raise RuntimeError("results_by_class não encontrado. Execute os blocos anteriores (PAM/diagnóstico).")
if 'df_protos' not in globals():
    raise RuntimeError("df_protos não encontrado. Rode o bloco ProtoDash/MMD antes deste.")
if 'X_test' not in globals():
    raise RuntimeError("X_test não encontrado.")

# helpers exigidos (do bloco ProtoDash/MMD)
_need_helpers = ['_compute_kernel_fixed', '_nnls_weights_for_selected', '_resolve_gamma_value']
_missing = [h for h in _need_helpers if h not in globals()]
if _missing:
    raise RuntimeError(f"Funções auxiliares ausentes: {_missing}. Execute o bloco ProtoDash/MMD antes.")

# escolher colunas numéricas coerentes com o restante do notebook
def _infer_used_numeric_columns(X_df, exclude_cols):
    if 'USED_FEATURE_COLS' in globals() and isinstance(USED_FEATURE_COLS, (list, tuple)) and len(USED_FEATURE_COLS) > 0:
        cols = [c for c in USED_FEATURE_COLS if c in X_df.columns]
        cols = [c for c in cols if pd.api.types.is_numeric_dtype(X_df[c])]
        return cols
    excl = set(exclude_cols) if exclude_cols is not None else set()
    excl = set(list(excl) + ['__cls_tmp__'])
    return [c for c in X_df.columns if (c not in excl) and pd.api.types.is_numeric_dtype(X_df[c])]

EXCLUDE_COLUMNS = set(globals().get('EXCLUDE_COLUMNS', []))
USED_NUM_COLS = _infer_used_numeric_columns(X_test, EXCLUDE_COLUMNS)
if len(USED_NUM_COLS) == 0:
    raise RuntimeError("Nenhuma coluna numérica disponível. Verifique EXCLUDE_COLUMNS/USED_FEATURE_COLS.")

DPI = int(globals().get('DPI', 150))
IMG_DIR = str(globals().get('OUT_IMG', globals().get('IMG_DIR', 'images')))
CSV_DIR = str(globals().get('OUT_CSV', globals().get('CSV_DIR', 'csv')))
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

# ---------------------- Aparência global (sem seaborn) ----------------------
plt.rcParams.update({
    "figure.facecolor": "#f7f8fa",
    "axes.facecolor":   "#ffffff",
    "axes.edgecolor":   "#e0e3e8",
    "axes.labelcolor":  "#2a2a2a",
    "axes.grid": True,
    "grid.color": "#e9ecf2",
    "grid.linestyle": "-",
    "grid.linewidth": 0.7,
    "axes.titleweight": "bold",
    "xtick.color": "#444",
    "ytick.color": "#444",
})

# ---------------------- Paleta e regras ----------------------
COLOR_BY_LABEL = {"BOM":"#009E73","OK":"#E69F00","RUIM":"#D55E00"}

N_BOOT   = 300
SEED_BOOT = 123
rng_boot  = np.random.default_rng(SEED_BOOT)
MAX_COLS_PER_ROW = 4  # até 4 gráficos por linha

TH = {
    "mmd_rel":    (0.80, 0.50),   # MMD2_rel_random < 0.80 | 0.50
    "mmd_self":   (0.50, 0.60),   # MMD2_self_norm  < 0.50 | 0.60
    "neff_ratio": (0.60, 0.70),   # N_eff/m ≥ 0.60 | 0.70
    "tail_ratio": (0.75, 0.65),   # Tail_ratio < 0.75 | 0.65
}

# threshold(OK) por rótulo
TH_OK_BY_LABEL = {
    "MMD²":       min(TH["mmd_rel"]),
    "MMD²_self":  min(TH["mmd_self"]),
    "N_eff/m":    min(TH["neff_ratio"]),
    "Tail_ratio": min(TH["tail_ratio"]),
}
# threshold(RUIM) por rótulo
TH_RUIM_BY_LABEL = {
    "MMD²":       max(TH["mmd_rel"]),     # ↓ melhor → RUIM é o maior limiar
    "MMD²_self":  max(TH["mmd_self"]),    # ↓ melhor
    "N_eff/m":    min(TH["neff_ratio"]),  # ↑ melhor → RUIM é o menor limiar
    "Tail_ratio": max(TH["tail_ratio"]),  # ↓ melhor → RUIM é o maior limiar
}
# direção do “melhor é”
BEST_DIR_BY_LABEL = {"MMD²":"down","MMD²_self":"down","N_eff/m":"up","Tail_ratio":"down"}

# Ordem das métricas (col_media, col_lo, col_hi, rótulo curto)
metrics_order = [
    ("mmd2_mu",   "mmd2_lo",   "mmd2_hi",   "MMD²"),
    ("mmd2sn_mu", "mmd2sn_lo", "mmd2sn_hi", "MMD²_self"),
    ("neffr_mu",  "neffr_lo",  "neffr_hi",  "N_eff/m"),
    ("tail_mu",   "tail_lo",   "tail_hi",   "Tail_ratio"),
]

# ---------------------- helpers métricas (bootstrap) ----------------------
def _weights_nnls_from_boot(K_zz, mu_z):
    m = len(mu_z)
    if m == 0: return np.array([])
    w = _nnls_weights_for_selected(K_zz, mu_z, np.arange(m))
    s = w.sum()
    return (w/s) if s > 0 else np.full_like(w, 1.0/m)

def _metrics_from_sample(X_s, Z, gamma):
    K_xx = _compute_kernel_fixed(X_s, X_s, gamma)
    Kxx_mean = float(K_xx.mean()) if K_xx.size else np.nan

    if len(Z):
        K_xz = _compute_kernel_fixed(X_s, Z, gamma)
        K_zz = _compute_kernel_fixed(Z, Z, gamma)
        mu_z = K_xz.mean(axis=0)
        w    = _weights_nnls_from_boot(K_zz, mu_z)
        cross = float(np.dot(w, mu_z))
        selfQ = float(w @ K_zz @ w)
        mmd2  = float(Kxx_mean - 2*cross + selfQ)
        mmd2_self = float(mmd2 / max(Kxx_mean, 1e-12))
        D = pairwise_distances(X_s, Z, metric="euclidean")
        p95 = float(np.quantile(D.min(axis=1), 0.95))
        neff_ratio = float(1.0 / np.sum((w / max(w.sum(),1e-12))**2)) / max(len(Z),1)
    else:
        mmd2 = mmd2_self = p95 = neff_ratio = np.nan

    return dict(mmd2=mmd2, mmd2_self=mmd2_self, p95=p95, neff_ratio=neff_ratio)

def _bootstrap_cluster(X_c, Z, gamma, n_boot=N_BOOT, rng=rng_boot):
    if len(X_c)==0 or len(Z)==0:
        return {k:(np.nan,np.nan,np.nan) for k in ["mmd2","mmd2_self","p95","neff_ratio"]}
    vals = {"mmd2":[], "mmd2_self":[], "p95":[], "neff_ratio":[]}
    n = len(X_c)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        out = _metrics_from_sample(X_c[idx], Z, gamma)
        for k in vals: vals[k].append(out[k])
    out = {}
    for k, v in vals.items():
        a  = np.asarray(v, float)
        mu = float(np.nanmean(a))
        lo, hi = np.quantile(a, [0.025, 0.975])
        out[k] = (mu, float(lo), float(hi))
    return out

# ---------------------- Tail baseline p/ classe (p75 dos p95) ----------------------
p75_p95_by_cls = {}
for cls, res in results_by_class.items():
    idx_global_cls = np.asarray(res["idx_global"], int)
    labels_loc     = np.asarray(res["labels"], int)
    # usa somente colunas numéricas coerentes
    X_cls = X_test.iloc[idx_global_cls][USED_NUM_COLS].to_numpy(dtype=float, copy=True)
    p95s = []
    for c in np.sort(np.unique(labels_loc)):
        idxs = np.where(labels_loc == c)[0]
        X_c  = X_cls[idxs]
        df_g = df_protos[(df_protos["classe"]==int(cls)) & (df_protos["cluster"]==int(c))]
        if df_g.empty: continue
        sel_g = df_g["idx_global_no_X_test"].astype(int).values

        # mapeia para posições locais do cluster (somente se pertencerem ao cluster)
        gids  = idx_global_cls[idxs]
        loc   = [int(np.where(gids==g)[0][0]) for g in sel_g if g in set(gids)]
        if not loc: continue
        Z = X_c[loc]
        D = pairwise_distances(X_c, Z, metric="euclidean")
        p95s.append(float(np.quantile(D.min(axis=1), 0.95)))
    p75_p95_by_cls[int(cls)] = (np.quantile(p95s, 0.75) if p95s else np.nan)

# ---------------------- Computa médias/IC por cluster ----------------------
rows = []
for cls, res in results_by_class.items():
    idx_global_cls = np.asarray(res["idx_global"], int)
    labels_loc     = np.asarray(res["labels"], int)
    X_cls = X_test.iloc[idx_global_cls][USED_NUM_COLS].to_numpy(dtype=float, copy=True)

    for c in np.sort(np.unique(labels_loc)):
        idxs = np.where(labels_loc == c)[0]
        X_c  = X_cls[idxs]

        df_g = df_protos[(df_protos["classe"]==int(cls)) & (df_protos["cluster"]==int(c))]
        if df_g.empty: continue
        sel_g = df_g["idx_global_no_X_test"].astype(int).values

        gids  = idx_global_cls[idxs]
        loc   = [int(np.where(gids==g)[0][0]) for g in sel_g if g in set(gids)]
        if not loc: continue

        Z = X_c[loc]
        gamma = _resolve_gamma_value(X_c)

        boot = _bootstrap_cluster(X_c, Z, gamma)
        base = p75_p95_by_cls.get(int(cls), np.nan)

        def _ratio(ci):
            mu, lo, hi = ci
            if not np.isfinite(base) or base<=0: return (np.nan,np.nan,np.nan)
            return (mu/base, lo/base, hi/base)

        tail_mu, tail_lo, tail_hi = _ratio(boot["p95"])
        rows.append({
            "classe": int(cls), "cluster": int(c),
            "mmd2_mu": boot["mmd2"][0],    "mmd2_lo": boot["mmd2"][1],    "mmd2_hi": boot["mmd2"][2],
            "mmd2sn_mu": boot["mmd2_self"][0],"mmd2sn_lo": boot["mmd2_self"][1],"mmd2sn_hi": boot["mmd2_self"][2],
            "neffr_mu": boot["neff_ratio"][0],"neffr_lo": boot["neff_ratio"][1],"neffr_hi": boot["neff_ratio"][2],
            "tail_mu": tail_mu, "tail_lo": tail_lo, "tail_hi": tail_hi,
        })

df_boot = pd.DataFrame(rows).sort_values(["classe","cluster"]).reset_index(drop=True)
CSV_BOOT = os.path.join(CSV_DIR, "bootstrap_metrics_por_cluster.csv")
df_boot.to_csv(CSV_BOOT, index=False, float_format="%.6f")

# ---------------------- Função de plot — 1 figura por classe (N subplots) ----------------------
def plot_class_panels_pretty(clsid: int, savefig=True):
    dfc = df_boot[df_boot["classe"] == clsid].copy()
    if dfc.empty:
        print(f"(classe {clsid}) — sem dados.")
        return

    clusts = sorted(dfc["cluster"].tolist())
    nC = len(clusts)
    ncols = min(MAX_COLS_PER_ROW, nC)
    nrows = int(math.ceil(nC / ncols))

    fig_w = 4.2 * ncols + 1.0
    fig_h = 3.8 * nrows + 0.8
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)
    fig.suptitle(f"Métricas com IC95% — classe {clsid}", fontsize=16, y=1.02)

    patches = [Patch(facecolor=COLOR_BY_LABEL[k], edgecolor="#222", label=k)
               for k in ["BOM", "OK", "RUIM"]]
    fig.legend(handles=patches,
               labels=[p.get_label() for p in patches],
               loc="upper right", bbox_to_anchor=(0.99, 1.02),
               ncol=3, frameon=True)

    # mapeia (classe, cluster) -> avaliação final
    eval_map = {(int(r.classe), int(r.cluster)): r.avaliacao
                for r in df_cluster_metrics.itertuples(index=False)}

    for ax, c in zip(axes.ravel(), clusts):
        row = dfc[dfc["cluster"] == c].iloc[0]
        cat = eval_map.get((int(clsid), int(c)), "OK")
        col = COLOR_BY_LABEL.get(cat, "#999999")

        xs   = np.arange(len(metrics_order))
        barw = 0.72

        mu = [row[m[0]] for m in metrics_order]
        lo = [row[m[1]] for m in metrics_order]
        hi = [row[m[2]] for m in metrics_order]
        err = np.vstack([np.array(mu) - np.array(lo), np.array(hi) - np.array(mu)])

        # barras + IC
        ax.bar(xs, mu, width=barw, color=col, edgecolor="#222", linewidth=0.8, zorder=2)
        ax.errorbar(xs, mu, yerr=err, fmt="none", ecolor="#222",
                    elinewidth=0.9, capsize=3, capthick=0.9, zorder=3)

        # linhas pontilhadas + seta azul (direção do "melhor é")
        thr_list = []
        arrow_tops = []
        for i, (_, _, _, lab) in enumerate(metrics_order):
            thr = TH_RUIM_BY_LABEL.get(lab, None)
            thr_list.append(thr)
            if thr is None or not np.isfinite(thr):
                continue
            x0 = xs[i] - barw/2.0
            x1 = xs[i] + barw/2.0
            ax.hlines(thr, x0, x1, linestyles="dashed", linewidth=1.2, colors="#444", zorder=4)
            ax.text(x1 + 0.03, thr, "RUIM", va="center", ha="left", fontsize=8, color="#444")

            y_ref = max([h for h in hi if np.isfinite(h)] + [thr, 1e-6])
            dy = 0.08 * y_ref
            off = 0.015 * y_ref
            if BEST_DIR_BY_LABEL.get(lab, "down") == "down":
                y_start = max(thr - (dy + off), 0.0)
                y_end   = max(thr - off, 0.0)
            else:
                y_start = thr + (dy + off)
                y_end   = thr + off
            x_arrow = xs[i] - barw * 0.18
            ax.annotate("", xy=(x_arrow, y_end), xytext=(x_arrow, y_start),
                        arrowprops=dict(arrowstyle="-|>", lw=1.8, color="#2C14B3", mutation_scale=14),
                        zorder=5)
            arrow_tops.append(max(y_start, y_end))

        y_top = max([h for h in hi if np.isfinite(h)] +
                    [t for t in thr_list if t is not None and np.isfinite(t)] +
                    arrow_tops + [0])
        ax.set_ylim(0, y_top * 1.10)

        for i, (_, _, _, lab) in enumerate(metrics_order):
            ax.text(i, 0.02, lab, ha="center", va="bottom",
                    fontsize=9, fontweight="bold", color="white",
                    bbox=dict(boxstyle="round,pad=0.2", fc="#5a6272", ec="none", alpha=0.85),
                    transform=ax.get_xaxis_transform())

        ax.set_xticks(xs)
        ax.set_xticklabels([""] * len(xs))
        ax.set_ylabel("valor (± IC95%)", fontsize=10)
        ax.set_title(f"cluster {c}  —  avaliação: {cat}", fontsize=12, pad=6)
        ax.grid(False)

    total_axes = nrows * ncols
    for ax in axes.ravel()[nC:total_axes]:
        ax.axis("off")

    fig.text(0.02, -0.02,
             ("Regras (BOM | OK):   "
              f"MMD2_rel_random < {TH['mmd_rel'][1]:.2f} | {TH['mmd_rel'][0]:.2f}   ·   "
              f"MMD2_self_norm < {TH['mmd_self'][0]:.2f} | {TH['mmd_self'][1]:.2f}   ·   "
              f"N_eff/m ≥ {TH['neff_ratio'][1]:.2f} | {TH['neff_ratio'][0]:.2f}   ·   "
              f"Tail_ratio < {TH['tail_ratio'][1]:.2f} | {TH['tail_ratio'][0]:.2f}"),
             fontsize=10)

    fig.tight_layout()
    if savefig:
        fname = os.path.join(IMG_DIR, f"barras_ic95_classe_{clsid}.png")
        fig.savefig(fname, dpi=DPI, bbox_inches="tight")
        print(f"[Salvo] {fname}")
    plt.show()

# ---------------------- Executa para cada classe ----------------------
for clsid in sorted(df_boot["classe"].unique()):
    plot_class_panels_pretty(int(clsid), savefig=True)

# Exporta IDs/ilocs selecionados (se variáveis existirem)
if 'ENTRADAS_SELECIONADAS_PROTODASH' in globals() and 'DF_GLOBAL' in globals() and 'ID_COL' in globals():
    ilocs = []
    for v in ENTRADAS_SELECIONADAS_PROTODASH.values():
        ilocs.extend(v)
    ilocs = sorted(set(ilocs))
    ids = DF_GLOBAL.iloc[ilocs][ID_COL].tolist() if len(ilocs) > 0 else []
    pd.DataFrame({'id': ids}).to_csv(os.path.join(CSV_DIR, 'ENTRADAS_SELECIONADAS_PROTODASH_IDDF.csv'), index=False)
    pd.DataFrame({'iloc': ilocs}).to_csv(os.path.join(CSV_DIR, 'ENTRADAS_SELECIONADAS_PROTODASH_ILOC.csv'), index=False)
    print('[Salvo] ENTRADAS_SELECIONADAS_PROTODASH_IDDF.csv e ENTRADAS_SELECIONADAS_PROTODASH_ILOC.csv')
else:
    print('[Aviso] Não foi possível salvar arquivos ENTRADAS_SELECIONADAS_PROTODASH_IDDF/ILOC (variáveis não encontradas)')


In [ ]:
# Seleção Aleatória por (classe, cluster), sem sobrepor KCenter/ProtoDash — compatível com 3+ classes
# Cria:
#   ENTRADAS_SELECIONADAS_ALEATORIO  (dict {(classe,cluster): [iloc_global,...]})
#   ENTRADAS_SELECIONADAS_ALEATORIAS (alias)
#   ENTRADAS_SELECIONADAS_ALEATORIAS_LIST (lista achatada de ilocs)
# E CSVs auxiliares com IDs/ILOCs.
# =================================================================================

import math
import numpy as np
import pandas as pd
import os

# ---------- PARÂMETROS ----------
SEED_RANDOM = 123
rng = np.random.default_rng(SEED_RANDOM)

# Quantidade alvo de aleatórios por cluster (escolha UM modo):
RANDOM_PER_CLUSTER = 2            # inteiro fixo (mínimo desejado por cluster)
RANDOM_FRAC_PER_CLUSTER = None    # fração do tamanho do cluster (ex.: 0.10 = 10%)
MIN_RANDOM_PER_CLUSTER = 2        # nunca escolher menos que isso (se houver candidatos)
MAX_RANDOM_PER_CLUSTER = None     # opcional: teto por cluster (None = sem teto)

# Diretórios de saída
RESULTS_DIR = str(globals().get('RESULTS_DIR', '.'))
CSV_DIR = str(globals().get('OUT_CSV', globals().get('CSV_DIR', RESULTS_DIR)))
os.makedirs(CSV_DIR, exist_ok=True)

CSV_RANDOM_OUT   = os.path.join(CSV_DIR, "selecionados_aleatorio_por_classe_cluster.csv")
CSV_PROTODASH_ID = os.path.join(CSV_DIR, "selecionados_protodash_id_por_classe_cluster.csv")
CSV_ALEATORIAS_ID= os.path.join(CSV_DIR, "selecionados_aleatorios_id_por_classe_cluster.csv")

# ---------- PRÉ-COND. ----------
if 'results_by_class' not in globals():
    raise RuntimeError("results_by_class não encontrado. Rode o bloco de diagnóstico/PAM antes.")
if 'X_test' not in globals():
    raise RuntimeError("X_test não encontrado.")

# ---------- HELPERS ----------
def _normalize_selection_to_map(selection):
    """
    Converte seleção em dict {(classe, cluster): set(iloc_global)}.
    Suporta:
      1) dict plano {(classe,cluster): iterable}
      2) dict aninhado {classe: {cluster: iterable}}
      3) DataFrame com ['classe','cluster', id_col] (id_col ∈ {'idx_global','idx_global_no_X_test','index_global','id_global'})
    """
    m = {}
    if selection is None:
        return m

    # 1) dict plano {(classe,cluster): [...]} 
    if isinstance(selection, dict):
        # detecta se é aninhado {classe: {cluster: [...]}}
        is_nested = all(isinstance(v, dict) for v in selection.values()) and not any(isinstance(k, tuple) for k in selection.keys())
        if is_nested:
            for cls, dclus in selection.items():
                try:
                    icls = int(cls)
                except Exception:
                    continue
                for c, v in dclus.items():
                    try:
                        ic = int(c)
                    except Exception:
                        continue
                    m[(icls, ic)] = set(int(x) for x in list(v))
            return m
        else:
            for k, v in selection.items():
                if isinstance(k, tuple) and len(k) == 2:
                    try:
                        cls, c = int(k[0]), int(k[1])
                    except Exception:
                        continue
                    m[(cls, c)] = set(int(x) for x in list(v))
            return m

    # 3) DataFrame
    if isinstance(selection, pd.DataFrame):
        id_cols_try = ["idx_global", "idx_global_no_X_test", "index_global", "id_global"]
        id_col = next((col for col in id_cols_try if col in selection.columns), None)
        if id_col is None:
            return m
        if not {"classe","cluster"}.issubset(selection.columns):
            return m
        for (cls, c), grp in selection.groupby(["classe", "cluster"]):
            try:
                icls, ic = int(cls), int(c)
            except Exception:
                continue
            m[(icls, ic)] = set(int(x) for x in grp[id_col].tolist())
        return m

    return m

def _target_random_count(n_c):
    """Decide k alvo por cluster, combinando inteiro fixo / fração / mínimos / máximos."""
    if RANDOM_FRAC_PER_CLUSTER is not None:
        base = math.ceil(float(RANDOM_FRAC_PER_CLUSTER) * n_c)
    else:
        base = int(RANDOM_PER_CLUSTER)
    base = max(base, int(MIN_RANDOM_PER_CLUSTER))
    if MAX_RANDOM_PER_CLUSTER is not None:
        base = min(base, int(MAX_RANDOM_PER_CLUSTER))
    return max(base, 0)

def _safe_row_id(df, iloc_row, id_col_name="id"):
    """Retorna ID se coluna existir; caso contrário volta ao próprio iloc."""
    try:
        if id_col_name in df.columns:
            return int(df.iloc[iloc_row][id_col_name])
    except Exception:
        pass
    return int(iloc_row)

# ---------- ENTRADAS de referência ----------
KSEL = _normalize_selection_to_map(globals().get("ENTRADAS_SELECIONADAS_KCENTER", None))
PSEL = _normalize_selection_to_map(globals().get("ENTRADAS_SELECIONADAS_PROTODASH", None))

# Fallback: montar PSEL a partir de df_protos, se existir e PSEL estiver vazio
if not PSEL and "df_protos" in globals():
    try:
        df_pd_norm = df_protos.rename(columns={"idx_global_no_X_test":"idx_global"})
        if {"classe","cluster","idx_global"}.issubset(df_pd_norm.columns):
            PSEL = _normalize_selection_to_map(df_pd_norm[["classe","cluster","idx_global"]])
    except Exception:
        pass

# ---------- LOOP por (classe, cluster) ----------
ENTRADAS_SELECIONADAS_ALEATORIO = {}   # {(classe, cluster): [iloc_global,...]}
rows_random = []

for cls in sorted(results_by_class.keys()):
    res = results_by_class[cls]
    idx_global_cls = np.asarray(res["idx_global"], dtype=int)
    labels_loc     = np.asarray(res["labels"], dtype=int)
    clusters = np.sort(np.unique(labels_loc))

    for c in clusters:
        idxs_loc_c = np.where(labels_loc == c)[0]
        if idxs_loc_c.size == 0:
            continue

        # universo de candidatos desse cluster (em índices globais no X_test)
        global_ids_c = idx_global_cls[idxs_loc_c]
        cand_set = set(int(g) for g in global_ids_c.tolist())

        # excluir já selecionados por KCenter e ProtoDash
        already = set()
        already |= KSEL.get((int(cls), int(c)), set())
        already |= PSEL.get((int(cls), int(c)), set())

        candidates = list(cand_set - already)
        n_c = len(global_ids_c)
        k_target = _target_random_count(n_c)

        if len(candidates) == 0:
            picked = []
        else:
            k = min(k_target, len(candidates))
            picked = rng.choice(candidates, size=k, replace=False).tolist()

        ENTRADAS_SELECIONADAS_ALEATORIO[(int(cls), int(c))] = picked
        rows_random.extend({"classe": int(cls), "cluster": int(c), "idx_global": int(gid)} for gid in picked)

# Alias solicitado (mesmo objeto)
ENTRADAS_SELECIONADAS_ALEATORIAS = ENTRADAS_SELECIONADAS_ALEATORIO

# ---------- DataFrame e CSV ----------
df_selecao_aleatorio = (
    pd.DataFrame(rows_random)
    .sort_values(["classe","cluster","idx_global"])
    .reset_index(drop=True)
)
df_selecao_aleatorio.to_csv(CSV_RANDOM_OUT, index=False)

# ---------- Exporta IDs por cluster para ProtoDash e Aleatório ----------
def _export_ids_map_to_csv(sel_map, fname):
    rows = []
    for (cls, c), ids in sel_map.items():
        for gid in sorted(ids):
            rows.append({"classe": int(cls), "cluster": int(c), "idx_global": int(gid)})
    pd.DataFrame(rows).sort_values(["classe","cluster","idx_global"]).to_csv(fname, index=False)

_export_ids_map_to_csv(PSEL, CSV_PROTODASH_ID)
_export_ids_map_to_csv(ENTRADAS_SELECIONADAS_ALEATORIO, CSV_ALEATORIAS_ID)

# ---------- Variáveis globais (não sobrescreve os dicts originais) ----------
globals()["ENTRADAS_SELECIONADAS_ALEATORIAS"] = ENTRADAS_SELECIONADAS_ALEATORIO
globals()["ENTRADAS_SELECIONADAS_ALEATORIAS_ID"] = ENTRADAS_SELECIONADAS_ALEATORIO
globals()["ENTRADAS_SELECIONADAS_PROTODASH_ID"] = PSEL

print(f"✅ Seleção aleatória criada em ENTRADAS_SELECIONADAS_ALEATORIO (dict por (classe,cluster)).")
print(f"📄 CSV seleção aleatória: {CSV_RANDOM_OUT}")
print(f"📄 CSV IDs ProtoDash: {CSV_PROTODASH_ID}")
print(f"📄 CSV IDs Aleatórias: {CSV_ALEATORIAS_ID}")
print(f"Resumo: clusters com seleção aleatória > 0 = {sum(len(v)>0 for v in ENTRADAS_SELECIONADAS_ALEATORIO.values())}")

# --- CSV completo com entradas ProtoDash (classe, cluster) ---
CSV_PROTODASH_COMPLETO = os.path.join(CSV_DIR, "entradas_selecionadas_protodash_completo.csv")
protodash_full_rows = []
csv_path_value = globals().get("CSV_TEST_PATH", "N/A")

# lookup de medoid global por (classe, cluster) se existir
medoid_lookup = {}
if 'results_by_class' in globals():
    for cls_key, res in results_by_class.items():
        idx_global_cls = np.asarray(res.get("idx_global", []), dtype=int)
        for clus_key, centers_loc in res.get("centers_by_cluster", {}).items():
            if centers_loc:
                loc0 = int(centers_loc[0])
                if 0 <= loc0 < len(idx_global_cls):
                    medoid_lookup[(int(cls_key), int(clus_key))] = int(idx_global_cls[loc0])

if PSEL:
    for (cls_key, clus_key), ids in PSEL.items():
        for iloc_val in sorted(int(gid) for gid in ids):
            if not 0 <= iloc_val < len(X_test):
                continue
            protodash_full_rows.append({
                'ID':       _safe_row_id(X_test, iloc_val, id_col_name="id"),
                'ILOC':     int(iloc_val),
                'CLASSE':   int(cls_key),
                'CLUSTER':  int(clus_key),
                'CSV_PATH': csv_path_value,
                'IS_MEDOID': int(medoid_lookup.get((int(cls_key), int(clus_key))) == int(iloc_val))
            })

if protodash_full_rows:
    df_protodash_full = (
        pd.DataFrame(protodash_full_rows)
        .sort_values(["CLASSE","CLUSTER","ILOC"])
        .reset_index(drop=True)
    )
    df_protodash_full.to_csv(CSV_PROTODASH_COMPLETO, index=False)
    print(f"📄 CSV ProtoDash completo salvo em: {CSV_PROTODASH_COMPLETO}")
else:
    print("[Aviso] Não há entradas ProtoDash para salvar em entradas_selecionadas_protodash_completo.csv")

# --- CSV completo com entradas aleatórias (classe, cluster) ---
CSV_ALEATORIAS_COMPLETO = os.path.join(CSV_DIR, "entradas_aleatorias_completo.csv")
aleatorias_full_rows = []
for (cls_key, clus_key), ids in ENTRADAS_SELECIONADAS_ALEATORIO.items():
    for iloc_val in sorted(int(gid) for gid in ids):
        if not 0 <= iloc_val < len(X_test):
            continue
        aleatorias_full_rows.append({
            'ID':       _safe_row_id(X_test, iloc_val, id_col_name="id"),
            'ILOC':     int(iloc_val),
            'CLASSE':   int(cls_key),
            'CLUSTER':  int(clus_key),
            'CSV_PATH': csv_path_value,
            'IS_MEDOID': int(medoid_lookup.get((int(cls_key), int(clus_key))) == int(iloc_val))
        })

if aleatorias_full_rows:
    df_aleatorias_full = (
        pd.DataFrame(aleatorias_full_rows)
        .sort_values(["CLASSE","CLUSTER","ILOC"])
        .reset_index(drop=True)
    )
    df_aleatorias_full.to_csv(CSV_ALEATORIAS_COMPLETO, index=False)
    print(f"📄 CSV Aleatório completo salvo em: {CSV_ALEATORIAS_COMPLETO}")
else:
    print("[Aviso] Não há entradas aleatórias para salvar em entradas_aleatorias_completo.csv")

# Visualização rápida
display(df_selecao_aleatorio)

# Listas achatadas úteis (sem destruir os dicts originais)
ENTRADAS_SELECIONADAS_PROTODASH_LIST = []
for v in PSEL.values():
    ENTRADAS_SELECIONADAS_PROTODASH_LIST.extend(sorted(v))

ENTRADAS_SELECIONADAS_ALEATORIAS_LIST = []
for v in ENTRADAS_SELECIONADAS_ALEATORIO.values():
    ENTRADAS_SELECIONADAS_ALEATORIAS_LIST.extend(sorted(v))

print(f"Total ProtoDash (únicos): {len(set(ENTRADAS_SELECIONADAS_PROTODASH_LIST))}")
print(f"Total Aleatórios (únicos): {len(set(ENTRADAS_SELECIONADAS_ALEATORIAS_LIST))}")

# Mantém globais úteis
globals()["ENTRADAS_SELECIONADAS_PROTODASH_LIST"] = ENTRADAS_SELECIONADAS_PROTODASH_LIST
globals()["ENTRADAS_SELECIONADAS_ALEATORIAS_LIST"] = ENTRADAS_SELECIONADAS_ALEATORIAS_LIST


In [ ]:
# Array de repetidos entre KCENTER e PROTODASH (índices globais)
# + Unificação em estrutura única de IDs consolidados — compatível com 3+ classes
# Cria:
#   ENTRADAS_SELECIONADAS_REPETIDAS  -> np.array([...], int)
#   ENTRADAS_SELECIONADAS            -> np.array([...], int) (KCenter ∪ ProtoDash ∪ Aleatório)
#   ENTRADAS_SELECIONADAS_ID         -> dict {(classe, cluster): sorted[list de ids únicos]}
# ============================================================
import numpy as np
import pandas as pd
import os

RESULTS_DIR = str(globals().get('RESULTS_DIR', '.'))
CSV_DIR = str(globals().get('OUT_CSV', globals().get('CSV_DIR', RESULTS_DIR)))
os.makedirs(CSV_DIR, exist_ok=True)
CSV_REPETIDOS     = os.path.join(CSV_DIR, 'repetidos_kcenter_protodash.csv')
CSV_UNIFICADO     = os.path.join(CSV_DIR, 'selecionados_unificados_ids.csv')
CSV_UNIFICADO_MAP = os.path.join(CSV_DIR, 'selecionados_unificados_por_classe_cluster.csv')

def _to_id_set(selection):
    """
    Converte diferentes formatos de 'selection' em um set de ids (iloc globais, int).
    Suporta:
      - dict plano {(classe,cluster): [ids]}
      - dict aninhado {classe: {cluster: [ids]}}
      - DataFrame com colunas ['classe','cluster', id_col] (id_col ∈ {'idx_global','idx_global_no_X_test','index_global','id_global'})
      - lista/array simples de ids
    """
    if selection is None:
        return set()

    # dict aninhado {classe: {cluster: [...]}}
    if isinstance(selection, dict) and all(isinstance(v, dict) for v in selection.values()):
        out = set()
        for dclus in selection.values():
            try:
                for v in dclus.values():
                    out |= set(int(x) for x in list(v))
            except Exception:
                pass
        return out

    # dict plano {(classe,cluster): [...]}
    if isinstance(selection, dict):
        out = set()
        for k, v in selection.items():
            try:
                if isinstance(k, tuple) and len(k) == 2:
                    out |= set(int(x) for x in list(v))
            except Exception:
                pass
        return out

    # DataFrame
    if isinstance(selection, pd.DataFrame):
        for col in ["idx_global", "idx_global_no_X_test", "index_global", "id_global"]:
            if col in selection.columns:
                try:
                    return set(int(x) for x in selection[col].tolist())
                except Exception:
                    return set()
        return set()

    # iterável simples
    try:
        return set(int(x) for x in list(selection))
    except Exception:
        return set()

def _to_map(selection):
    """
    Normaliza em dict {(classe, cluster): sorted list(ids)} para reconstruir granularidade.
    Suporta os mesmos formatos de _to_id_set (menos listas puras, que não têm classe/cluster).
    """
    m = {}
    if selection is None:
        return m

    # dict aninhado {classe: {cluster: [...]}}
    if isinstance(selection, dict) and all(isinstance(v, dict) for v in selection.values()):
        for cls, dclus in selection.items():
            try:
                icls = int(cls)
            except Exception:
                continue
            for c, v in dclus.items():
                try:
                    ic = int(c)
                    ids = sorted({int(x) for x in list(v)})
                    if ids:
                        m[(icls, ic)] = ids
                except Exception:
                    pass
        return m

    # dict plano {(classe,cluster): [...]}
    if isinstance(selection, dict):
        for k, v in selection.items():
            if isinstance(k, tuple) and len(k) == 2:
                try:
                    cls, c = int(k[0]), int(k[1])
                    ids = sorted({int(x) for x in list(v)})
                    if ids:
                        m[(cls, c)] = ids
                except Exception:
                    pass
        return m

    # DataFrame
    if isinstance(selection, pd.DataFrame):
        id_col = None
        for col in ["idx_global", "idx_global_no_X_test", "index_global", "id_global"]:
            if col in selection.columns:
                id_col = col
                break
        if id_col is None or not {'classe','cluster'} <= set(selection.columns):
            return m
        for (cls, c), grp in selection.groupby(['classe','cluster']):
            try:
                icls, ic = int(cls), int(c)
                ids = sorted({int(x) for x in grp[id_col].tolist()})
                if ids:
                    m[(icls, ic)] = ids
            except Exception:
                pass
        return m

    # lista/array simples não preserva classe/cluster → não há como mapear
    return m

# ---- Conjuntos de cada seleção ----
k_ids = _to_id_set(globals().get('ENTRADAS_SELECIONADAS_KCENTER', None))
p_ids = _to_id_set(globals().get('ENTRADAS_SELECIONADAS_PROTODASH', None))

# fallback ProtoDash via df_protos (se necessário)
if not p_ids and 'df_protos' in globals():
    try:
        p_ids = _to_id_set(df_protos.rename(columns={'idx_global_no_X_test': 'idx_global'}))
    except Exception:
        pass

# Aleatórios (nomes possíveis)
a_random_obj = (globals().get('ENTRADAS_SELECIONADAS_ALEATORIO', None) or
                globals().get('ENTRADAS_SELECIONADAS_ALEATORIAS', None))
a_ids = _to_id_set(a_random_obj)

# ---- Repetidos (KCenter ∩ ProtoDash) ----
ENTRADAS_SELECIONADAS_REPETIDAS = np.array(sorted(k_ids & p_ids), dtype=int)

# ---- União total ----
ENTRADAS_SELECIONADAS = np.array(sorted(k_ids | p_ids | a_ids), dtype=int)

# ---- Construção do mapa unificado (classe, cluster) -> ids únicos ----
map_k = _to_map(globals().get('ENTRADAS_SELECIONADAS_KCENTER', None))
map_p = _to_map(globals().get('ENTRADAS_SELECIONADAS_PROTODASH', None))
# ProtoDash via df_protos se mapa vazio
if not map_p and 'df_protos' in globals():
    try:
        df_aux = df_protos.rename(columns={'idx_global_no_X_test': 'idx_global'})[['classe','cluster','idx_global']]
        map_p = _to_map(df_aux)
    except Exception:
        pass
map_a = _to_map(a_random_obj)

all_keys = set(map_k.keys()) | set(map_p.keys()) | set(map_a.keys())
ENTRADAS_SELECIONADAS_ID = {}
for key in sorted(all_keys):
    ids = set()
    ids |= set(map_k.get(key, []))
    ids |= set(map_p.get(key, []))
    ids |= set(map_a.get(key, []))
    if ids:
        ENTRADAS_SELECIONADAS_ID[key] = sorted(ids)

# ---- CSVs ----
pd.DataFrame({'idx_global': ENTRADAS_SELECIONADAS_REPETIDAS}).to_csv(CSV_REPETIDOS, index=False)
pd.DataFrame({'idx_global': ENTRADAS_SELECIONADAS}).to_csv(CSV_UNIFICADO, index=False)

rows_map = []
for (cls, c), ids in ENTRADAS_SELECIONADAS_ID.items():
    for gid in ids:
        rows_map.append({'classe': int(cls), 'cluster': int(c), 'idx_global': int(gid)})
pd.DataFrame(rows_map).sort_values(['classe','cluster','idx_global']).to_csv(CSV_UNIFICADO_MAP, index=False)

# ---- Exposição em globals ----
globals()['ENTRADAS_SELECIONADAS_REPETIDAS'] = ENTRADAS_SELECIONADAS_REPETIDAS
globals()['ENTRADAS_SELECIONADAS'] = ENTRADAS_SELECIONADAS
globals()['ENTRADAS_SELECIONADAS_ID'] = ENTRADAS_SELECIONADAS_ID

print(f"✅ Repetidos (KCenter ∩ ProtoDash): {ENTRADAS_SELECIONADAS_REPETIDAS.size} itens → {CSV_REPETIDOS}")
print(f"✅ Unificado total: {ENTRADAS_SELECIONADAS.size} itens únicos → {CSV_UNIFICADO}")
print(f"✅ Mapa por (classe, cluster): {len(ENTRADAS_SELECIONADAS_ID)} chaves → {CSV_UNIFICADO_MAP}")

# Previews (seguros, sem assumir nº de classes)
print('\nPreview repetidos:', ENTRADAS_SELECIONADAS_REPETIDAS[:20])
print('Preview unificado :', ENTRADAS_SELECIONADAS[:20])
if ENTRADAS_SELECIONADAS_ID:
    print('Exemplo chave -> primeiros ids:')
    for i, ((cls, c), ids) in enumerate(ENTRADAS_SELECIONADAS_ID.items()):
        print((cls, c), ids[:10])
        if i >= 4:  # mostra só algumas chaves
            break

# CSV final "flat"
final_csv = os.path.join(CSV_DIR, 'entradas_selecionadas_final.csv')
pd.DataFrame({'idx_global': ENTRADAS_SELECIONADAS}).to_csv(final_csv, index=False)
print('Salvando entradas_selecionadas_final.csv em:', final_csv)


In [ ]:
# PCA 2D global — N classes (dinâmico)
# Subplots QUADRADOS maiores + legenda em uma única linha na base
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.transforms import Affine2D
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D
import warnings

# -------- helpers --------
def legend_totals_plugin(H, L, *, p_ids, k_ids, a_ids, repetidos_ids):
    sel_union = (set(p_ids) | set(k_ids) | set(a_ids))
    total_sel = len(sel_union)
    n_p, n_k, n_a, n_rep = len(p_ids), len(k_ids), len(a_ids), len(repetidos_ids)
    dummy1 = Line2D([], [], linestyle='none', marker=None, label="")
    dummy2 = Line2D([], [], linestyle='none', marker=None, label="")
    H = list(H) + [dummy1, dummy2]
    L = list(L) + [f"Selecionados (união): {total_sel}",
                   f"P:{n_p} | K:{n_k} | A:{n_a} | K∩P:{n_rep}"]
    return H, L

def _to_id_set(selection):
    """Suporta dict aninhado {classe:{cluster:[ids]}}, dict {(classe,cluster):[...]}, DataFrame, lista."""
    if selection is None:
        return set()
    if isinstance(selection, dict) and all(isinstance(v, dict) for v in selection.values()):
        out = set()
        for dclus in selection.values():
            for v in dclus.values():
                try: out |= set(int(x) for x in list(v))
                except Exception: pass
        return out
    if isinstance(selection, dict):
        out = set()
        for v in selection.values():
            try: out |= set(int(x) for x in list(v))
            except Exception: pass
        return out
    if isinstance(selection, pd.DataFrame):
        for col in ["idx_global", "idx_global_no_X_test", "index_global", "id_global"]:
            if col in selection.columns:
                try: return set(int(x) for x in selection[col].tolist())
                except Exception: return set()
        return set()
    try:
        return set(int(x) for x in list(selection))
    except Exception:
        return set()

def make_clover_marker(scale=0.65, offset=0.55):
    unit_circle = MplPath.unit_circle()
    tfs = [
        Affine2D().scale(scale).translate(0,  offset),
        Affine2D().scale(scale).translate(0, -offset),
        Affine2D().scale(scale).translate( offset, 0),
        Affine2D().scale(scale).translate(-offset, 0),
    ]
    paths = [tf.transform_path(unit_circle) for tf in tfs]
    return MplPath.make_compound_path(*paths)

# -------- coleta seleções --------
k_ids = _to_id_set(globals().get("ENTRADAS_SELECIONADAS_KCENTER", None))
p_ids = _to_id_set(globals().get("ENTRADAS_SELECIONADAS_PROTODASH", None))
a_obj = globals().get("ENTRADAS_SELECIONADAS_ALEATORIAS", None) or globals().get("ENTRADAS_SELECIONADAS_ALEATORIO", None)
a_ids = _to_id_set(a_obj)

# -------- conjuntos --------
repetidos_ids = k_ids & p_ids
k_unique = k_ids - repetidos_ids
p_unique = p_ids - repetidos_ids
a_plot   = a_ids

# -------- base do PCA: só numéricos, tratar NaN/inf --------
X_all_for_pca = DF_GLOBAL_NORM.copy()
X_all_for_pca = X_all_for_pca.drop(columns=EXCLUDE_COLUMNS, errors='ignore')

# tenta manter apenas numéricos; se não houver, tenta coerção
num_cols = X_all_for_pca.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    # coerção para numérico, inválidos viram NaN
    X_all_for_pca = X_all_for_pca.apply(pd.to_numeric, errors='coerce')
    num_cols = X_all_for_pca.columns.tolist()
else:
    X_all_for_pca = X_all_for_pca[num_cols]

# substituir inf por NaN
X_all_for_pca = X_all_for_pca.replace([np.inf, -np.inf], np.nan)

# preenche NaN com mediana da coluna; remove colunas que fiquem todas NaN
all_nan_cols = [c for c in X_all_for_pca.columns if X_all_for_pca[c].isna().all()]
if all_nan_cols:
    warnings.warn(f"Removendo colunas sem dados numéricos: {all_nan_cols}")
    X_all_for_pca = X_all_for_pca.drop(columns=all_nan_cols)

for c in X_all_for_pca.columns:
    if X_all_for_pca[c].isna().any():
        med = X_all_for_pca[c].median()
        if pd.isna(med):
            # se a mediana também é NaN (coluna vazia), remove
            X_all_for_pca = X_all_for_pca.drop(columns=[c])
        else:
            X_all_for_pca[c] = X_all_for_pca[c].fillna(med)

if X_all_for_pca.shape[1] < 2:
    raise RuntimeError(f"PCA requer >=2 colunas numéricas após limpeza; encontrado {X_all_for_pca.shape[1]}.")

# -------- rótulo de classe por índice global --------
n = X_all_for_pca.shape[0]
y_cls = np.full(n, -1, dtype=int)
for cls, res in results_by_class.items():
    y_cls[np.asarray(res["idx_global"], dtype=int)] = int(cls)

classes_sorted = sorted([int(c) for c in np.unique(y_cls) if c >= 0])
if not classes_sorted:
    raise RuntimeError("Nenhuma classe mapeada em y_cls.")

# -------- PCA global --------
Z_all = PCA(n_components=2, random_state=42).fit_transform(X_all_for_pca.values)

# -------- estética --------
default_colors = {0: "#1b5e20", 1: "#0d47a1", 2: "#8e24aa"}
tab = plt.cm.tab10.colors
class_colors = {}
for i, c in enumerate(classes_sorted):
    class_colors[c] = default_colors.get(c, tab[i % len(tab)])

alpha_bg = 0.60
color_protodash_unique = "#800080"
color_kcenter_unique   = "#2DA820"
color_random_unique    = "#EF120A"
color_repeat           = "#ff7f0e"
markers = {"protodash": "s", "kcenter": "^", "random": "*"}
clover_marker = make_clover_marker()

# -------- contagens para a legenda (globais) --------
n_p_unique   = len(p_unique)
n_k_unique   = len(k_unique)
n_a_unique   = len(a_plot)
n_repetidos  = len(repetidos_ids)

n_total = int(X_all_for_pca.shape[0])
labels_bg = {}
sel_union = (p_ids | k_ids | a_ids)
for c in classes_sorted:
    n_cl = int((y_cls == c).sum())
    pct_cl = 100.0 * n_cl / max(n_total, 1)
    sel_c  = len(sel_union & set(np.where(y_cls == c)[0]))
    pct_sel_c = 100.0 * sel_c / max(n_cl, 1)
    labels_bg[c] = f"classe {c} ({n_cl}) [{pct_cl:.0f}%] [{pct_sel_c:.0f}% selecionados]"

labels_map = {
    ("protodash","unique"): f"ProtoDash — únicos ({n_p_unique})",
    ("kcenter","unique")  : f"KCenter — únicos ({n_k_unique})",
    ("random","unique")   : f"Aleatório — únicos ({n_a_unique})",
    ("both","repeat")     : f"KCenter ∩ ProtoDash — repetidos ({n_repetidos})",
}

def _scatter_ids(ax, ids, marker, color, label, size=64):
    if not ids: return
    idx = [i for i in ids if 0 <= i < n]
    if not idx: return
    ax.scatter(
        Z_all[idx, 0], Z_all[idx, 1],
        s=size, marker=marker, c=color, edgecolors="k", linewidths=0.6, alpha=0.98, label=label
    )

def _plot_class(ax, cls):
    mask = (y_cls == cls)
    ax.scatter(Z_all[mask, 0], Z_all[mask, 1],
               s=22, alpha=alpha_bg, c=class_colors[cls], edgecolors="none", label=labels_bg[cls])
    cls_set = set(np.where(mask)[0])
    _scatter_ids(ax, repetidos_ids & cls_set, clover_marker, color_repeat, labels_map[("both","repeat")], size=120)
    _scatter_ids(ax, p_unique   & cls_set, markers["protodash"], color_protodash_unique, labels_map[("protodash","unique")])
    _scatter_ids(ax, k_unique   & cls_set, markers["kcenter"],   color_kcenter_unique,   labels_map[("kcenter","unique")])
    _scatter_ids(ax, a_plot     & cls_set, markers["random"],    color_random_unique,    labels_map[("random","unique")])
    ax.set_title(f"PCA 2D — pontos (classe {cls})")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.grid(True, alpha=0.3)

# -------- figura: 1 eixo por classe, quadrados, legenda única --------
n_classes = len(classes_sorted)
fig_w = 12 * min(n_classes, 3)
fig_h = 12
fig = plt.figure(figsize=(fig_w, fig_h))
gs = fig.add_gridspec(
    1, n_classes,
    width_ratios=[1]*n_classes,
    left=0.02, right=0.98, top=0.94, bottom=0.12,
    wspace=0.02
)

axes = []
for j, cls in enumerate(classes_sorted):
    ax = fig.add_subplot(gs[0, j])
    ax.set_box_aspect(1)
    _plot_class(ax, cls)
    axes.append(ax)

handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles += h; labels += l
seen = set(); H = []; L = []
for h, l in zip(handles, labels):
    if l and l not in seen:
        H.append(h); L.append(l); seen.add(l)

H, L = legend_totals_plugin(H, L, p_ids=p_ids, k_ids=k_ids, a_ids=a_ids, repetidos_ids=repetidos_ids)

fig.legend(
    H, L,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.04),
    ncol=max(1, len(L)),
    frameon=True,
    fontsize=13,
    columnspacing=1,
    handlelength=1.3
)

plt.show()


In [ ]:
# =========================[ CÉLULA 2 — AUTOSSUFICIENTE ]=========================
# PCA 2D + Regiões de Voronoi por CLASSE (clusters) + MESMAS marcações
# Funciona com 3 classes (ou N) e recalcula o PCA caso Z_ALL_PCA não exista.
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.spatial import Voronoi
from sklearn.decomposition import PCA
from matplotlib.path import Path as MplPath
from matplotlib.transforms import Affine2D
import warnings

# ---------- Helpers utilitários ----------
def _to_id_set(selection):
    """Suporta dict {(classe,cluster):[...]}, dict simples, DataFrame, lista/array."""
    if selection is None:
        return set()
    if isinstance(selection, dict) and selection and all(isinstance(v, dict) for v in selection.values()):
        out = set()
        for dclus in selection.values():
            for v in dclus.values():
                try: out |= set(int(x) for x in list(v))
                except Exception: pass
        return out
    if isinstance(selection, dict):
        out=set()
        for v in selection.values():
            try: out |= set(int(x) for x in list(v))
            except Exception: pass
        return out
    if isinstance(selection, pd.DataFrame):
        for col in ["idx_global", "idx_global_no_X_test", "index_global", "id_global"]:
            if col in selection.columns:
                try: return set(int(x) for x in selection[col].tolist())
                except Exception: return set()
        return set()
    try:
        return set(int(x) for x in list(selection))
    except Exception:
        return set()

def _voronoi_finite_polygons_2d(vor, radius=None):
    if vor.points.shape[1] != 2:
        raise ValueError("Apenas suporte 2D.")
    new_regions = []
    new_vertices = vor.vertices.tolist()
    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max()*2
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))
    for p1, region_index in enumerate(vor.point_region):
        vertices = vor.regions[region_index]
        if -1 not in vertices:
            new_regions.append(vertices); continue
        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]
        for p2, v1, v2 in ridges:
            if v2 < 0 or v1 < 0:
                v = v1 if v1 >= 0 else v2
                t = vor.points[p2] - vor.points[p1]
                t /= np.linalg.norm(t)
                n = np.array([-t[1], t[0]])
                midpoint = (vor.points[p1] + vor.points[p2]) / 2
                direction = np.sign(np.dot(midpoint - center, n)) * n
                far_point = vor.vertices[v] + direction * radius
                new_vertices.append(far_point.tolist())
                new_region.append(len(new_vertices) - 1)
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        ang = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = np.array(new_region)[np.argsort(ang)]
        new_regions.append(new_region.tolist())
    return new_regions, np.asarray(new_vertices)

def _clip_poly_to_box(poly, xlim, ylim):
    x0, x1 = xlim; y0, y1 = ylim
    poly = np.asarray(poly).copy()
    poly[:,0] = np.clip(poly[:,0], x0, x1)
    poly[:,1] = np.clip(poly[:,1], y0, y1)
    return poly

def make_clover_marker(scale=0.65, offset=0.55):
    unit_circle = MplPath.unit_circle()
    from matplotlib.lines import Line2D
    tfs = [
        Affine2D().scale(scale).translate(0,  offset),
        Affine2D().scale(scale).translate(0, -offset),
        Affine2D().scale(scale).translate( offset, 0),
        Affine2D().scale(scale).translate(-offset, 0),
    ]
    paths = [tf.transform_path(unit_circle) for tf in tfs]
    return MplPath.make_compound_path(*paths)

# ---------- Garantir PCA global e variáveis necessárias ----------
if 'Z_ALL_PCA' not in globals() or 'Y_CLS_VEC' not in globals():
    # Base numérica para PCA
    X_all_for_pca = DF_GLOBAL_NORM.drop(columns=EXCLUDE_COLUMNS, errors='ignore').copy()
    num_cols = X_all_for_pca.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) == 0:
        X_all_for_pca = X_all_for_pca.apply(pd.to_numeric, errors='coerce')
    else:
        X_all_for_pca = X_all_for_pca[num_cols]
    X_all_for_pca = X_all_for_pca.replace([np.inf, -np.inf], np.nan)
    # limpa colunas NaN
    all_nan_cols = [c for c in X_all_for_pca.columns if X_all_for_pca[c].isna().all()]
    if all_nan_cols:
        warnings.warn(f"Removendo colunas sem dados numéricos: {all_nan_cols}")
        X_all_for_pca = X_all_for_pca.drop(columns=all_nan_cols, errors='ignore')
    # preenche NaNs com mediana ou dropa se mediana NaN
    for c in list(X_all_for_pca.columns):
        if X_all_for_pca[c].isna().any():
            med = X_all_for_pca[c].median()
            if pd.isna(med): X_all_for_pca.drop(columns=[c], inplace=True)
            else: X_all_for_pca[c] = X_all_for_pca[c].fillna(med)
    if X_all_for_pca.shape[1] < 2:
        raise RuntimeError(f"PCA requer ≥2 colunas numéricas após limpeza; encontrado {X_all_for_pca.shape[1]}.")

    # y_cls a partir de results_by_class
    n = X_all_for_pca.shape[0]
    y_cls = np.full(n, -1, dtype=int)
    for cls, res in results_by_class.items():
        y_cls[np.asarray(res["idx_global"], dtype=int)] = int(cls)

    # PCA
    Z_all = PCA(n_components=2, random_state=42).fit_transform(X_all_for_pca.values)
    globals()['Z_ALL_PCA'] = Z_all
    globals()['Y_CLS_VEC'] = y_cls

    # Seleções (se não vieram da célula 1)
    k_ids = _to_id_set(globals().get("ENTRADAS_SELECIONADAS_KCENTER", None))
    p_ids = _to_id_set(globals().get("ENTRADAS_SELECIONADAS_PROTODASH", None))
    a_obj = (globals().get("ENTRADAS_SELECIONADAS_ALEATORIAS", None)
             or globals().get("ENTRADAS_SELECIONADAS_ALEATORIO", None))
    a_ids = _to_id_set(a_obj)
    repetidos_ids = k_ids & p_ids
    globals()['k_unique'] = k_ids - repetidos_ids
    globals()['p_unique'] = p_ids - repetidos_ids
    globals()['a_plot']   = a_ids
    globals()['repetidos_ids'] = repetidos_ids

    # Cores por classe (3 classes ok)
    classes_sorted = sorted([int(c) for c in np.unique(y_cls) if c >= 0])
    default_colors = {0: "#1b5e20", 1: "#0d47a1", 2: "#8e24aa"}
    tab = plt.cm.tab10.colors
    class_colors = {}
    for i, c in enumerate(classes_sorted):
        class_colors[c] = default_colors.get(c, tab[i % len(tab)])
    globals()['class_colors'] = class_colors

    # Labels de fundo
    n_total = int(X_all_for_pca.shape[0])
    labels_bg = {}
    sel_union = (p_ids | k_ids | a_ids)
    for c in classes_sorted:
        n_cls  = int((y_cls == c).sum())
        pct    = 100.0 * n_cls / max(n_total, 1)
        sel_c  = len(sel_union & set(np.where(y_cls == c)[0]))
        pctsel = 100.0 * sel_c / max(n_cls, 1)
        labels_bg[c] = f"classe {c} ({n_cls}) [{pct:.0f}%] [{pctsel:.0f}% selecionados]"
    globals()['labels_bg'] = labels_bg

    # Estética geral
    globals().setdefault('alpha_bg', 0.60)
    globals().setdefault('color_protodash_unique', "#800080")
    globals().setdefault('color_kcenter_unique',   "#2DA820")
    globals().setdefault('color_random_unique',    "#EF120A")
    globals().setdefault('color_repeat',           "#ff7f0e")
    globals().setdefault('markers', {"protodash": "s", "kcenter": "^", "random": "*"})
    globals().setdefault('clover_marker', make_clover_marker())
    # labels_map
    n_p_unique = len(globals()['p_unique'])
    n_k_unique = len(globals()['k_unique'])
    n_a_unique = len(globals()['a_plot'])
    n_repetidos = len(globals()['repetidos_ids'])
    globals()['labels_map'] = {
        ("protodash","unique"): f"ProtoDash — únicos ({n_p_unique})",
        ("kcenter","unique")  : f"KCenter — únicos ({n_k_unique})",
        ("random","unique")   : f"Aleatório — únicos ({n_a_unique})",
        ("both","repeat")     : f"KCenter ∩ ProtoDash — repetidos ({n_repetidos})",
    }
    # PCA_VIEW se não existir
    globals().setdefault('PCA_VIEW', {})

# ---------- A partir daqui, use as variáveis globais (existentes ou recém-criadas) ----------
Z_all   = globals()['Z_ALL_PCA']
y_cls   = globals()['Y_CLS_VEC']
class_colors           = globals()['class_colors']
alpha_bg               = globals()['alpha_bg']
color_protodash_unique = globals()['color_protodash_unique']
color_kcenter_unique   = globals()['color_kcenter_unique']
color_random_unique    = globals()['color_random_unique']
color_repeat           = globals()['color_repeat']
markers                = globals()['markers']
clover_marker          = globals()['clover_marker']
k_unique               = globals()['k_unique']
p_unique               = globals()['p_unique']
a_plot                 = globals()['a_plot']
repetidos_ids          = globals()['repetidos_ids']
labels_map             = globals()['labels_map']
labels_bg              = globals()['labels_bg']
PCA_VIEW               = globals().get('PCA_VIEW', {})

# --- parâmetros de clusters por classe ---
SILHUET = globals().get('SILHUET', {})  # ex: {0:3, 1:12, 2:5}
DEFAULT_K = 3

n = Z_all.shape[0]
classes_sorted = sorted([int(c) for c in np.unique(y_cls) if c >= 0])

def _scatter_ids(ax, ids, marker, color, label, size=64):
    if not ids: return
    idx = [i for i in ids if 0 <= i < n]
    if not idx: return
    ax.scatter(Z_all[idx, 0], Z_all[idx, 1],
               s=size, marker=marker, c=color, edgecolors="k", linewidths=0.6, alpha=0.98, label=label)

def _plot_class_with_voronoi(ax, cls, xlim=None, ylim=None):
    mask = (y_cls == cls)
    pts  = Z_all[mask]
    idx_class = np.where(mask)[0]
    # define K por classe
    try:
        K = int(SILHUET.get(int(cls), DEFAULT_K))
        if K < 1: K = DEFAULT_K
    except Exception:
        K = DEFAULT_K
    # k-means e centros
    if len(pts) >= max(2, K):
        km = KMeans(n_clusters=K, random_state=42, n_init=10)
        labels = km.fit_predict(pts)
        centers = km.cluster_centers_
    else:
        labels = np.zeros(len(pts), dtype=int)
        centers = pts.mean(axis=0, keepdims=True)
    # Voronoi dos centros
    if centers.shape[0] >= 2:
        vor = Voronoi(centers)
        regions, vertices = _voronoi_finite_polygons_2d(vor)
    else:
        vor = None; regions, vertices = [], np.empty((0,2))
    # limites (mantém da etapa 1 se existem)
    if xlim is None or ylim is None:
        if isinstance(PCA_VIEW, dict) and int(cls) in PCA_VIEW:
            xlim = PCA_VIEW[int(cls)].get('xlim', None)
            ylim = PCA_VIEW[int(cls)].get('ylim', None)
    if xlim is None or ylim is None:
        xx = Z_all[:,0]; yy = Z_all[:,1]
        pad_x = 0.05 * (xx.max() - xx.min() + 1e-9)
        pad_y = 0.05 * (yy.max() - yy.min() + 1e-9)
        xlim = (xx.min()-pad_x, xx.max()+pad_x)
        ylim = (yy.min()-pad_y, yy.max()+pad_y)
    # polígonos
    if vor is not None:
        for region in regions:
            polygon = vertices[region]
            polygon = _clip_poly_to_box(polygon, xlim, ylim)
            ax.fill(polygon[:,0], polygon[:,1],
                    facecolor=class_colors[cls], alpha=0.10, edgecolor=class_colors[cls], linewidth=0.8)
        ax.scatter(centers[:,0], centers[:,1],
                   s=110, c=class_colors[cls], marker='X', edgecolors='k', linewidths=0.8, alpha=0.95, label=f'centros (K={K})')
        offsets = [(8, 8), (8, -8), (-8, 8), (-8, -8)]
        for i, (cx, cy) in enumerate(centers):
            dx, dy = offsets[i % len(offsets)]
            ax.annotate(f'{i}', xy=(cx, cy), xytext=(dx, dy),
                        textcoords='offset points', ha='center', va='center',
                        fontsize=10, weight='bold', color='k',
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1.5))
    # fundo e marcações
    ax.scatter(pts[:,0], pts[:,1], s=20, alpha=alpha_bg, c=class_colors[cls], edgecolors="none", label=labels_bg[cls])
    cls_set = set(idx_class.tolist())
    _scatter_ids(ax, set(repetidos_ids) & cls_set, make_clover_marker(), color_repeat, labels_map[("both","repeat")], size=120)
    _scatter_ids(ax, set(p_unique) & cls_set, markers["protodash"], color_protodash_unique, labels_map[("protodash","unique")])
    _scatter_ids(ax, set(k_unique) & cls_set, markers["kcenter"],  color_kcenter_unique,   labels_map[("kcenter","unique")])
    _scatter_ids(ax, set(a_plot)   & cls_set, markers["random"],   color_random_unique,    labels_map[("random","unique")])
    ax.set_title(f"PCA 2D + Voronoi — classe {cls} (K={K})")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.grid(True, alpha=0.3)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_box_aspect(1)

# -------- figura: 1 painel por classe (3 colunas quando 3 classes) --------
n_classes = len(classes_sorted)
fig_w = 12 * min(n_classes, 3)
fig_h = 12
fig = plt.figure(figsize=(fig_w, fig_h))
gs = fig.add_gridspec(1, n_classes, width_ratios=[1]*n_classes,
                      left=0.02, right=0.98, top=0.94, bottom=0.12, wspace=0.02)

axes = []
for j, cls in enumerate(classes_sorted):
    ax = fig.add_subplot(gs[0, j])
    _plot_class_with_voronoi(ax, cls)
    axes.append(ax)

# Legenda única
handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles += h; labels += l
seen = set(); H = []; L = []
for h, l in zip(handles, labels):
    if l and l not in seen:
        H.append(h); L.append(l); seen.add(l)
fig.legend(H, L, loc="lower center", bbox_to_anchor=(0.5, 0.04),
           ncol=max(1, len(L)), frameon=True, fontsize=13, columnspacing=1.2, handlelength=1.3)

plt.show()


In [ ]:
# =========================[ CÉLULA 2 — COMPLETA (multi-classes) ]=========================
# PCA 2D + Regiões de Voronoi por CLASSE (clusters) + mesmas marcações
# - Suporta 3 classes (ou N)
# - Preenche a área com Voronoi (recorte Sutherland–Hodgman)
# - Círculo AMARELO translúcido contornando cada centroide (X)
# - Legenda por classe (pontos, X, círculo) com contagem de selecionados
# - Salva PNG e PDF em OUT_IMG
# ========================================================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Patch
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.spatial import Voronoi
import warnings

# ======== Parâmetros visuais (globais com fallback) ========
CLUSTER_BG_ALPHA   = float(globals().get('CLUSTER_BG_ALPHA', 0.18))
CLUSTER_LINE_COLOR = str(globals().get('CLUSTER_LINE_COLOR', "#056559"))
MEDOID_COLOR       = str(globals().get('MEDOID_COLOR', "#056559"))

# Cores dos pontos por classe (garante 0/1/2; expande com tab10 se houver mais)
POINT_COLORS = globals().get('POINT_COLORS', {0: "#0B900F", 1: "#0C78EC", 2: "#8e24aa"})

# ======== Diretórios de saída ========
OUT_IMG = Path(globals().get('OUT_IMG', Path('./images')))
OUT_IMG.mkdir(parents=True, exist_ok=True)
DPI = int(globals().get('DPI', 180))

# ======== Utilitários ========
def _to_id_set_safe(selection):
    """Converte dict/list/array/DataFrame em set[int] com tolerância a formatos."""
    if selection is None:
        return set()
    if isinstance(selection, dict):
        out = set()
        for v in selection.values():
            try:
                out |= set(int(x) for x in np.asarray(list(v)).ravel().tolist())
            except Exception:
                pass
        return out
    if isinstance(selection, pd.DataFrame):
        for col in ["idx_global", "idx_global_no_X_test", "index_global", "id_global"]:
            if col in selection.columns:
                try:
                    return set(int(x) for x in selection[col].tolist())
                except Exception:
                    return set()
        return set()
    try:
        arr = np.asarray(list(selection)).ravel()
        return set(int(x) for x in arr.tolist())
    except Exception:
        return set()

def _voronoi_finite_polygons_2d(vor, radius=None):
    """Converte células Voronoi infinitas em polígonos finitos."""
    if vor.points.shape[1] != 2:
        raise ValueError("Apenas suporte 2D.")
    new_regions, new_vertices = [], vor.vertices.tolist()
    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max()*2
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))
    for p1, region_idx in enumerate(vor.point_region):
        verts = vor.regions[region_idx]
        if verts and (-1 not in verts):
            new_regions.append(verts); continue
        ridges = all_ridges.get(p1, [])
        new_region = [v for v in (verts or []) if v >= 0]
        for p2, v1, v2 in ridges:
            if v1 < 0 or v2 < 0:
                v = v1 if v1 >= 0 else v2
                t = vor.points[p2] - vor.points[p1]
                norm = np.linalg.norm(t)
                if norm == 0:
                    continue
                t /= norm
                nvec = np.array([-t[1], t[0]])
                midpoint = (vor.points[p1] + vor.points[p2]) / 2
                direction = np.sign(np.dot(midpoint - center, nvec)) * nvec
                far_point = vor.vertices[v] + direction * radius
                new_vertices.append(far_point.tolist())
                new_region.append(len(new_vertices) - 1)
        if len(new_region) == 0:
            continue
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        ang = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = np.array(new_region)[np.argsort(ang)]
        new_regions.append(new_region.tolist())
    return new_regions, np.asarray(new_vertices)

def _segment_intersection(a, b, x0, x1, y0, y1):
    ax, ay = a; bx, by = b
    dx, dy = bx - ax, by - ay
    inters = []
    def add_if(x, y):
        if (x0-1e-9) <= x <= (x1+1e-9) and (y0-1e-9) <= y <= (y1+1e-9):
            inters.append([x, y])
    if dx != 0:
        t = (x0 - ax) / dx
        if 0 <= t <= 1: add_if(x0, ay + t*dy)
        t = (x1 - ax) / dx
        if 0 <= t <= 1: add_if(x1, ay + t*dy)
    if dy != 0:
        t = (y0 - ay) / dy
        if 0 <= t <= 1: add_if(ax + t*dx, y0)
        t = (y1 - ay) / dy
        if 0 <= t <= 1: add_if(ax + t*dx, y1)
    return inters[0] if inters else None

def _clip_polygon_to_box(poly, xlim, ylim):
    x0, x1 = xlim; y0, y1 = ylim
    def clip(subjectPolygon, inside_fn):
        output = []
        if len(subjectPolygon) == 0:
            return output
        s = subjectPolygon[-1]
        for e in subjectPolygon:
            if inside_fn(e):
                if not inside_fn(s):
                    inter = _segment_intersection(s, e, x0, x1, y0, y1)
                    if inter is not None: output.append(inter)
                output.append(e)
            elif inside_fn(s):
                inter = _segment_intersection(s, e, x0, x1, y0, y1)
                if inter is not None: output.append(inter)
            s = e
        return output
    def inside_left(p):   return p[0] >= x0
    def inside_right(p):  return p[0] <= x1
    def inside_bottom(p): return p[1] >= y0
    def inside_top(p):    return p[1] <= y1
    poly = np.asarray(poly, float).tolist()
    for fn in (inside_left, inside_right, inside_bottom, inside_top):
        poly = clip(poly, fn)
        if not poly: break
    return np.asarray(poly, float)

def _scatter_ids(ax, ids, Z_all, color, marker, label, size=64, edge='k'):
    if not ids: return
    n = Z_all.shape[0]
    idx = [i for i in ids if 0 <= i < n]
    if not idx: return
    ax.scatter(Z_all[idx, 0], Z_all[idx, 1],
               s=size, marker=marker, c=color, edgecolors=edge, linewidths=0.6, alpha=0.98, label=label)

# ======== Garantir PCA/variáveis se não vieram da célula 1 ========
if 'Z_ALL_PCA' not in globals() or 'Y_CLS_VEC' not in globals():
    # Base numérica para PCA: DF_GLOBAL_NORM - EXCLUDE_COLUMNS
    DF_GLOBAL_NORM = globals()['DF_GLOBAL_NORM']
    EXCLUDE_COLUMNS = globals().get('EXCLUDE_COLUMNS', [])
    X_all_for_pca = DF_GLOBAL_NORM.drop(columns=EXCLUDE_COLUMNS, errors='ignore').copy()
    num_cols = X_all_for_pca.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) == 0:
        X_all_for_pca = X_all_for_pca.apply(pd.to_numeric, errors='coerce')
    else:
        X_all_for_pca = X_all_for_pca[num_cols]
    X_all_for_pca = X_all_for_pca.replace([np.inf, -np.inf], np.nan)
    all_nan_cols = [c for c in X_all_for_pca.columns if X_all_for_pca[c].isna().all()]
    if all_nan_cols:
        warnings.warn(f"Removendo colunas sem dados numéricos: {all_nan_cols}")
        X_all_for_pca = X_all_for_pca.drop(columns=all_nan_cols, errors='ignore')
    for c in list(X_all_for_pca.columns):
        if X_all_for_pca[c].isna().any():
            med = X_all_for_pca[c].median()
            if pd.isna(med): X_all_for_pca.drop(columns=[c], inplace=True)
            else: X_all_for_pca[c] = X_all_for_pca[c].fillna(med)
    if X_all_for_pca.shape[1] < 2:
        raise RuntimeError(f"PCA requer ≥2 colunas numéricas; encontrado {X_all_for_pca.shape[1]}.")

    # y_cls a partir de results_by_class
    results_by_class = globals()['results_by_class']
    n_total = X_all_for_pca.shape[0]
    y_cls = np.full(n_total, -1, dtype=int)
    for cls, res in results_by_class.items():
        y_cls[np.asarray(res["idx_global"], dtype=int)] = int(cls)

    Z_all = PCA(n_components=2, random_state=42).fit_transform(X_all_for_pca.values)
    globals()['Z_ALL_PCA'] = Z_all
    globals()['Y_CLS_VEC'] = y_cls
else:
    Z_all = globals()['Z_ALL_PCA']
    y_cls = globals()['Y_CLS_VEC']

# Seleções (com fallback para nomes alternativos)
p_ids = _to_id_set_safe(globals().get("ENTRADAS_SELECIONADAS_PROTODASH", None))
k_ids = _to_id_set_safe(globals().get("ENTRADAS_SELECIONADAS_KCENTER", None))
a_ids = _to_id_set_safe(globals().get("ENTRADAS_SELECIONADAS_ALEATORIAS", None)
                        or globals().get("ENTRADAS_SELECIONADAS_ALEATORIO", None))
repetidos_ids = p_ids & k_ids
p_unique = p_ids - repetidos_ids
k_unique = k_ids - repetidos_ids
a_plot   = a_ids
sel_union = p_ids | k_ids | a_ids

# Labels de fundo por classe (contagem + % selecionados)
classes_sorted = sorted([int(c) for c in np.unique(y_cls) if c >= 0])
n_total = Z_all.shape[0]
labels_bg = {}
for c in classes_sorted:
    n_c  = int((y_cls == c).sum())
    sel_c = len(sel_union & set(np.where(y_cls == c)[0]))
    pct   = 100.0 * n_c / max(n_total, 1)
    pctsel= 100.0 * sel_c / max(n_c, 1)
    labels_bg[c] = f"classe {c} (N={n_c}) [{pct:.0f}%] [{pctsel:.0f}% sel.]"

# SILHUET e K default
SILHUET   = globals().get('SILHUET', {})   # ex.: {0:3, 1:8, 2:5}
DEFAULT_K = int(globals().get('DEFAULT_K', 3))

# ======== Função de plot de cada classe ========
def _plot_class_with_voronoi(ax, cls, xlim=None, ylim=None):
    mask = (y_cls == cls)
    pts  = Z_all[mask]
    idx_class = np.where(mask)[0]
    n_cls = len(idx_class)

    # K por classe
    try:
        K = int(SILHUET.get(int(cls), DEFAULT_K))
        if K < 1: K = DEFAULT_K
    except Exception:
        K = DEFAULT_K

    # KMeans e centros (usa n_init=10 p/ compatibilidade ampla)
    if len(pts) >= max(1, K):
        km = KMeans(n_clusters=max(1, K), random_state=42, n_init=10)
        labels = km.fit_predict(pts)
        centers = km.cluster_centers_
    else:
        labels = np.zeros(len(pts), int) if len(pts) else np.array([], int)
        centers = pts.mean(axis=0, keepdims=True) if len(pts) else np.array([[0.0, 0.0]])

    # Voronoi dos centros
    if centers.shape[0] >= 2:
        vor = Voronoi(centers)
        regions, vertices = _voronoi_finite_polygons_2d(vor)
    else:
        vor = None
        regions, vertices = [], np.empty((0, 2))

    # limites (mantém faixa confortável)
    xx = Z_all[:,0]; yy = Z_all[:,1]
    pad_x = 0.05 * (xx.max() - xx.min() + 1e-9)
    pad_y = 0.05 * (yy.max() - yy.min() + 1e-9)
    if xlim is None: xlim = (xx.min()-pad_x, xx.max()+pad_x)
    if ylim is None: ylim = (yy.min()-pad_y, yy.max()+pad_y)

    # fundo: pontos da classe
    if len(pts):
        ax.scatter(pts[:,0], pts[:,1],
                   s=20, alpha=0.60,
                   c=POINT_COLORS.get(int(cls), "#999999"),
                   edgecolors="none", label=labels_bg.get(int(cls), f"classe {cls}"))

    # polígonos Voronoi recortados para PREENCHER o quadrado
    if vor is not None and len(regions) > 0:
        for region in regions:
            polygon = vertices[region]
            polygon = _clip_polygon_to_box(polygon, xlim, ylim)
            if len(polygon) >= 3:
                ax.fill(polygon[:,0], polygon[:,1],
                        facecolor=POINT_COLORS.get(int(cls), "#999999"),
                        alpha=CLUSTER_BG_ALPHA,
                        edgecolor=CLUSTER_LINE_COLOR,
                        linewidth=0.8)

    # centros: X + círculo amarelo translúcido
    if centers.size:
        ax.scatter(centers[:,0], centers[:,1],
                   s=130, c=MEDOID_COLOR, marker='X', edgecolors='k', linewidths=0.9, alpha=0.95,
                   label=f'centros (K={centers.shape[0]})')
        rx = (xlim[1] - xlim[0]); ry = (ylim[1] - ylim[0])
        base_r = 0.035 * max(rx, ry)
        for (cx, cy) in centers:
            circ = Circle((cx, cy), base_r,
                          facecolor=(1.0, 1.0, 0.0, 0.12),
                          edgecolor='yellow', linewidth=2.0)
            ax.add_patch(circ)

        # rótulos deslocados (não sobrepor o X)
        offsets = [(9, 9), (9, -9), (-9, 9), (-9, -9)]
        for i, (cx, cy) in enumerate(centers):
            dx, dy = offsets[i % len(offsets)]
            ax.annotate(f'{i}', xy=(cx, cy), xytext=(dx, dy),
                        textcoords='offset points', ha='center', va='center',
                        fontsize=10, weight='bold', color='k',
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=1.5))

    # marcações adicionais (limitadas à classe)
    cls_set = set(idx_class.tolist())
    _scatter_ids(ax, (repetidos_ids & cls_set), Z_all, "#ff7f0e", 'p', "K∩P — repetidos", size=120, edge='k')
    _scatter_ids(ax, (p_unique - repetidos_ids) & cls_set, Z_all, "#800080", 's', "ProtoDash — únicos")
    _scatter_ids(ax, (k_unique - repetidos_ids) & cls_set, Z_all, "#2DA820", '^', "KCenter — únicos")
    _scatter_ids(ax, a_plot & cls_set, Z_all, "#EF120A", '*', "Aleatório — únicos")

    # estética
    ax.set_title(f"PCA 2D + Voronoi — classe {cls}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_box_aspect(1)

# ======== Figura: 1 painel por classe (3 classes => 3 colunas) ========
nC = len(classes_sorted)
fig_w = 12 * min(nC, 3)      # largura cresce até 3 colunas
fig_h = 12
fig = plt.figure(figsize=(fig_w, fig_h))
gs = fig.add_gridspec(1, nC, width_ratios=[1]*nC,
                      left=0.02, right=0.98, top=0.90, bottom=0.18, wspace=0.02)

axes = []
for j, cls in enumerate(classes_sorted):
    ax = fig.add_subplot(gs[0, j])
    _plot_class_with_voronoi(ax, cls)
    axes.append(ax)

# ======== Legenda: UMA LINHA por classe (3 itens: pontos, X, círculo) ========
proxies = []
labels  = []
for cls in classes_sorted:
    sel_c = len(sel_union & set(np.where(y_cls == cls)[0]))
    n_c   = int((y_cls == cls).sum())
    pts  = Line2D([], [], linestyle='none', marker='o', markersize=8,
                  markerfacecolor=POINT_COLORS.get(int(cls), "#999999"), markeredgecolor='none')
    cent = Line2D([], [], linestyle='none', marker='X', markersize=10,
                  markerfacecolor=MEDOID_COLOR, markeredgecolor='k')
    ring = Patch(facecolor=(1.0,1.0,0.0,0.12), edgecolor='yellow', linewidth=2.0)
    proxies += [pts, cent, ring]
    labels  += [f"Classe {cls} — sel={sel_c} | total={n_c}", "Centros (X)", "Círculo dos centros"]

fig.legend(proxies, labels,
           loc="lower center", bbox_to_anchor=(0.5, 0.06),
           ncol=3, frameon=True, fontsize=12, columnspacing=1.4, handlelength=1.6)

# ======== Salvar ========
suffix = f"{nC}_classes"
fname_png = OUT_IMG / f"pca_voronoi_{suffix}.png"
fname_pdf = OUT_IMG / f"pca_voronoi_{suffix}.pdf"
fig.savefig(fname_png, dpi=DPI, bbox_inches="tight")
fig.savefig(fname_pdf, dpi=DPI, bbox_inches="tight")
print(f"📁 Figuras salvas em:\n - {fname_png}\n - {fname_pdf}")

plt.show()


In [ ]:
# ====================== PDF resumo com suporte a 3+ classes ======================
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import math
import textwrap
import re
import os

pdf_dir = Path(globals().get('OUT_PDF', Path('.')))
pdf_dir.mkdir(parents=True, exist_ok=True)
pdf_path = pdf_dir / 'RF_CLUSTERS_SUMMARY.pdf'

# --- diretórios candidatos p/ buscar CSV/IMG ---
results_dirs = []
for key in ('RESULTS_DIR', 'OUT_CSV', 'OUT_BASE'):
    base = globals().get(key)
    if not base:
        continue
    try:
        candidate = Path(base)
    except TypeError:
        try:
            candidate = Path(str(base))
        except Exception:
            candidate = None
    if candidate and candidate not in results_dirs:
        results_dirs.append(candidate)
if 'OUT_IMG' in globals():
    try:
        candidate = Path(globals()['OUT_IMG'])
        if candidate not in results_dirs:
            results_dirs.append(candidate)
    except Exception:
        pass
parent_dir = pdf_dir.parent
if parent_dir not in results_dirs:
    results_dirs.append(parent_dir)

def _safe_df(name):
    obj = globals().get(name)
    return obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame()

def _load_if_empty(df, candidates):
    if not df.empty:
        return df
    for cand in candidates:
        if not cand:
            continue
        try:
            cand_path = Path(cand)
        except TypeError:
            try: cand_path = Path(str(cand))
            except Exception: continue
        search_paths = []
        if cand_path.is_absolute():
            search_paths.append(cand_path)
        else:
            search_paths.extend(base / cand_path for base in results_dirs)
        for path in search_paths:
            try:
                if path.exists() and path.is_file():
                    return pd.read_csv(path)
            except Exception:
                continue
    return df

def _collect_file_summary(extensions):
    extensions = {ext.lower().lstrip('.') for ext in extensions if ext}
    if not extensions:
        return pd.DataFrame()
    rows, seen = [], set()
    for base in results_dirs:
        try:
            base_path = Path(base)
        except TypeError:
            continue
        if not base_path.exists():
            continue
        for path in base_path.rglob('*'):
            if not path.is_file(): continue
            suffix = path.suffix.lower().lstrip('.')
            if suffix not in extensions: continue
            if path in seen: continue
            seen.add(path)
            try:
                stat = path.stat()
            except OSError:
                continue
            try:
                rel_parent = path.parent.relative_to(base_path)
                rel_parent_str = str(rel_parent) if str(rel_parent) != '.' else '.'
            except ValueError:
                rel_parent_str = str(path.parent)
            rows.append({
                'arquivo': path.name,
                'pasta': rel_parent_str,
                'tamanho_kb': round(stat.st_size / 1024.0, 1),
                'modificado_em': datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M'),
                'full_path': str(path)
            })
    if not rows: return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(['pasta', 'arquivo']).reset_index(drop=True)

# -------- Agrupamento de imagens por “base name” (multi-classes) --------
# Detecta tokens "classe X" (ou "class X" / "c X") e usa o restante do nome como chave.
CLASS_PAT = re.compile(r'(classe|class|c)\s*[-_ ]*\s*(\d+)', flags=re.IGNORECASE)

def _split_base_and_class(name: str):
    """Retorna (base_name, class_id ou None)."""
    nm = name
    m = CLASS_PAT.search(nm)
    if not m:
        return (nm.lower(), None)
    cid = m.group(2)
    # remove token "classe X" do base
    base = CLASS_PAT.sub('classe *', nm).lower()
    return (base, int(cid))

def _group_image_entries_generic(entries):
    """
    Agrupa entradas por base_name; cada grupo pode ter 1, 2, 3+ classes.
    Retorna lista de grupos (cada grupo = lista de entries ordenadas por classe)
    e lista de 'singles' sem classe detectada.
    """
    groups = {}
    singles = []
    for e in entries:
        base, clsid = _split_base_and_class(e['name'])
        if clsid is None:
            singles.append(e)
        else:
            groups.setdefault(base, {})[clsid] = e
    grouped = []
    for base, d in groups.items():
        ordered = [d[k] for k in sorted(d.keys())]
        grouped.append(ordered)
    return grouped, singles

# -------- Páginas --------
def _image_page(pdf, entries, title, landscape=True):
    if not entries:
        return
    cols = len(entries)
    figsize = (11.69, 8.27) if landscape else (8.27, 11.69)
    fig, axes = plt.subplots(1, cols, figsize=figsize)
    if cols == 1:
        axes = [axes]
    for ax, entry in zip(axes, entries):
        try:
            img = mpimg.imread(str(entry['path']))
            ax.imshow(img); ax.axis('off')
        except Exception:
            ax.text(0.5, 0.5, 'Erro ao carregar imagem', ha='center', va='center', fontsize=10)
            ax.axis('off')
        subtitle_parts = [entry.get('name', ''), entry.get('pasta', '')]
        subtitle = ' — '.join(part for part in subtitle_parts if part)
        if subtitle: ax.set_title(subtitle, fontsize=10)
    if title:
        fig.suptitle(title, fontsize=14, fontweight='bold')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

def _text_page(pdf, title, lines, figsize=(8.27, 11.69), fontsize=11):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    ax.set_title(title, pad=20, fontsize=16, fontweight='bold')
    wrapped = []
    for line in lines:
        wrapped_lines = textwrap.wrap(str(line), width=95) or ['']
        wrapped.extend(wrapped_lines)
    ax.text(0, 1, '\n'.join(wrapped), ha='left', va='top', fontsize=fontsize, family='monospace')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

def _table_pages(pdf, df, title, max_rows=24, landscape=True, font_size=8):
    if df.empty: return
    df = df.copy()
    df.replace({np.nan: ''}, inplace=True)
    total = len(df)
    pages = max(1, math.ceil(total / max_rows))
    figsize = (11.69, 8.27) if landscape else (8.27, 11.69)
    for page in range(pages):
        subset = df.iloc[page * max_rows:(page + 1) * max_rows]
        fig, ax = plt.subplots(figsize=figsize)
        ax.axis('off')
        ax.set_title(f"{title} (pagina {page + 1}/{pages})", pad=20, fontsize=14, fontweight='bold')
        table = ax.table(cellText=subset.values, colLabels=list(subset.columns), loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(font_size)
        table.scale(1.0, 1.1 if font_size <= 8 else 1.0)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

# -------- Carregamentos opcionais --------
df_cluster = _safe_df('df_cluster_metrics')
df_cluster = _load_if_empty(df_cluster, [globals().get('CSV_CLUSTER'), 'protodash_repr_por_cluster.csv'])
df_cluster_view = _safe_df('df_cluster_metrics_view')
if df_cluster_view.empty and not df_cluster.empty:
    df_cluster_view = df_cluster.copy()

df_resumo = _safe_df('df_resumo_cobertura')
df_resumo = _load_if_empty(df_resumo, ['resumo_cobertura_requirements.csv', globals().get('df_resumo_path')])

df_diag = _safe_df('df_diag')
df_diag = _load_if_empty(df_diag, ['diagnostico_requirements.csv', globals().get('df_diag_path')])

df_resumo_sil = _safe_df('df_resumo_silhueta')
df_resumo_sil = _load_if_empty(df_resumo_sil, [
    'resumo_silhueta_usado_vs_sugerido.csv',
    'resumo_silhueta_usado_vs_sugerido_recomendado.csv',
    'resumo_silhueta_usado_vs_sugerido_usuario.csv'
])

df_elbow_rec = _safe_df('df_resumo_elbow_rec')
df_elbow_rec = _load_if_empty(df_elbow_rec, ['resumo_elbow_usado_vs_sugerido_recomendado.csv'])

df_elbow_user = _safe_df('df_resumo_elbow_user')
df_elbow_user = _load_if_empty(df_elbow_user, ['resumo_elbow_usado_vs_sugerido_usuario.csv'])

df_m_needed = _safe_df('df_m_needed')
df_m_needed = _load_if_empty(df_m_needed, ['kcenter_m_necessario_por_cluster_por_meta.csv'])

csv_summary_df = _collect_file_summary({'.csv'})
img_summary_df = _collect_file_summary({'.png', '.jpg', '.jpeg', '.svg', '.webp'})

# -------- PDF --------
with PdfPages(pdf_path) as pdf:
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M')

    # valores de capa com fallback
    CSV_TRAIN_PATH   = globals().get('CSV_TRAIN_PATH', 'N/A')
    CSV_TEST_PATH    = globals().get('CSV_TEST_PATH', 'N/A')
    TARGET_COLUMN    = globals().get('TARGET_COLUMN', 'N/A')
    N_CLUSTERS       = globals().get('N_CLUSTERS', 'N/A')
    THRESHOLD        = globals().get('THRESHOLD', 'N/A')
    DATASET_NORMALIZED = globals().get('DATASET_NORMALIZED', 'N/A')
    NORMALIZED_METHOD  = globals().get('NORMALIZED_METHOD', 'N/A')
    OUT_CSV = globals().get('OUT_CSV', 'N/A')
    OUT_IMG = globals().get('OUT_IMG', 'N/A')
    OUT_PDF = globals().get('OUT_PDF', 'N/A')

    cover_lines = [
        f'Data de geracao: {now_str}',
        'Notebook: RF_Cluster_from_RF_predictions.ipynb',
        f'CSV treino: {CSV_TRAIN_PATH}',
        f'CSV teste : {CSV_TEST_PATH}',
        f'Coluna alvo/base: {TARGET_COLUMN}',
        f'Clusters por classe: {N_CLUSTERS}',
        f'Threshold de decisao: {THRESHOLD}',
        f'Dados normalizados previamente?: {DATASET_NORMALIZED}',
        f'Metodo de normalizacao: {NORMALIZED_METHOD}',
        'Diretorios de saida:',
        f'  CSV : {OUT_CSV}',
        f'  IMG : {OUT_IMG}',
        f'  PDF : {OUT_PDF}',
    ]
    _text_page(pdf, 'Relatorio de Clusters - Random Forest', cover_lines)

    total_clusters = int(df_cluster.shape[0]) if not df_cluster.empty else 0
    total_protos = int(df_cluster['m_protos'].sum()) if 'm_protos' in df_cluster.columns else 0
    avaliacao_counts = df_cluster['avaliacao'].value_counts().to_dict() if 'avaliacao' in df_cluster.columns else {}
    classe_counts = (df_cluster.groupby('classe')['cluster'].nunique().to_dict()
                     if (not df_cluster.empty and 'cluster' in df_cluster.columns) else {})
    metrics_rows = []
    results_dict = globals().get('results_by_class', {})
    if isinstance(results_dict, dict):
        for cls, info in results_dict.items():
            metrics = info.get('metrics', {}) if isinstance(info, dict) else {}
            labels = info.get('labels', []) if isinstance(info, dict) else []
            metrics_rows.append({
                'classe': cls,
                'silhouette': metrics.get('silhouette'),
                'davies_bouldin': metrics.get('davies_bouldin'),
                'calinski_harabasz': metrics.get('calinski_harabasz'),
                'n_clusters_detectados': len(np.unique(labels)) if hasattr(labels, '__len__') else np.nan
            })
    metrics_df = pd.DataFrame(metrics_rows)

    summary_lines = [
        f'Total de clusters avaliados: {total_clusters}',
        f'Total de prototipos selecionados: {total_protos}',
    ]
    if classe_counts:
        summary_lines.append('Clusters unicos por classe: ' + ', '.join(f'{cls}: {cnt}' for cls, cnt in classe_counts.items()))
    if avaliacao_counts:
        summary_lines.append('Avaliacoes (BOM/OK/RUIM): ' + ', '.join(f'{k}: {v}' for k, v in avaliacao_counts.items()))
    summary_lines.append(f'PDF salvo em: {pdf_path}')
    _text_page(pdf, 'Sumario Geral', summary_lines, figsize=(11.69, 8.27), fontsize=10)

    if not metrics_df.empty:
        _table_pages(pdf, metrics_df.round(4), 'Metricas por classe', max_rows=18, landscape=True, font_size=8)
    _table_pages(pdf, df_resumo.round(3), 'Resumo de cobertura por cluster', max_rows=18, landscape=True, font_size=8)
    _table_pages(pdf, df_diag.round(3), 'Diagnostico de requisitos', max_rows=18, landscape=True, font_size=8)
    _table_pages(pdf, df_resumo_sil.round(3), 'Resumo de silhueta (recomendado vs usuario)', max_rows=18, landscape=True, font_size=8)
    _table_pages(pdf, df_elbow_rec.round(3), 'Resumo elbow recomendado', max_rows=18, landscape=True, font_size=8)
    _table_pages(pdf, df_elbow_user.round(3), 'Resumo elbow informado pelo usuario', max_rows=18, landscape=True, font_size=8)
    _table_pages(pdf, df_cluster_view.round(4), 'Metricas principais por cluster', max_rows=12, landscape=True, font_size=8)

    if not df_cluster_view.empty and 'analise' in df_cluster_view.columns:
        analise_df = df_cluster_view[['classe', 'cluster', 'avaliacao', 'analise']].copy()
        _table_pages(pdf, analise_df, 'Analises textuais por cluster', max_rows=14, landscape=True, font_size=8)

    if not df_resumo.empty:
        if 'ganho_compactacao_%' not in df_resumo.columns and 'r_percent_%' in df_resumo.columns:
            try: df_resumo['ganho_compactacao_%'] = 100.0 - df_resumo['r_percent_%']
            except Exception: df_resumo['ganho_compactacao_%'] = np.nan
        destaque_cols = ['classe', 'cluster', 'percentual_selecionados_%', 'ganho_compactacao_%', 'observacao']
        disponiveis = [c for c in destaque_cols if c in df_resumo.columns]
        if disponiveis:
            destaques = df_resumo[disponiveis].copy()
            for col in destaque_cols:
                if col not in destaques.columns:
                    destaques[col] = np.nan
            destaques = destaques[destaque_cols]
            _table_pages(pdf, destaques, 'Cobertura - destaques principais', max_rows=16, landscape=True, font_size=8)

    if not df_cluster.empty and 'avaliacao' in df_cluster.columns:
        agg_map = {'m_protos': 'sum'}
        if 'N_eff (↑)' in df_cluster.columns:
            agg_map['N_eff (↑)'] = 'mean'
        agg_df = df_cluster.groupby(['classe', 'avaliacao']).agg(agg_map).reset_index()
        if 'N_eff (↑)' in agg_df.columns:
            agg_df.rename(columns={'N_eff (↑)': 'N_eff_medio'}, inplace=True)
        _table_pages(pdf, agg_df.round(3), 'Agrupamento classe x avaliacao', max_rows=18, landscape=True, font_size=8)

    if not df_m_needed.empty:
        _table_pages(pdf, df_m_needed.round(2), 'M necessario por cluster alvo', max_rows=18, landscape=True, font_size=8)

    if not csv_summary_df.empty:
        csv_table = csv_summary_df.drop(columns=['full_path']) if 'full_path' in csv_summary_df.columns else csv_summary_df
        _table_pages(pdf, csv_table, 'Sumario de arquivos CSV', max_rows=25, landscape=True, font_size=8)

    if not img_summary_df.empty:
        img_table = img_summary_df.drop(columns=['full_path']) if 'full_path' in img_summary_df.columns else img_summary_df
        _table_pages(pdf, img_table, 'Sumario de imagens geradas', max_rows=25, landscape=True, font_size=8)

        # Galeria de imagens agrupadas por base (classe 0/1/2 …)
        image_entries = []
        for _, row in img_summary_df.iterrows():
            full_path = row.get('full_path')
            if not full_path: continue
            path = Path(full_path)
            if not path.exists(): continue
            image_entries.append({
                'path': path,
                'name': row.get('arquivo', path.name),
                'pasta': row.get('pasta', '')
            })
        if image_entries:
            groups, singles = _group_image_entries_generic(image_entries)
            # grupos com múltiplas classes (até 3 por página)
            for group in groups:
                title = f"Comparativo por classe: " + " | ".join(g['name'] for g in group)
                # se grupo tiver mais de 3, pagina em blocos de 3
                for i in range(0, len(group), 3):
                    _image_page(pdf, group[i:i+3], title if i == 0 else None, landscape=True)
            # imagens isoladas (sem classe no nome)
            for single in singles:
                _image_page(pdf, [single], single['name'], landscape=False)

print(f'PDF resumo gerado em: {pdf_path}')


In [ ]:
import pandas as pd
from pathlib import Path

# ============================================
# Caminhos
# ============================================
BASE = Path("../UCI-THYROID-DXBIN")

PATH_IDX   = BASE / "clusters_reports/random_forest/csv/entradas_selecionadas_final.csv"

PATH_TEST  = BASE / "model_reports/random_forest/basico/csv/X_test_basic_full.csv"
PATH_TRAIN = BASE / "model_reports/random_forest/basico/csv/X_train_basic_full.csv"

OUT_PATH_FULL = BASE / "clusters_reports/random_forest/csv/entradas_selecionadas_final_id.csv"
OUT_PATH_ID   = BASE / "clusters_reports/random_forest/csv/entradas_selecionadas_final_id_only.csv"

# ============================================
# 1) Carregar arquivos
# ============================================
df_idx   = pd.read_csv(PATH_IDX)
df_test  = pd.read_csv(PATH_TEST)
df_train = pd.read_csv(PATH_TRAIN)

# ============================================
# 2) Juntar train + test (ambos têm orig_index e id)
# ============================================
df_full = pd.concat([df_train, df_test], axis=0, ignore_index=True)
df_full.rename(columns={"Unnamed: 0": "orig_index"}, inplace=True)

# Garantir que patient_id é int
df_full["patient_id"] = df_full["patient_id"].astype(int)

# ============================================
# 3) Índices selecionados (idx_global)
# ============================================
idx_global = df_idx["idx_global"].astype(int).tolist()

# ============================================
# 4) Selecionar linhas pela coluna orig_index
#    (orig_index == idx_global)
# ============================================
df_sel = df_full[df_full["orig_index"].isin(idx_global)].copy()

# Opcional: garantir ordenação pelo idx_global
df_sel = df_sel.sort_values("idx_global")
df_sel


In [ ]:
import pandas as pd
from pathlib import Path

# ============================================
# Caminhos
# ============================================
BASE = Path("../UCI-THYROID-DXBIN")

PATH_IDX   = BASE / "clusters_reports/random_forest/csv/entradas_selecionadas_final.csv"

PATH_TEST  = BASE / "model_reports/random_forest/basico/csv/X_test_basic_full.csv"
PATH_TRAIN = BASE / "model_reports/random_forest/basico/csv/X_train_basic_full.csv"

OUT_PATH_FULL = BASE / "clusters_reports/random_forest/csv/entradas_selecionadas_final_id.csv"
OUT_PATH_ID   = BASE / "clusters_reports/random_forest/csv/entradas_selecionadas_final_id_only.csv"

# ============================================
# 1) Carregar arquivos
# ============================================
df_idx   = pd.read_csv(PATH_IDX)
df_test  = pd.read_csv(PATH_TEST)
df_train = pd.read_csv(PATH_TRAIN)

# ============================================
# 2) Juntar train + test (ambos têm orig_index e id)
# ============================================
df_full = pd.concat([df_train, df_test], axis=0, ignore_index=True)
df_full.rename(columns={"Unnamed: 0": "orig_index"}, inplace=True)

# Garantir que patient_id é int
df_full["patient_id"] = df_full["patient_id"].astype(int)

# ============================================
# 3) Índices selecionados (idx_global)
# ============================================
idx_global = df_idx["idx_global"].astype(int).tolist()

# ============================================
# 4) Selecionar linhas pela coluna orig_index
#    (orig_index == idx_global)
# ============================================
df_sel = df_full[df_full["orig_index"].isin(idx_global)].copy()
df_sel

# ============================================
# 5) Arquivo 1: idx_global + id
# ============================================
df_sel.to_csv(OUT_PATH_FULL, index=False)

# ============================================
# 6) Arquivo 2: apenas id
# ============================================
df_out_id = df_sel[["patient_id"]].copy()
df_out_id.to_csv(OUT_PATH_ID, index=False)

print(f"✅ Arquivo com idx_global + id salvo em:\n  {OUT_PATH_FULL}")
print(f"✅ Arquivo somente com id salvo em:\n  {OUT_PATH_ID}")


In [ ]:
aux = pd.read_csv("../UCI-THYROID-DXBIN/data/raw/uci_thyroid_dxbin.csv", sep =';')
aux
